# Projekt 1 – model scoringowy/ratingowy (wymagania 3.1–3.6)

Notebook ma dwie części:
* **Część I (rozdziały 0–11, wymagania 3.1–3.2)** – definicja problemu, podział danych i schemat walidacji, kontrola data leakage, jakość danych, EDA oraz przygotowanie cech dla modelu interpretowalnego (karta WoE + regresja logistyczna) i modelu black-box.
* **Część II (rozdziały 12–19, wymagania 3.3–3.6)** – model interpretowalny, LightGBM, wyjaśnialność (SHAP, LIME, PDP/ICE, what-if), wierność i stabilność wyjaśnień, kalibracja i korekta do PD = 4%, progi decyzyjne, klasy ratingowe, audyt podgrup i analiza kontrfaktyczna.

Część II korzysta z obiektów utworzonych w części I, dlatego notebook należy uruchamiać w całości (Restart & Run All).

**Zasady**
1. Zbiór testowy nie jest używany w EDA, przy dopasowaniu transformacji, w CV, przy kalibracji ani wyborze progów – służy wyłącznie do oceny końcowej (rozdziały 17.4 i 18).
2. Transformacje z parametrami wyznaczanymi z danych (kwantyle, mediany, granice binów, WoE, rzadkie kategorie, selekcja cech) są dopasowywane wyłącznie przez `.fit()` na zbiorze treningowym lub na foldzie treningowym.
3. Przed podziałem wykonywane są tylko operacje bezstanowe, liczone osobno dla każdego wiersza.
4. Konwencje: `default` = 1 oznacza default, PD = P(y = 1); WoE = ln(%dobrych / %złych) (Lab 1) – WoE > 0 oznacza niższe ryzyko, więc w regresji logistycznej na WoE oczekujemy β < 0.

## 0. Konfiguracja

Wszystkie progi i parametry są zdefiniowane w tej komórce i używane w dalszej części notebooka.

In [ ]:
import os, sys, json, hashlib, platform, warnings, itertools
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
from scipy import stats
import sklearn
import joblib
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import (
    RepeatedStratifiedKFold, StratifiedGroupKFold, StratifiedKFold, train_test_split)
from matplotlib.patches import FancyArrowPatch, Rectangle

sns.set_theme(style="whitegrid")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

# --- dane i cel --- #
DATA_PATH = "zbiór_5.csv"
TARGET = "default"
TARGET_PD = 0.04          # tendencja centralna PD; NIE zastępuje DR w próbie

# --- podziały i walidacja --- #
RNG_SEED = 42
TEST_SIZE = 0.20          # udział testu w całości
VALID_SIZE = 0.25         # udział validation w części deweloperskiej (train+validation) -> 20% całości
N_FOLDS = 5               # Stratified K-fold (wymóg: min. 5 foldów)
N_REPEATS = 3             # powtórzenia CV w teście poprawności pipeline'u (rozdz. 9)

# --- polityka czyszczenia (FinancialCleaner) --- #
CLEAN = dict(
    max_missing=0.50,      # usuwamy cechę, gdy NaN (prawdziwe braki) >= 50% obserwacji train
    quasi_const=0.99,      # usuwamy cechę, gdy najczęstsza wartość >= 99% obserwacji z wartością
    winsor=(0.01, 0.99),   # capping black-box: kwantyle 1% i 99% wyznaczone na train
    indicator_min=0.02,    # wskaźnik braku / wartości specjalnej tworzymy od 2% udziału na train
)

# --- binning, WoE i selekcja (model interpretowalny); wartości jak w Lab 1 --- #
BIN = dict(
    max_bins=5,            # maks. liczba przedziałów regularnych (Lab 1: max_bins = 5)
    min_bin_frac=0.05,     # min. udział obserwacji w przedziale (Lab 1: min_share = 0.05)
    smoothing=0.5,         # wygładzanie +0,5 chroni przed log(0) (Lab 1)
    iv_min=0.02,           # IV < 0,02 -> cecha bezużyteczna (Siddiqi 2017, Lab 1)
    corr_max=0.70,         # z pary cech WoE o |r| >= 0,70 zostaje cecha o wyższym IV
    max_features=8,        # limit cech karty: ~107 defaultów / 9 parametrów ≈ 12 zdarzeń na parametr (reguła EPV ≥ 10)
)
MIN_CAT_SHARE = BIN["min_bin_frac"]  # kategoria rzadsza niż 5% train -> "INNE" (spójnie z binami liczbowymi)
IV_SUSPICIOUS = 0.50                 # IV > 0,5 -> "podejrzanie silna – sprawdź przeciek" (Lab 1)
AUC_ALARM = 0.75                     # jednowymiarowa siła AUC >= 0,75 -> sygnał do audytu (Lab 3, 1.4)

ART_DIR = Path("artifacts")
FIG_DIR = ART_DIR / "fig"
FIG_DIR.mkdir(parents=True, exist_ok=True)

def savefig(name):
    """Zapisuje bieżący wykres do artifacts/fig/<name>.png (wszystkie wykresy w jednym katalogu)."""
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{name}.png", dpi=130, bbox_inches="tight")

assert Path(DATA_PATH).is_file(), f"Brak pliku: {DATA_PATH}. Ustaw DATA_PATH na poprawną ścieżkę."
assert 0 < TARGET_PD < 1
assert 0 < TEST_SIZE < 1 and 0 < VALID_SIZE < 1
assert isinstance(N_FOLDS, int) and N_FOLDS >= 5, "Wymóg projektu: co najmniej 5 foldów."
assert 0 < CLEAN["max_missing"] <= 1 and 0 < CLEAN["quasi_const"] <= 1
assert 0 <= CLEAN["winsor"][0] < CLEAN["winsor"][1] <= 1
assert BIN["max_bins"] >= 2 and 0 < BIN["min_bin_frac"] < 1 and 0 < BIN["corr_max"] <= 1

print("Python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__,
      "| scikit-learn", sklearn.__version__, "| scipy", scipy.__version__,
      "| matplotlib", matplotlib.__version__)
print(f"Planowany podział: train {(1-TEST_SIZE)*(1-VALID_SIZE):.0%}, "
      f"validation {(1-TEST_SIZE)*VALID_SIZE:.0%}, test {TEST_SIZE:.0%}; CV: {N_FOLDS} foldów")

## 1. Definicja PD, moment decyzji i dostępność zmiennych

**Zdarzenie i horyzont.** `default` = 1 oznacza zdarzenie ryzyka w przyjętym horyzoncie obserwacji. Dane nie zawierają dat, więc długości horyzontu nie da się sprawdzić w danych – przyjmujemy definicję z opisu zmiennych.

**Moment decyzji.** Zakładamy, że ocena jest wykonywana po sporządzeniu sprawozdania finansowego. Do modelu mogą trafić tylko informacje dostępne w tym momencie.

**Charakter danych.** Zbiór jest przekrojowy – bez dat i bez identyfikatora podmiotu, dlatego:
* podział w czasie (out-of-time) nie jest możliwy – stosujemy losowy podział warstwowy po `default`;
* powtórzenia podmiotów można sprawdzić tylko pośrednio (rozdział 2.2).

O wykluczeniu cechy decyduje jej opis i moment dostępności. Wysokie AUC lub IV pojedynczej cechy jest sygnałem do sprawdzenia, a nie dowodem przecieku.

## 2. Wczytanie danych i kontrola wejścia

Kontrole opisowe na całym pliku: typy kolumn, target, duplikaty, braki. W tym rozdziale nie są wyznaczane parametry transformacji.

In [ ]:
df_local = pd.read_csv(DATA_PATH)
assert not df_local.empty, "Dane są puste."
assert df_local.columns.is_unique, "Powtarzające się nazwy kolumn."
assert TARGET in df_local.columns, f"Brakuje targetu {TARGET!r}."
assert df_local.index.is_unique and not df_local.index.hasnans

print("Rozmiar danych:", df_local.shape)
print("Typy kolumn:", df_local.dtypes.astype(str).value_counts().to_dict())
display(df_local.head(5))

NUM_COLS_df = df_local.select_dtypes(include="number").columns.tolist()
CAT_COLS_df = df_local.select_dtypes(exclude="number").columns.tolist()
print("Kolumny liczbowe:", len(NUM_COLS_df), "| kategoryczne/tekstowe:", len(CAT_COLS_df), CAT_COLS_df)

### 2.1. Kolumny kodowe i poprawność targetu

`pkdKod`, `formaWlasnosci_Symbol` i `szczegolnaFormaPrawna_Symbol` są zapisane jako liczby, ale są kodami (zmiennymi kategorycznymi). Sprawdzamy liczbę ich poziomów oraz to, czy target jest binarny i kompletny.

In [ ]:
CODES = ["pkdKod", "formaWlasnosci_Symbol", "szczegolnaFormaPrawna_Symbol"]
missing_codes = sorted(set(CODES) - set(df_local.columns))
assert not missing_codes, f"Brakujące kolumny kodowe: {missing_codes}"
for col in CODES:
    print(f"\n{col} | liczba unikalnych wartości: {df_local[col].nunique(dropna=True)} | braki: {df_local[col].isna().sum()}")
    display(df_local[col].value_counts(dropna=False).head(12).to_frame("liczebność"))

# Walidacja PRZED konwersją: astype(int) mogłoby ukryć niepożądane wartości.
y_candidate = pd.to_numeric(df_local[TARGET], errors="coerce")
assert y_candidate.notna().all(), "Target zawiera NaN lub wartości nienumeryczne."
assert y_candidate.isin([0, 1]).all(), "Target nie jest binarny 0/1."

X_raw = df_local.drop(columns=[TARGET]).copy()
y_raw = y_candidate.astype("int64").rename(TARGET)
assert TARGET not in X_raw.columns
assert X_raw.index.equals(y_raw.index) and len(X_raw) == len(y_raw)
assert y_raw.nunique() == 2, "Do stratyfikacji i AUC potrzebne są obie klasy."

display(pd.DataFrame({"liczba": y_raw.value_counts().sort_index(),
                      "udział (%)": (100 * y_raw.value_counts(normalize=True).sort_index()).round(2)}))
print(f"Default rate (DR) próby = {y_raw.mean():.2%} ({int(y_raw.sum())} defaultów); "
      f"kotwica PD (cel kalibracji) = {TARGET_PD:.2%}")

`szczegolnaFormaPrawna_Symbol` ma jeden poziom i nie trafia do modeli. DR próby (5,93%) jest wyższy niż docelowe PD = 4% – różnicę koryguje kalibracja poziomu PD na zbiorze validation (rozdział 17), a nie zmiana danych uczących.

### 2.2. Duplikaty i powtarzające się podmioty

Plik nie zawiera identyfikatora firmy. Sprawdzamy identyczne wiersze oraz identyczne odciski obserwacji (`pkdKod`, `formaWlasnosci_Symbol`, `wsk_liczba_dni_istnienia`, `Aktywa`). Powtórzony odcisk oznaczałby, że ta sama obserwacja występuje w pliku więcej niż raz.

In [ ]:
print("W pełni identyczne obserwacje (łącznie z targetem):", int(df_local.duplicated().sum()))
print("W pełni identyczne obserwacje (z wyłączeniem targetu):", int(X_raw.duplicated().sum()))

FINGERPRINT = ["pkdKod", "formaWlasnosci_Symbol", "wsk_liczba_dni_istnienia", "Aktywa"]
assert set(FINGERPRINT).issubset(X_raw.columns), (
    f"Brakuje pól do fingerprintu: {set(FINGERPRINT) - set(X_raw.columns)}")
fp = pd.util.hash_pandas_object(X_raw[FINGERPRINT], index=False)
fp.name = "fingerprint"
assert fp.index.equals(X_raw.index)

dup_fp = fp.duplicated(keep=False)
GROUPS = fp.copy() if dup_fp.any() else None
print("Wiersze z powtórzonym odciskiem:", int(dup_fp.sum()))
print("Unikalne odciski:", fp.nunique())
print("Podział grupowy:", "tak" if GROUPS is not None else "nie")
if GROUPS is not None:
    print("Największe liczebności grup:")
    display(fp.value_counts().head(10).to_frame("liczebnosc"))
    warnings.warn("Fingerprint nie jest identyfikatorem podmiotu. Zweryfikuj, czy grupowanie jest uzasadnione.")

**Wniosek.** Brak identycznych wierszy i powtórzonych odcisków, więc w rozdziale 4 stosujemy podział warstwowy bez grup. Kontrola wykrywa tylko identyczne zapisy – bez identyfikatora podmiotu nie da się wykluczyć, że ta sama firma występuje z danymi z różnych lat.

### 2.3. Zduplikowane kolumny

Kolumny o identycznych wartościach są tu tylko raportowane; usuwa je `FinancialCleaner` w `fit()` na zbiorze treningowym.

In [ ]:
DUPLICATES = X_raw.columns[X_raw.T.duplicated()].tolist()
cechy_zduplikowane = []
for i, col in enumerate(X_raw.columns):
    for prev_col in X_raw.columns[:i]:
        if X_raw[col].equals(X_raw[prev_col]):
            cechy_zduplikowane.append(prev_col)
            break
assert len(cechy_zduplikowane) == len(DUPLICATES)
assert all(X_raw[a].equals(X_raw[b]) for a, b in zip(DUPLICATES, cechy_zduplikowane))
print("Dokładnie zduplikowane kolumny:", len(DUPLICATES))
display(pd.DataFrame({"duplikat": DUPLICATES, "odpowiada kolumnie": cechy_zduplikowane}))

### 2.4. Braki, wartości nieskończone i dominacja zer

Przegląd całego pliku. Rozróżniamy `NaN` (brak wartości) i `±inf` (dzielenie przez zero we wskaźnikach). Ich znaczenie sprawdzamy na zbiorze treningowym w rozdziale 6.2.

In [ ]:
X_num = X_raw.select_dtypes(include="number").astype(float)
nan_cnt = X_num.isna().sum()
posinf_cnt = pd.Series(np.isposinf(X_num.to_numpy()).sum(axis=0), index=X_num.columns)
neginf_cnt = pd.Series(np.isneginf(X_num.to_numpy()).sum(axis=0), index=X_num.columns)
przeglad = pd.DataFrame({"NaN": nan_cnt, "+inf": posinf_cnt, "-inf": neginf_cnt})
przeglad["NaN (%)"] = 100 * przeglad["NaN"] / len(X_raw)
przeglad["inf (%)"] = 100 * (przeglad["+inf"] + przeglad["-inf"]) / len(X_raw)
print(f"NaN łącznie: {int(nan_cnt.sum())} | +inf: {int(posinf_cnt.sum())} | -inf: {int(neginf_cnt.sum())}")
print("Kolumny z NaN lub inf:")
display(przeglad[(przeglad["NaN"] > 0) | (przeglad["+inf"] + przeglad["-inf"] > 0)]
        .sort_values(["NaN (%)", "inf (%)"], ascending=False).round(2))

X_num_finite = X_num.replace([np.inf, -np.inf], np.nan)
niebrakujace = X_num_finite.notna().sum()
liczba_zer = (X_num_finite == 0).sum()
udzial_zer = liczba_zer.div(niebrakujace.replace(0, np.nan))
duzo_zer = pd.DataFrame({
    "niebrakujące": niebrakujace,
    "zera": liczba_zer,
    "udział zer": udzial_zer,
    "unikalne": X_num_finite.nunique(dropna=True),
})
duzo_zer = duzo_zer[duzo_zer["udział zer"] > 0.90].sort_values("udział zer", ascending=False)
print("Cechy numeryczne z >90% zer wśród wartości dostępnych:", len(duzo_zer))
display(duzo_zer.round(4))

## 3. Transformery danych

* **Transformacje bezstanowe** (`add_stateless_features`) – liczone osobno dla każdego wiersza, więc mogą być wykonane przed podziałem: dział PKD, `log1p` aktywów, `signed_log1p` pozycji kwotowych.
* **Transformacje z parametrami** – klasy zgodne z API scikit-learn; parametry są wyznaczane w `fit()` i stosowane w `transform()`:

| Klasa | Co wyznacza w `fit()` | Użycie |
|---|---|---|
| `FinancialCleaner` | cechy do usunięcia (braki, quasi-stałe, duplikaty), granice cappingu, wartości zastępujące `±inf`, mediany, średnie i odchylenia | obie ścieżki |
| `CategoryGrouper` | listę wystarczająco licznych kategorii; pozostałe → `INNE` | obie ścieżki |
| `WoEEncoder` | granice binów monotonicznych, WoE, IV, selekcję cech | model interpretowalny |
| `ScoringFeatureBuilder` | łączy powyższe w ścieżkę `mode="woe"` albo `mode="blackbox"` | obie ścieżki |

**Transformacje bezstanowe.**

In [ ]:
def signed_log1p(x):
    """
    sign(x) * ln(1 + |x|): kompresuje skalę kwot, zachowuje znak i zero; inf pozostaje inf.
    """
    return np.sign(x) * np.log1p(np.abs(x))

def add_stateless_features(data, level_cols=()):
    """
    Cechy liczone wiersz po wierszu, bez żadnych statystyk z próby.
    """
    out = data.copy()
    pkd = pd.to_numeric(out["pkdKod"], errors="coerce")
    out["pkd_div"] = (pkd // 100).astype("Int64").astype("string").str.zfill(2).fillna("BRAK").astype(str)
    out["forma_wl"] = out["formaWlasnosci_Symbol"].astype("string").fillna("BRAK").astype(str)
    aktywa = pd.to_numeric(out["Aktywa"], errors="coerce")
    out["log_aktywa"] = np.log1p(aktywa.where(aktywa >= 0))      # ujemne aktywa -> NaN (wartość niemożliwa)
    nowe = {f"slog_{c}": signed_log1p(pd.to_numeric(out[c], errors="coerce")) for c in level_cols}
    return pd.concat([out, pd.DataFrame(nowe, index=out.index)], axis=1)

assert np.isclose(signed_log1p(-9), -np.log(10)) and np.isclose(signed_log1p(9), np.log(10))
assert signed_log1p(0) == 0

**`FinancialCleaner`** – czyszczenie cech liczbowych. W `fit()` wyznacza: cechy do usunięcia (braki ≥ `max_missing`, cechy quasi-stałe, duplikaty kolumn), granice cappingu i wartości zastępujące `±inf`, mediany do imputacji oraz średnie i odchylenia do standaryzacji. W trybie `inf_policy="keep"` (karta WoE) tylko usuwa cechy, a wartości specjalne obsługuje binning.

In [ ]:
class FinancialCleaner(BaseEstimator, TransformerMixin):
    """
    Czyszczenie zmiennych liczbowych; wszystkie parametry wyznaczane w fit() na train.
    """

    def __init__(self, max_missing=0.50, quasi_const=0.99, winsor=(0.01, 0.99), impute=True,
                 indicator_min=0.02, scale=False, inf_policy="cap", drop_duplicates=True):
        self.max_missing = max_missing              # maks. udział NaN (inf to nie brak)
        self.quasi_const = quasi_const              # maks. udział najczęstszej wartości
        self.winsor = winsor                        # kwantyle cappingu (None = bez cappingu)
        self.impute = impute                        # imputacja medianą + flagi __isna
        self.indicator_min = indicator_min          # min. udział NaN / inf, od którego tworzymy flagę
        self.scale = scale                          # standaryzacja (SVM / sieć)
        self.inf_policy = inf_policy                # "cap" (black-box) albo "keep" (karta WoE)
        self.drop_duplicates = drop_duplicates

    def fit(self, X, y=None):
        assert self.inf_policy in ("cap", "keep"), "inf_policy: 'cap' albo 'keep'."
        assert X.columns.is_unique, "Powtarzające się nazwy kolumn przy fit()."
        X = X.astype(float)
        inf_mask = np.isinf(X)
        nan_share = X.isna().mean()
        inf_share = inf_mask.mean()
        top_share = X.apply(lambda s: s.value_counts(normalize=True, dropna=True).iloc[0]
                            if s.notna().any() else 1.0)

        drop_missing = nan_share[nan_share >= self.max_missing].index.tolist()
        drop_const = [c for c in X.columns if c not in drop_missing and top_share[c] >= self.quasi_const]
        kept = [c for c in X.columns if c not in drop_missing + drop_const]
        drop_dup = {}
        if self.drop_duplicates:
            uniq = []
            for c in kept:
                twin = next((u for u in uniq if X[c].equals(X[u])), None)
                if twin is None:
                    uniq.append(c)
                else:
                    drop_dup[c] = twin
            kept = uniq
        self.dropped_ = {"za_duzo_brakow": drop_missing, "quasi_stale": drop_const,
                         "duplikat": list(drop_dup), "duplikat_kolumny": drop_dup}
        self.cols_ = kept
        self.n_features_out_hint_ = len(kept)
        if self.inf_policy == "keep":
            return self

        Xf = X[kept].mask(inf_mask[kept])                            # tylko wartości skończone
        if self.winsor is not None:
            self.lo_, self.hi_ = Xf.quantile(self.winsor[0]), Xf.quantile(self.winsor[1])
        else:
            self.lo_, self.hi_ = Xf.min(), Xf.max()
        self.posinf_fill_ = self.hi_.fillna(0.0)                     # +inf -> górna granica z train
        self.neginf_fill_ = self.lo_.fillna(0.0)                     # -inf -> dolna granica z train
        self.inf_ind_cols_ = [c for c in kept if inf_share[c] >= self.indicator_min]
        self.nan_ind_cols_ = [c for c in kept if nan_share[c] >= self.indicator_min] if self.impute else []
        Xc = self._cap(X[kept])
        self.median_ = Xc.median().fillna(0.0)
        Xi = Xc.fillna(self.median_) if self.impute else Xc
        self.mean_ = Xi.mean()
        self.std_ = Xi.std().replace(0, 1.0).fillna(1.0)
        return self

    def _cap(self, X):
        arr = X.to_numpy(dtype=float, copy=True)
        pos, neg = np.isposinf(arr), np.isneginf(arr)
        arr[pos] = np.broadcast_to(self.posinf_fill_.to_numpy(), arr.shape)[pos]
        arr[neg] = np.broadcast_to(self.neginf_fill_.to_numpy(), arr.shape)[neg]
        out = pd.DataFrame(arr, index=X.index, columns=X.columns)
        if self.winsor is not None:
            out = out.clip(lower=self.lo_, upper=self.hi_, axis=1)
        return out

    def transform(self, X):
        X = X.astype(float).reindex(columns=self.cols_)
        if self.inf_policy == "keep":
            return X
        parts = []
        if self.inf_ind_cols_:
            parts.append(np.isinf(X[self.inf_ind_cols_]).astype(int).add_suffix("__inf"))
        if self.nan_ind_cols_:
            parts.append(X[self.nan_ind_cols_].isna().astype(int).add_suffix("__isna"))
        Z = self._cap(X)
        if self.impute:
            Z = Z.fillna(self.median_)
        if self.scale:
            Z = (Z - self.mean_) / self.std_
        return pd.concat([Z] + parts, axis=1)

**`CategoryGrouper`** – poziomy rzadsze niż `min_share` zbioru treningowego oraz poziomy niewystępujące w treningu są łączone w `INNE` (jak w Lab 1). Opcjonalnie tworzy kolumny one-hot ustalone w `fit()`.

In [ ]:
class CategoryGrouper(BaseEstimator, TransformerMixin):
    def __init__(self, min_share=0.05, onehot=False):
        self.min_share = min_share
        self.onehot = onehot

    @staticmethod
    def _as_str(s):
        return s.astype("string").fillna("BRAK").astype(str)

    def fit(self, X, y=None):
        self.cols_ = list(X.columns)
        min_count = int(np.ceil(self.min_share * len(X)))
        self.levels_ = {}
        for c in self.cols_:
            vc = self._as_str(X[c]).value_counts()
            self.levels_[c] = sorted(vc[vc >= min_count].index)
        self.dummy_cols_ = self._dummies(self._group(X)).columns.tolist() if self.onehot else None
        return self

    def _group(self, X):
        out = {}
        for c in self.cols_:
            s = self._as_str(X[c])
            out[c] = s.where(s.isin(self.levels_[c]), "INNE")
        return pd.DataFrame(out, index=X.index)

    @staticmethod
    def _dummies(G):
        return pd.get_dummies(G, prefix_sep="=", dtype=int)

    def transform(self, X):
        G = self._group(X)
        if not self.onehot:
            return G
        return self._dummies(G).reindex(columns=self.dummy_cols_, fill_value=0)

**Binning monotoniczny i WoE.** Konwencja Lab 1: $WoE_i = \ln\frac{\%\text{dobrych}_i}{\%\text{złych}_i}$, $IV = \sum_i(\%\text{dobrych}_i - \%\text{złych}_i)\cdot WoE_i$, wygładzanie +0,5 w liczebnościach. Algorytm dla cechy liczbowej (wartości skończone):
1. 20 wstępnych przedziałów kwantylowych,
2. łączenie przedziałów mniejszych niż `min_bin_frac` z sąsiadem o najbardziej podobnej stopie defaultu,
3. wymuszenie monotoniczności (łączenie sąsiadów łamiących porządek) w obu kierunkach i wybór kierunku o wyższym IV,
4. redukcja do `max_bins` przez łączenie sąsiadów o najbliższych log-szansach,
5. osobne przedziały dla `NaN`, `+inf`, `-inf`; przedział specjalny mniejszy niż `min_bin_frac` dostaje WoE = 0.

Kierunek `inc` oznacza, że wraz ze wzrostem wartości cechy rośnie stopa defaultu (WoE maleje); `dec` – odwrotnie.

In [ ]:
SPECIAL = {"nan": "BRAK", "posinf": "+INF", "neginf": "-INF"}

def _woe_iv(bad, good, s=0.5):
    """
    WoE = ln(%dobrych / %złych) z wygładzaniem s; zwraca (WoE, wkłady IV).
    """
    bad, good = np.asarray(bad, float), np.asarray(good, float)
    k = len(bad)
    pb = (bad + s) / (bad.sum() + s * k)
    pg = (good + s) / (good.sum() + s * k)
    woe = np.log(pg / pb)
    return woe, (pg - pb) * woe

def _merge(edges, tot, bad, i):
    """
    Łączy przedziały i oraz i+1.
    """
    edges.pop(i)
    tot[i] += tot[i + 1]; bad[i] += bad[i + 1]
    del tot[i + 1]; del bad[i + 1]

def _smooth_odds(tot, bad, s):
    """
    Wygładzone szanse złego (bad+s)/(good+s); monotoniczność szans <=> monotoniczność WoE.
    """
    tot, bad = np.asarray(tot, float), np.asarray(bad, float)
    return (bad + s) / (tot - bad + s)

def _pav(edges, tot, bad, direction, s=0.5):
    edges, tot, bad = list(edges), list(tot), list(bad)
    while True:
        odds = _smooth_odds(tot, bad, s)
        viol = [i for i in range(len(odds) - 1)
                if (direction == "inc" and odds[i] > odds[i + 1]) or (direction == "dec" and odds[i] < odds[i + 1])]
        if not viol:
            return edges, tot, bad
        _merge(edges, tot, bad, viol[0])

def fit_numeric_bins(x, y, max_bins=5, min_frac=0.05, monotonic=True, n_init=20, smoothing=0.5):
    """
    Binning monotoniczny + przedziały wartości specjalnych. Zwraca słownik z granicami, WoE i IV.
    """
    v = np.asarray(x, dtype=float)
    yv_all = np.asarray(y, dtype=float)
    n_total = len(v)
    min_size = max(int(np.ceil(min_frac * n_total)), 1)
    fin = np.isfinite(v)
    xv, yv = v[fin], yv_all[fin]
    edges = (list(np.unique(np.quantile(xv, np.linspace(0, 1, n_init + 1)[1:-1])))
             if len(xv) and np.ptp(xv) > 0 else [])

    idx = np.searchsorted(edges, xv, side="right")
    tot = np.bincount(idx, minlength=len(edges) + 1).astype(float).tolist()
    bad = np.bincount(idx, weights=yv, minlength=len(edges) + 1).tolist()
    keep = [i for i, t in enumerate(tot) if t > 0]                  # puste przedziały (remisy) usuwamy
    if len(keep) < len(tot):
        new_edges = [edges[i - 1] for i in keep[1:]]
        edges, tot, bad = new_edges, [tot[i] for i in keep], [bad[i] for i in keep]
    while len(tot) > 1 and min(tot) < min_size:
        i = int(np.argmin(tot)); rate = np.array(bad) / np.array(tot)
        j = min([k for k in (i - 1, i + 1) if 0 <= k < len(tot)], key=lambda k: abs(rate[k] - rate[i]))
        _merge(edges, tot, bad, min(i, j))
    direction = "n/d"
    if monotonic and len(tot) > 1:
        best = None
        for d in ("inc", "dec"):
            e2, t2, b2 = _pav(edges, tot, bad, d, smoothing)
            iv_d = _woe_iv(b2, np.array(t2) - np.array(b2), smoothing)[1].sum()
            if best is None or iv_d > best[0]:
                best = (iv_d, d, e2, t2, b2)
        _, direction, edges, tot, bad = best
    while len(tot) > max_bins:
        lo = np.log(_smooth_odds(tot, bad, smoothing))
        _merge(edges, tot, bad, int(np.argmin(np.abs(np.diff(lo)))))
        if direction in ("inc", "dec"):
            edges, tot, bad = (list(z) for z in _pav(edges, tot, bad, direction, smoothing))
    if len(tot) == 1:
        direction = "n/d"

    special = {}
    for key, mask in (("nan", np.isnan(v)), ("posinf", np.isposinf(v)), ("neginf", np.isneginf(v))):
        if mask.any():
            special[key] = {"n": int(mask.sum()), "bad": float(yv_all[mask].sum())}
    keys = list(special)
    tot_all = list(tot) + [special[k]["n"] for k in keys]
    bad_all = list(bad) + [special[k]["bad"] for k in keys]
    woe, ivc = _woe_iv(bad_all, np.array(tot_all) - np.array(bad_all), smoothing)
    for j, k in enumerate(keys, start=len(tot)):
        small = special[k]["n"] < min_size
        special[k]["woe"] = 0.0 if small else float(woe[j])
        special[k]["iv"] = 0.0 if small else float(ivc[j])
        special[k]["neutral"] = small
    woe_bins = woe[:len(tot)]
    mono = bool(np.all(np.diff(woe_bins) >= -1e-12) or np.all(np.diff(woe_bins) <= 1e-12))
    iv = float(ivc[:len(tot)].sum() + sum(special[k]["iv"] for k in keys))
    return dict(kind="num", edges=np.array(edges, dtype=float), woe=np.asarray(woe_bins, float),
                iv_bins=np.asarray(ivc[:len(tot)], float), iv=iv, special=special,
                n_bins=len(tot) + len(keys), monotonic=mono, direction=direction,
                counts=np.array(tot), bads=np.array(bad), min_size=min_size)

def fit_categorical_woe(x, y, min_frac=0.05, smoothing=0.5):
    """
    WoE dla kategorii (po grupowaniu rzadkich poziomów do INNE); poziom za rzadki -> WoE = 0.
    """
    d = pd.DataFrame({"x": x.astype(str).to_numpy(), "y": np.asarray(y)})
    g = d.groupby("x")["y"].agg(["sum", "count"])
    min_size = max(int(np.ceil(min_frac * len(d))), 1)
    woe, ivc = _woe_iv(g["sum"].to_numpy(), (g["count"] - g["sum"]).to_numpy(), smoothing)
    small = g["count"].to_numpy() < min_size
    woe, ivc = np.where(small, 0.0, woe), np.where(small, 0.0, ivc)
    return dict(kind="cat", mapping=dict(zip(g.index, woe)), counts=g["count"].to_dict(),
                bads=g["sum"].to_dict(), iv_levels=dict(zip(g.index, ivc)), iv=float(ivc.sum()),
                n_bins=len(g), monotonic=np.nan, direction="n/d", min_size=min_size)

def iv_label(v):
    """
    Słowna ocena IV wg Siddiqi (2017), jak w Lab 1.
    """
    return ("bezużyteczna" if v < 0.02 else "słaba" if v < 0.1 else "średnia" if v < 0.3
            else "silna" if v < 0.5 else "podejrzanie silna – sprawdź przeciek")

**`WoEEncoder`** – binning i WoE dla wszystkich cech, a następnie selekcja: (1) IV ≥ `iv_min`, (2) cechy wybierane według malejącego IV, jeśli korelacja ich WoE z każdą już wybraną cechą jest < `corr_max`, (3) limit `max_features`. Tabela `iv_table_` zawiera powód decyzji dla każdej cechy.

In [ ]:
class WoEEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cat_cols=(), max_bins=5, min_bin_frac=0.05, monotonic=True,
                 iv_min=0.02, corr_max=0.70, smoothing=0.5, max_features=12):
        self.cat_cols = cat_cols
        self.max_bins = max_bins
        self.min_bin_frac = min_bin_frac
        self.monotonic = monotonic
        self.iv_min = iv_min
        self.corr_max = corr_max
        self.smoothing = smoothing
        self.max_features = max_features

    def fit(self, X, y):
        y = np.asarray(y)
        self.bins_ = {}
        for c in X.columns:
            if c in self.cat_cols:
                self.bins_[c] = fit_categorical_woe(X[c], y, self.min_bin_frac, self.smoothing)
            else:
                self.bins_[c] = fit_numeric_bins(X[c], y, self.max_bins, self.min_bin_frac,
                                                 self.monotonic, smoothing=self.smoothing)
        iv = pd.Series({c: b["iv"] for c, b in self.bins_.items()}).sort_values(ascending=False)
        cand = iv[iv >= self.iv_min].index.tolist()
        Z = self._woe(X[cand]) if cand else pd.DataFrame(index=X.index)
        corr = Z.corr().abs() if cand else None
        kept, reason = [], {}
        for c in cand:
            if self.max_features and len(kept) >= self.max_features:
                reason[c] = "limit liczby cech"
                continue
            blockers = [k for k in kept if pd.notna(corr.loc[c, k]) and corr.loc[c, k] >= self.corr_max]
            if blockers:
                k = max(blockers, key=lambda b: corr.loc[c, b])
                reason[c] = f"korelacja z {k} (|r| = {corr.loc[c, k]:.2f})"
            else:
                kept.append(c); reason[c] = "wybrana"
        for c in iv.index:
            reason.setdefault(c, "IV < iv_min")
        self.selected_ = kept
        self.iv_table_ = pd.DataFrame({
            "IV": iv,
            "ocena_IV": [iv_label(v) for v in iv],
            "n_binow": [self.bins_[c]["n_bins"] for c in iv.index],
            "monotoniczna": [self.bins_[c]["monotonic"] for c in iv.index],
            "kierunek_ryzyka": [self.bins_[c]["direction"] for c in iv.index],
            "status": pd.Series(reason),
        })
        return self

    def _woe(self, X):
        out = {}
        for c in X.columns:
            b = self.bins_[c]
            if b["kind"] == "cat":
                out[c] = X[c].astype(str).map(b["mapping"]).fillna(0.0).astype(float).to_numpy()
            else:
                v = X[c].to_numpy(dtype=float)
                w = b["woe"][np.searchsorted(b["edges"], np.where(np.isfinite(v), v, 0.0), side="right")]
                for key, mask in (("nan", np.isnan(v)), ("posinf", np.isposinf(v)), ("neginf", np.isneginf(v))):
                    w = np.where(mask, b["special"].get(key, {"woe": 0.0})["woe"], w)  # niewidziane -> 0
                out[c] = w
        return pd.DataFrame(out, index=X.index)

    def transform(self, X):
        Z = self._woe(X[self.selected_]) if self.selected_ else pd.DataFrame(index=X.index)
        return Z.add_prefix("woe__")

    def profile(self, col):
        """Tabela przedziałów: liczebność, defaulty, DR, % dobrych/złych, WoE, IV – do raportu."""
        b = self.bins_[col]
        G_tot = B_tot = None
        rows = []
        if b["kind"] == "cat":
            for lev, w in b["mapping"].items():
                n, d = b["counts"][lev], b["bads"][lev]
                rows.append(dict(przedzial=lev, n=int(n), defaulty=int(d), WoE=w, IV=b["iv_levels"][lev],
                                 uwaga="za mało obs. -> WoE 0" if n < b["min_size"] else ""))
        else:
            e = np.r_[-np.inf, b["edges"], np.inf]
            for i in range(len(b["counts"])):
                rows.append(dict(przedzial=f"[{e[i]:.4g}, {e[i+1]:.4g})", n=int(b["counts"][i]),
                                 defaulty=int(b["bads"][i]), WoE=b["woe"][i], IV=b["iv_bins"][i], uwaga=""))
            for key, sp in b["special"].items():
                rows.append(dict(przedzial=SPECIAL[key], n=sp["n"], defaulty=int(sp["bad"]), WoE=sp["woe"],
                                 IV=sp["iv"], uwaga="za mało obs. -> WoE 0" if sp["neutral"] else "wartość specjalna"))
        p = pd.DataFrame(rows)
        G_tot = (p["n"] - p["defaulty"]).sum(); B_tot = p["defaulty"].sum()
        p.insert(3, "DR", p["defaulty"] / p["n"])
        p.insert(4, "% dobrych", (p["n"] - p["defaulty"]) / G_tot)
        p.insert(5, "% złych", p["defaulty"] / B_tot)
        return p

**`ScoringFeatureBuilder`** – wspólny interfejs dla obu modeli:
* `mode="woe"`: `FinancialCleaner` (bez imputacji i cappingu) → `CategoryGrouper` → `WoEEncoder`;
* `mode="blackbox"`: `FinancialCleaner` (capping p1–p99, `±inf` → granice cappingu, imputacja medianą, flagi `__isna` / `__inf`) → one-hot kategorii; opcja `scale=True` dla SVM lub sieci.

In [ ]:
class ScoringFeatureBuilder(BaseEstimator, TransformerMixin):
    def __init__(self, num_cols, cat_cols, mode="woe", min_cat_share=0.05,
                 max_missing=0.50, quasi_const=0.99, winsor=(0.01, 0.99), indicator_min=0.02,
                 max_bins=5, min_bin_frac=0.05, smoothing=0.5, iv_min=0.02, corr_max=0.70,
                 max_features=12, scale=False):
        self.num_cols = num_cols
        self.cat_cols = cat_cols
        self.mode = mode
        self.min_cat_share = min_cat_share
        self.max_missing = max_missing
        self.quasi_const = quasi_const
        self.winsor = winsor
        self.indicator_min = indicator_min
        self.max_bins = max_bins
        self.min_bin_frac = min_bin_frac
        self.smoothing = smoothing
        self.iv_min = iv_min
        self.corr_max = corr_max
        self.max_features = max_features
        self.scale = scale

    def _check(self, X, y):
        assert self.mode in ("woe", "blackbox"), "mode: 'woe' albo 'blackbox'."
        assert 0 < self.max_missing <= 1 and 0 < self.quasi_const <= 1
        assert self.winsor is None or 0 <= self.winsor[0] < self.winsor[1] <= 1
        assert self.max_bins >= 2 and 0 < self.min_bin_frac < 1 and 0 < self.min_cat_share < 1
        assert 0 < self.corr_max <= 1 and self.iv_min >= 0 and (self.max_features or 1) >= 1
        assert not (set(self.num_cols) & set(self.cat_cols)), "Cechy liczbowe i kategoryczne nachodzą na siebie."
        assert set(self.num_cols) | set(self.cat_cols) <= set(X.columns), "Brak części kolumn w X."
        if isinstance(y, (pd.Series, pd.DataFrame)):
            assert X.index.equals(y.index), "Niezgodne indeksy X i y w fit()."
        assert len(X) == len(y), "Różne długości X i y w fit()."

    def fit(self, X, y):
        self._check(X, y)
        woe = self.mode == "woe"
        self.cleaner_ = FinancialCleaner(
            max_missing=self.max_missing, quasi_const=self.quasi_const,
            winsor=None if woe else self.winsor, impute=not woe, indicator_min=self.indicator_min,
            scale=self.scale and not woe, inf_policy="keep" if woe else "cap",
        ).fit(X[list(self.num_cols)])
        self.grouper_ = CategoryGrouper(min_share=self.min_cat_share, onehot=not woe).fit(X[list(self.cat_cols)])
        if woe:
            self.encoder_ = WoEEncoder(
                cat_cols=list(self.cat_cols), max_bins=self.max_bins, min_bin_frac=self.min_bin_frac,
                smoothing=self.smoothing, iv_min=self.iv_min, corr_max=self.corr_max,
                max_features=self.max_features,
            ).fit(self._assemble(X), y)
            self.iv_table_ = self.encoder_.iv_table_
        self.feature_names_ = self.transform(X.head(2)).columns.tolist()
        return self

    def _assemble(self, X):
        return pd.concat([self.cleaner_.transform(X[list(self.num_cols)]),
                          self.grouper_.transform(X[list(self.cat_cols)])], axis=1)

    def transform(self, X):
        Z = self._assemble(X)
        return self.encoder_.transform(Z) if self.mode == "woe" else Z

### Role zmiennych i cechy bezstanowe

* `wsk_*` – wskaźniki finansowe i sektorowe (oba modele);
* pozostałe kolumny liczbowe – pozycje kwotowe sprawozdań: w black-boxie po `signed_log1p`, w karcie tylko `log_aktywa` (wielkość firmy);
* `pkdKod` (agregowany do 2-cyfrowego działu), forma własności i schematy sprawozdań – zmienne kategoryczne.

In [ ]:
CAT_RAW = ["szczegolnaFormaPrawna_Symbol", "formaWlasnosci_Symbol", "pkdKod", "schemat_wsk_bilans", "schemat_wsk_rzis"]
assert set(CAT_RAW).issubset(X_raw.columns), f"Brak kodów: {set(CAT_RAW) - set(X_raw.columns)}"
RATIO_COLS = [c for c in X_raw.columns if c.startswith("wsk_")]
LEVEL_COLS = [c for c in X_raw.columns if c not in set(RATIO_COLS) | set(CAT_RAW)]
SUSPICIOUS = ["wsk_index_aktywne", "wsk_index_skr_zaw", "wsk_index_nowe"]   # wskaźniki sektorowe (rozdz. 1)

assert not (set(RATIO_COLS) & set(LEVEL_COLS)) and not (set(CAT_RAW) & set(LEVEL_COLS))
assert set(RATIO_COLS) | set(LEVEL_COLS) | set(CAT_RAW) == set(X_raw.columns)
assert all(pd.api.types.is_numeric_dtype(X_raw[c]) for c in RATIO_COLS + LEVEL_COLS)
assert set(SUSPICIOUS) <= set(RATIO_COLS)

X_git = add_stateless_features(X_raw, LEVEL_COLS)       # cechy surowe + bezstanowe, ten sam indeks co X_raw
CAT_COLS = ["pkd_div", "forma_wl", "schemat_wsk_bilans", "schemat_wsk_rzis"]
NUM_INTERP = RATIO_COLS + ["log_aktywa"]
NUM_BLACK = NUM_INTERP + [f"slog_{c}" for c in LEVEL_COLS]

assert X_git.index.equals(X_raw.index) and X_git.columns.is_unique and TARGET not in X_git.columns
assert set(NUM_BLACK + CAT_COLS) <= set(X_git.columns) and not (set(NUM_BLACK) & set(CAT_COLS))
assert len(NUM_BLACK) == len(set(NUM_BLACK))

print(f"Wskaźniki wsk_*: {len(RATIO_COLS)} | pozycje kwotowe: {len(LEVEL_COLS)}")
print(f"Kandydaci – model interpretowalny: {len(NUM_INTERP)} liczbowych + {len(CAT_COLS)} kategoryczne")
print(f"Kandydaci – black-box: {len(NUM_BLACK)} liczbowych + {len(CAT_COLS)} kategoryczne")
print("Poziomy PKD przed/po agregacji do działu:", X_raw["pkdKod"].nunique(), "→", X_git["pkd_div"].nunique())
print("Ujemne aktywa (log_aktywa = NaN):", int((X_raw["Aktywa"] < 0).sum()))

## 4. Podział danych i schemat walidacji

Podział: train 60%, validation 20%, test 20%, stratyfikowany po `default`, z ustalonym ziarnem. W rozdziale 2.2 nie wykryto powtórzeń, więc stosujemy `train_test_split(stratify=...)`. Jeśli `GROUPS` nie jest puste, kod wykonuje podziały i CV przez `StratifiedGroupKFold`, tak aby ta sama grupa nie trafiła do dwóch zbiorów.

| Zbiór | Zastosowanie | Nie jest używany do |
|---|---|---|
| TRAIN (60%) | dopasowanie transformacji; 5-fold CV do wyboru cech i hiperparametrów (pipeline dopasowywany od nowa w każdym foldzie) | – |
| VALIDATION (20%) | kalibracja PD (kształt i poziom – średnie PD = 4%), progi decyzyjne, klasy ratingowe | dopasowania transformacji cech |
| TEST (20%) | jednorazowa końcowa ocena modeli | strojenia, kalibracji, wyboru cech, EDA |

`cv_dev` (5 foldów na train + validation) jest przygotowany na wypadek kalibracji na predykcjach out-of-fold.

In [ ]:
def stratified_group_holdout(X, y, groups, n_splits, seed):
    """Jeden fold jako holdout, reszta jako development; pozycje względem X."""
    assert X.index.equals(y.index) and X.index.equals(groups.index)
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return next(sgkf.split(X, y, groups=groups))

if GROUPS is None:
    X_dev, X_test, y_dev, y_test = train_test_split(X_raw, y_raw, test_size=TEST_SIZE, stratify=y_raw, shuffle=True, random_state=RNG_SEED)
    X_train, X_valid, y_train, y_valid = train_test_split(X_dev, y_dev, test_size=VALID_SIZE, stratify=y_dev, shuffle=True, random_state=RNG_SEED)
    SPLIT_MODE = "warstwowy losowy (stratyfikacja po default)"
else:   # powtórzenia podmiotów -> żadna grupa nie trafia do dwóch zbiorów
    dev_pos, test_pos = stratified_group_holdout(X_raw, y_raw, GROUPS, 5, RNG_SEED)
    X_dev, X_test, y_dev, y_test = X_raw.iloc[dev_pos], X_raw.iloc[test_pos], y_raw.iloc[dev_pos], y_raw.iloc[test_pos]
    tr_pos, va_pos = stratified_group_holdout(X_dev, y_dev, GROUPS.loc[X_dev.index], 4, RNG_SEED + 1)
    X_train, X_valid, y_train, y_valid = X_dev.iloc[tr_pos], X_dev.iloc[va_pos], y_dev.iloc[tr_pos], y_dev.iloc[va_pos]
    SPLIT_MODE = "warstwowy grupowy (StratifiedGroupKFold)"
    for name, part, desired in [("test", X_test, TEST_SIZE), ("validation", X_valid, (1 - TEST_SIZE) * VALID_SIZE)]:
        if abs(len(part) / len(X_raw) - desired) > 0.03:
            warnings.warn(f"Podział grupowy: udział {name} = {len(part)/len(X_raw):.1%} (cel {desired:.0%}).")


# Pozycje GLOBALNE (względem X_raw / X_git) – jedyny układ indeksów używany do wybierania wierszy w dalszych rozdziałach.
POS = pd.Series(np.arange(len(X_raw)), index=X_raw.index)
idx_train, idx_val, idx_test, idx_dev = (POS.loc[s.index].to_numpy() for s in (X_train, X_valid, X_test, X_dev))
y_all = y_raw.to_numpy()

for name, X, y in [("train", X_train, y_train), ("validation", X_valid, y_valid),
                   ("dev", X_dev, y_dev), ("test", X_test, y_test)]:
    assert len(X) > 0 and X.index.equals(y.index), f"Indeksy {name} nie są zgodne."
    assert y.nunique() == 2, f"W zbiorze {name} nie ma obu klas."

def podsumowanie(nazwa, y):
    return {"zbiór": nazwa, "n": len(y), "udział w całości (%)": round(100 * len(y) / len(y_raw), 1),
            "defaulty": int(y.sum()), "DR (%)": round(100 * y.mean(), 2)}

print("Strategia:", SPLIT_MODE)
split_table = pd.DataFrame([podsumowanie("train", y_train), podsumowanie("validation", y_valid),
                            podsumowanie("dev = train + validation", y_dev), podsumowanie("test", y_test),
                            podsumowanie("RAZEM", y_raw)]).set_index("zbiór")
display(split_table)

Liczba defaultów jest mała (107 / 35 / 36). Dlatego karta ma ograniczoną liczbę cech, przedziały WoE mają co najmniej 5% obserwacji, a przy kalibracji na validation (35 defaultów) wyniki regresji izotonicznej mogą być niestabilne.

### 4.1. Stratified K-Fold

5 foldów na train (`cv_train`) i na dev (`cv_dev`) oraz powtarzane CV 5×3 (`cv_train_rep`) do testów w rozdziale 9. Foldy są zapisywane jako pozycje lokalne (w `X_train` / `X_dev`) i globalne (w `X_git`); rozdziały 9 i 12–18 używają pozycji globalnych.

In [ ]:
assert y_train.value_counts().min() >= N_FOLDS, "Za mało obserwacji klasy mniejszościowej na CV train."
assert y_dev.value_counts().min() >= N_FOLDS, "Za mało obserwacji klasy mniejszościowej na CV dev."

if GROUPS is None:
    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RNG_SEED)
    cv_train = list(cv.split(X_train, y_train))
    cv_dev = list(cv.split(X_dev, y_dev))
    # powtarzane CV (5 foldów × 3 powtórzenia) – tylko do testów w rozdz. 9
    rcv = RepeatedStratifiedKFold(n_splits=N_FOLDS, n_repeats=N_REPEATS, random_state=RNG_SEED)
    cv_train_rep = list(rcv.split(X_train, y_train))
else:
    cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=RNG_SEED)
    cv_train = list(cv.split(X_train, y_train, groups=GROUPS.loc[X_train.index]))
    cv_dev = list(cv.split(X_dev, y_dev, groups=GROUPS.loc[X_dev.index]))
    cv_train_rep = [f for r in range(N_REPEATS) for f in StratifiedGroupKFold(
        n_splits=N_FOLDS, shuffle=True, random_state=RNG_SEED + r).split(X_train, y_train, groups=GROUPS.loc[X_train.index])]

# pozycje lokalne (w X_train / X_dev) -> globalne (w X_git); rozdz. 9 i 12–18 używają GLOBALNYCH
cv_train_glob = [(idx_train[tr], idx_train[va]) for tr, va in cv_train]
cv_train_rep_glob = [(idx_train[tr], idx_train[va]) for tr, va in cv_train_rep]
cv_dev_glob = [(idx_dev[tr], idx_dev[va]) for tr, va in cv_dev]

wyniki_foldow = []
for fold, (tr, va) in enumerate(cv_train, start=1):
    y_fold_train, y_fold_valid = y_train.iloc[tr], y_train.iloc[va]
    wyniki_foldow.append({
        "fold": fold,
        "n train": len(tr), "n valid": len(va),
        "defaulty na train": int(y_fold_train.sum()),
        "defaulty na valid": int(y_fold_valid.sum()),
        "DR na train (%)": 100 * y_fold_train.mean(),
        "DR na valid (%)": 100 * y_fold_valid.mean(),
    })
assert len(wyniki_foldow) == N_FOLDS
display(pd.DataFrame(wyniki_foldow).round(2).style.hide(axis="index"))
print(f"Foldy: cv_train {len(cv_train)}, cv_train_rep {len(cv_train_rep)} (5×{N_REPEATS}), cv_dev {len(cv_dev)}; "
      f"wersje globalne utworzone.")

### 4.2. Schemat podziału i walidacji

In [ ]:
fig, ax = plt.subplots(figsize=(12.5, 6.8))
ax.axis("off"); ax.set_xlim(0, 100); ax.set_ylim(0, 67)
n = len(y_raw)
szer = lambda k: 100 * k / n
KOL = {"train": "#2a78d6", "valid": "#eb6834", "test": "#1baf7a"}

def pole(x, y, w, h, tekst, kolor, alpha=0.20, bold=False, fs=9.5):
    ax.add_patch(Rectangle((x + 0.25, y), w - 0.5, h, facecolor=kolor, alpha=alpha, edgecolor="none"))
    ax.text(x + w / 2, y + h / 2, tekst, ha="center", va="center", fontsize=fs,
            fontweight="bold" if bold else "normal", linespacing=1.35)

def strzalka(x0, y0, x1, y1):
    ax.add_patch(FancyArrowPatch((x0, y0), (x1, y1), arrowstyle="-|>", mutation_scale=13, lw=1.3, color="grey"))

opis_grup = (f"klastry podmiotów powiązanych (zawsze w jednym zbiorze): {int((GROUPS.value_counts() > 1).sum())}"
             if GROUPS is not None else "bez grup podmiotów")
pole(0, 59, 100, 6.5, f"Cały zbiór: N = {n}, defaulty = {int(y_raw.sum())} ({y_raw.mean():.2%})   |   "
     f"brak dat → brak podziału out-of-time   |   {opis_grup}", "grey", 0.22, True, 8.6)
strzalka(50, 58.6, 50, 55.1)
ax.text(51.2, 56.9, f"podział {SPLIT_MODE}, stratyfikacja po default, seed = {RNG_SEED}", fontsize=9, va="center")

x0, lewe = 0, {}
for nazwa, yy, etykieta in [("train", y_train, "TRAIN"), ("valid", y_valid, "VALIDATION"), ("test", y_test, "TEST")]:
    w = szer(len(yy))
    pole(x0, 45.5, w, 9.0, f"{etykieta}\nn = {len(yy)}\ndefaulty = {int(yy.sum())} ({yy.mean():.2%})", KOL[nazwa], 0.28, True, 9.8)
    lewe[nazwa] = (x0, w); x0 += w

xt, wt = lewe["train"]
ax.text(xt + 0.3, 43.2, f"{'StratifiedGroupKFold' if GROUPS is not None else 'StratifiedKFold'} – "
        f"{N_FOLDS} foldów wewnątrz TRAIN", fontsize=9.5, fontweight="bold", va="center")
fw = (wt - 7) / N_FOLDS
for k in range(N_FOLDS):
    yk = 37.8 - k * 3.6
    ax.text(xt + 0.3, yk + 1.5, f"fold {k + 1}", fontsize=8.5, va="center")
    for j in range(N_FOLDS):
        walid = j == k
        pole(xt + 7 + j * fw, yk, fw, 3.0, "walidacja" if walid else "uczenie",
             KOL["valid"] if walid else KOL["train"], 0.45 if walid else 0.16, False, 8.5)
ax.text(xt + 0.3, 20.4, "W każdym foldzie CAŁY pipeline (czyszczenie → binning/WoE →\nselekcja cech → model) "
        f"jest dopasowywany na {N_FOLDS-1}/{N_FOLDS} TRAIN\ni oceniany na pozostałej 1/{N_FOLDS}.",
        fontsize=9, va="center", linespacing=1.4)

xv, wv = lewe["valid"]; xs, ws = lewe["test"]
pole(xv, 19.0, wv, 23.0, f"kalibracja PD:\nkształt (Platt / beta)\n+ poziom: średnie PD = {TARGET_PD:.0%}\n\n"
     "progi i klasy\nratingowe\n\n(opcjonalnie)\nearly stopping", KOL["valid"], 0.12, False, 9)
pole(xs, 19.0, ws, 23.0, "wyłącznie\nkońcowa ocena\nobu modeli\n\nbez strojenia,\nbez kalibracji,\nbez wyboru cech,\nbez EDA",
     KOL["test"], 0.12, False, 9)

kroki = [("1. CV na TRAIN", "wybór cech, binningu\ni hiperparametrów", "train"),
         ("2. Uczenie na całym TRAIN", "finalne parametry\ntransformacji i modeli", "train"),
         ("3. VALIDATION", "kalibracja do 4%,\nprogi, klasy ratingowe", "valid"),
         ("4. TEST", "jednorazowa\nocena końcowa", "test")]
odstep = 3.6
wk = (100 - odstep * (len(kroki) - 1)) / len(kroki)
for i, (tytul, opis, z) in enumerate(kroki):
    xk = i * (wk + odstep)
    pole(xk, 2.5, wk, 10.5, "", KOL[z], 0.16)
    ax.text(xk + wk / 2, 10.2, tytul, ha="center", va="center", fontsize=9.5, fontweight="bold")
    ax.text(xk + wk / 2, 6.0, opis, ha="center", va="center", fontsize=9, linespacing=1.35)
    if i < len(kroki) - 1:
        strzalka(xk + wk + 0.2, 7.7, xk + wk + odstep - 0.2, 7.7)
ax.text(0.3, 14.8, "Kolejność prac", fontsize=9.5, fontweight="bold", va="center")
savefig("schemat_walidacji")
plt.show()

## 5. Kontrola data leakage

Sprawdzamy, czy: (1) ta sama obserwacja nie występuje w dwóch zbiorach, (2) żadna cecha nie zawiera informacji z przyszłości, (3) transformacje i selekcja cech nie korzystają z danych spoza zbioru treningowego (test permutacji w 9.2).

### 5.1. Rozłączność zbiorów, zgodność X–y i foldy bez udziału testu

In [ ]:
from collections import Counter

assert X_raw.index.is_unique, "Indeksy w X_raw nie są unikalne."
assert X_raw.index.equals(y_raw.index)
sets = {
    "train": set(X_train.index),
    "val": set(X_valid.index),
    "test": set(X_test.index),
    "dev": set(X_dev.index),
}

assert sets["train"].isdisjoint(sets["val"])
assert sets["train"].isdisjoint(sets["test"])
assert sets["val"].isdisjoint(sets["test"])
assert sets["train"] | sets["val"] | sets["test"] == set(X_raw.index)
assert sets["train"] | sets["val"] == sets["dev"]
assert sets["dev"].isdisjoint(sets["test"])
assert len(X_train) + len(X_valid) + len(X_test) == len(X_raw)
assert set(X_train.columns) == set(X_valid.columns) == set(X_test.columns) == set(X_raw.columns)

for name, X, y in [
    ("train", X_train, y_train), ("validation", X_valid, y_valid),
    ("dev", X_dev, y_dev), ("test", X_test, y_test),
]:
    assert X.index.equals(y.index), f"Złe dopasowanie X/y: {name}"
    pd.testing.assert_frame_equal(X, X_raw.loc[X.index])
    pd.testing.assert_series_equal(y, y_raw.loc[y.index])


def check_cv_folds(folds, X, y, label):
    """Pozycje tr/va są lokalne dla X; sprawdzamy również grupy i pokrycie walidacją."""
    assert len(folds) == N_FOLDS, f"Niewłaściwa liczba foldów: {label}"
    universe = set(range(len(X)))
    all_valid = []
    for k, (tr, va) in enumerate(folds, 1):
        tr, va = np.asarray(tr), np.asarray(va)
        assert len(tr) > 0 and len(va) > 0
        assert np.issubdtype(tr.dtype, np.integer) and np.issubdtype(va.dtype, np.integer)
        assert len(set(tr)) == len(tr) and len(set(va)) == len(va)
        assert set(tr).isdisjoint(set(va)), f"Train i valid nachodzą na siebie: {label}/{k}"
        assert set(tr) | set(va) == universe, f"Fold nie pokrywa X: {label}/{k}"
        assert y.iloc[tr].nunique() == 2 and y.iloc[va].nunique() == 2, (
            f"Brakuje klasy 0 lub 1: {label}/{k}"
        )
        assert sets["test"].isdisjoint(set(X.index[tr]) | set(X.index[va]))
        if GROUPS is not None:
            gt, gv = set(GROUPS.loc[X.index[tr]]), set(GROUPS.loc[X.index[va]])
            assert gt.isdisjoint(gv), f"Te same grupy w dwóch częściach foldu {label}/{k}"
        all_valid.extend(va.tolist())
    counts = Counter(all_valid)
    assert set(counts) == universe, f"Nie wszystkie obserwacje walidowane: {label}"
    assert all(n == 1 for n in counts.values()), f"Rekord pojawił się wielokrotnie w walidacji: {label}"
    print(f"OK: {label} – {N_FOLDS} foldów, pełne pokrycie, każda obserwacja dokładnie raz w walidacji.")

check_cv_folds(cv_train, X_train, y_train, "cv_train")
check_cv_folds(cv_dev, X_dev, y_dev, "cv_dev")

# Sprawdzenie rzeczywistych proporcji: strict tylko przy niezgrupowanym podziale.
if GROUPS is None:
    assert abs(len(X_test)/len(X_raw) - TEST_SIZE) < 1 / len(X_raw) + 1e-12
    assert abs(len(X_valid)/len(X_raw) - (1-TEST_SIZE)*VALID_SIZE) < 1/len(X_raw) + 1e-12

print("OK: rozłączność train/validation/test, zgodność X–y, dane w CV bez udziału testu.")

### 5.2. Identyczne obserwacje w różnych zbiorach

Sprawdzamy, czy między train, validation i test nie powtarzają się identyczne wektory cech ani odciski z 2.2.

In [ ]:
zbiory = {"train": X_train, "val": X_valid, "test": X_test}
row_hash = {k: pd.util.hash_pandas_object(v, index=False) for k, v in zbiory.items()}
raport_przeciekow = []
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    common_rows = set(row_hash[a]) & set(row_hash[b])
    common_fp = set(fp.loc[zbiory[a].index]) & set(fp.loc[zbiory[b].index])
    common_grp = (set(GROUPS.loc[zbiory[a].index]) & set(GROUPS.loc[zbiory[b].index])) if GROUPS is not None else set()
    raport_przeciekow.append({"para": f"{a}–{b}", "wspólne wektory X": len(common_rows),
                              "wspólne odciski": len(common_fp), "wspólne grupy podmiotów": len(common_grp)})
    assert not common_rows and not common_fp and not common_grp, f"Przeciek między {a} i {b}"
display(pd.DataFrame(raport_przeciekow).set_index("para"))
print("OK: brak wspólnych wektorów cech, odcisków i grup podmiotów między zbiorami.")

### 5.3. Jednowymiarowa siła cech (TRAIN)

Dla każdej cechy liczbowej liczymy na train:
* AUC, siłę AUC = max(AUC, 1 − AUC) i kierunek zależności (`↑ ryzyko` – wyższa wartość, wyższe ryzyko),
* IV z binningu monotonicznego.

Alarm (wymagane sprawdzenie pod kątem przecieku): IV > 0,5 lub siła AUC ≥ 0,75. Alarm nie dotyczy cech usuwanych z powodu braków (≥ 50%), bo ich AUC jest liczone na kilkudziesięciu firmach.

In [ ]:
Xtr_raw = X_train.copy()                      # surowe cechy TRAIN (bez targetu)
Xtr_feat = X_git.loc[X_train.index].copy()    # cechy surowe + bezstanowe TRAIN
ytr = y_train.copy()
assert Xtr_feat.index.equals(ytr.index) and TARGET not in Xtr_feat.columns

def univ_auc(s, y):
    s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
    m = s.notna()
    if m.sum() <= 20 or s[m].nunique() <= 1 or y[m].nunique() < 2:
        return np.nan
    return float(roc_auc_score(y[m], s[m]))

num_for_scan = [c for c in NUM_BLACK if c in Xtr_feat.columns]
assert len(num_for_scan) == len(NUM_BLACK)
uni = pd.DataFrame({"AUC": {c: univ_auc(Xtr_feat[c], ytr) for c in num_for_scan}})
uni["siła AUC"] = np.maximum(uni["AUC"], 1 - uni["AUC"])
uni["kierunek"] = np.select([uni["AUC"].isna(), uni["AUC"] >= 0.5], ["n/d", "↑ ryzyko"], "↓ ryzyko")
uni["IV"] = {c: fit_numeric_bins(Xtr_feat[c], ytr, BIN["max_bins"], BIN["min_bin_frac"],
                                 smoothing=BIN["smoothing"])["iv"] for c in num_for_scan}
uni["NaN (%)"] = 100 * Xtr_feat[num_for_scan].isna().mean()
uni["inf (%)"] = 100 * np.isinf(Xtr_feat[num_for_scan].astype(float)).mean()
uni["wsk. sektorowy"] = uni.index.isin(SUSPICIOUS)
uni["usuwana (braki)"] = uni["NaN (%)"] >= 100 * CLEAN["max_missing"]
uni["alarm"] = ~uni["usuwana (braki)"] & ((uni["IV"] > IV_SUSPICIOUS) | (uni["siła AUC"] >= AUC_ALARM))
uni = uni.sort_values("siła AUC", ascending=False)
assert uni["AUC"].dropna().between(0, 1).all()

print("15 najsilniejszych cech jednowymiarowo (TRAIN; bez cech usuwanych z powodu braków):")
display(uni[~uni["usuwana (braki)"]].head(15).round(4))
print("Cechy usuwane z powodu braków (AUC liczone na nielicznych firmach):")
display(uni[uni["usuwana (braki)"]].round(4))
print(f"Sygnały alarmowe (IV > {IV_SUSPICIOUS} lub siła AUC >= {AUC_ALARM}):")
display(uni[uni["alarm"]].round(4))

### 5.4. Wskaźniki sektorowe `wsk_index_*`

`wsk_index_nowe` ma IV > 0,5, a `wsk_index_nowe` i `wsk_index_skr_zaw` są najsilniejszymi cechami jednowymiarowymi. Sprawdzamy, czy wartości tych wskaźników są wspólne dla firm z tym samym kodem PKD.

In [ ]:
I_COLS = SUSPICIOUS
t = Xtr_raw[["pkdKod"] + I_COLS].assign(default=ytr)
t["trojka"] = t.groupby(I_COLS).ngroup()      # numer unikalnej kombinacji (aktywne, skr_zaw, nowe)

pkd = t.groupby("pkdKod")["trojka"].agg(firmy="size", trojki="nunique").query("firmy >= 2")
wsp = t.groupby("trojka")["pkdKod"].agg(firmy="size", kody="nunique").query("firmy >= 2")

print(f"Kody PKD z >= 2 firmami: {len(pkd)}; jedna wspólna trójka w {(pkd['trojki'] == 1).mean():.1%} z nich.")
print(f"Różnych trójek: {t['trojka'].nunique()} na {len(t)} firm; firmy dzielące trójkę z inną: {wsp['firmy'].sum()}.")
print(f"Wspólne trójki: {len(wsp)}; w więcej niż jednym kodzie PKD: {(wsp['kody'] > 1).sum()}.")
display(t[I_COLS].corr(method="spearman").round(3))

top_pkd = t["pkdKod"].mode()[0]
print(f"Najliczniejszy kod PKD {top_pkd} – największe klastry wartości:")
display(t[t["pkdKod"] == top_pkd].groupby(I_COLS)["default"]
        .agg(firmy="size", defaulty="sum").nlargest(8, "firmy"))

**Wynik.** W obrębie jednego kodu PKD występuje kilka różnych zestawów wartości `wsk_index_*`, a żaden zestaw nie występuje w więcej niż jednym kodzie PKD. Wskaźniki są więc przypisane do branży, ale nie są stałe w jej obrębie. Okresu, za który zostały policzone, nie da się ustalić z danych.

**Decyzja:** wskaźniki zostają w modelu (`DROP_SUSPICIOUS = False`), pod warunkiem że opis zmiennych potwierdzi pomiar sprzed daty oceny. Ich wpływ na wyniki sprawdza test wrażliwości w 9.3. Jeśli pomiar dotyczy okresu po dacie oceny, należy ustawić `DROP_SUSPICIOUS = True` i uruchomić notebook ponownie.

In [ ]:
DROP_SUSPICIOUS = False   # ręczna decyzja po weryfikacji okresu pomiaru wsk_index_* w opisie zmiennych

if DROP_SUSPICIOUS:
    NUM_INTERP = [c for c in NUM_INTERP if c not in SUSPICIOUS]
    NUM_BLACK = [c for c in NUM_BLACK if c not in SUSPICIOUS]
assert TARGET not in NUM_INTERP + NUM_BLACK
assert len(NUM_INTERP) == len(set(NUM_INTERP)) and len(NUM_BLACK) == len(set(NUM_BLACK))
print("Usunięcie wsk_index_*:", DROP_SUSPICIOUS,
      "| kandydaci liczbowi: interpretowalny", len(NUM_INTERP), "| black-box", len(NUM_BLACK))

### 5.5. Podsumowanie kontroli leakage

| Etap | Dane | Reguła | Kontrola |
|---|---|---|---|
| Kontrola wejścia (rozdz. 2) | cały plik | tylko opis, bez decyzji modelowych | – |
| Podział (rozdz. 4) | `X_raw`, `y_raw` | warstwowo, z ustalonym ziarnem | asercje 5.1, kontrola 5.2 |
| Transformacje bezstanowe | pojedynczy wiersz | `log1p`, `signed_log1p`, dział PKD | brak statystyk z próby |
| Czyszczenie, capping, imputacja, binning, WoE, selekcja | train lub fold treningowy | parametry tylko w `.fit()` | test permutacji 9.2 |
| Cechy z przyszłości | opis zmiennych | moment dostępności cechy (rozdz. 1) | 5.3–5.4, test wrażliwości 9.3 |
| Strojenie | `cv_train` | pipeline dopasowywany od nowa w każdym foldzie | asercje 5.1 |
| Kalibracja, progi, klasy ratingowe | validation | nigdy na train ani test | – |
| Ocena końcowa | test | jednorazowo | – |

Do potwierdzenia w opisie zmiennych: okres pomiaru `wsk_index_*`, definicja defaultu i horyzont.

## 6. Jakość danych

Wszystkie statystyki są liczone na zbiorze treningowym. Rozdział opisuje problemy w danych i przyjęte reguły; transformacje wykonuje `FinancialCleaner` w `fit()` (zgodność sprawdza 6.6).

### 6.1. Profil jakości zmiennych liczbowych

In [ ]:
quality_cols = RATIO_COLS + LEVEL_COLS
Xtr_q = Xtr_raw[quality_cols].astype(float)
inf_tr = np.isinf(Xtr_q)
Xtr = Xtr_q.mask(inf_tr)                      # wartości skończone (inf -> NaN tylko na potrzeby statystyk)
n_fin = Xtr.notna().sum()
most_common_share = Xtr_q.apply(lambda s: s.value_counts(normalize=True, dropna=True).iloc[0] if s.notna().any() else np.nan)

dq = pd.DataFrame({
    "NaN (%)": 100 * Xtr_q.isna().mean(),
    "inf (%)": 100 * inf_tr.mean(),
    "skończone": n_fin,
    "zera (%)": 100 * (Xtr == 0).sum().div(n_fin.replace(0, np.nan)),
    "dominanta (%)": 100 * most_common_share,
    "unikalne": Xtr.nunique(),
    "min": Xtr.min(), "p01": Xtr.quantile(0.01), "mediana": Xtr.median(),
    "p99": Xtr.quantile(0.99), "max": Xtr.max(), "skosnosc": Xtr.skew(),
})
dq["usuniecie: braki"] = dq["NaN (%)"] >= 100 * CLEAN["max_missing"]
dq["usuniecie: quasi-stala"] = ~dq["usuniecie: braki"] & (dq["dominanta (%)"] >= 100 * CLEAN["quasi_const"])

assert dq.index.equals(Xtr_q.columns)
assert dq["NaN (%)"].between(0, 100).all() and dq["dominanta (%)"].dropna().between(0, 100).all()
assert (dq["skończone"] + Xtr_q.isna().sum() + inf_tr.sum() == len(Xtr_q)).all()
mq = dq[["p01", "mediana", "p99"]].notna().all(axis=1)
assert (dq.loc[mq, "p01"] <= dq.loc[mq, "mediana"]).all() and (dq.loc[mq, "mediana"] <= dq.loc[mq, "p99"]).all()

print(f"Zmienne liczbowe: {len(quality_cols)} | obserwacje train: {len(Xtr_q)}")
print(f"NaN na train: {int(Xtr_q.isna().sum().sum())} | ±inf na train: {int(inf_tr.sum().sum())}")
print(f"Do usunięcia – braki >= {CLEAN['max_missing']:.0%}: {int(dq['usuniecie: braki'].sum())}; "
      f"quasi-stałe (dominanta >= {CLEAN['quasi_const']:.0%}): {int(dq['usuniecie: quasi-stala'].sum())}; "
      f"stałe: {int((dq['unikalne'] <= 1).sum())}")
display(dq.loc[(dq["NaN (%)"] > 0) | (dq["inf (%)"] > 0),
               ["NaN (%)", "inf (%)", "skończone", "dominanta (%)", "mediana", "p99", "max"]]
        .sort_values(["NaN (%)", "inf (%)"], ascending=False).round(3))
print("Cechy quasi-stałe (dominanta >= 99%):")
display(dq.loc[dq["usuniecie: quasi-stala"], ["skończone", "zera (%)", "dominanta (%)", "unikalne"]]
        .sort_values("dominanta (%)", ascending=False).round(3))

miss = dq["NaN (%)"][dq["NaN (%)"] > 0].sort_values()
infs = dq["inf (%)"][dq["inf (%)"] > 0].sort_values()
fig, axes = plt.subplots(1, 2, figsize=(13, 0.35 * max(len(miss), len(infs)) + 1.6))
miss.plot.barh(ax=axes[0], color="#d95f02"); axes[0].set_xlabel("% NaN (train)"); axes[0].set_title("Braki danych (NaN)")
infs.plot.barh(ax=axes[1], color="#7570b3"); axes[1].set_xlabel("% ±inf (train)"); axes[1].set_title("Wartości nieskończone")
savefig("dq_braki_inf")
plt.show()

Siedem zmiennych z rachunku przepływów pieniężnych ma 96–99% braków, a dziewięć pozycji kwotowych ma ≥ 99% zer – wszystkie zostaną usunięte. Pozostałe braki i wartości nieskończone dotyczą trzech wskaźników (6.2).

### 6.2. Wartości specjalne `±inf` i `NaN`

Sprawdzamy, czy `+inf` i `NaN` we wskaźnikach wynikają z zerowego mianownika, i porównujemy stopę defaultu (z 95% przedziałem Wilsona) w tych grupach.

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n; d = 1 + z**2 / n
    c = (p + z**2 / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d
    return c - h, c + h

MIANOWNIK = {   # wskaźnik -> pozycja bilansowa w mianowniku (zweryfikowane poniżej na danych)
    "wsk_plynnosc_biez_1": "Zobowiazania_krotkoterminowe",
    "wsk_kapital_do_aktywa": "Aktywa",
    "wsk_struktura_kap_wlasnego_s_2": "Aktywa_trwale",
}
wiersze = []
for w, mian in MIANOWNIK.items():
    x = Xtr_raw[w].astype(float); m0 = Xtr_raw[mian].eq(0)
    for nazwa, maska in [("+inf", np.isposinf(x)), ("NaN", x.isna()), ("wartość skończona", np.isfinite(x))]:
        k, n_ = int(ytr[maska].sum()), int(maska.sum())
        lo, hi = wilson(k, n_)
        wiersze.append({"wskaźnik": w, "wartość": nazwa, "n": n_,
                        "% z mianownikiem = 0": 100 * m0[maska].mean() if n_ else np.nan,
                        "DR": k / n_ if n_ else np.nan, "DR 95% CI": f"[{lo:.3f}; {hi:.3f}]" if n_ else ""})
display(pd.DataFrame(wiersze).set_index(["wskaźnik", "wartość"]).round(4))
for w, mian in MIANOWNIK.items():
    x = Xtr_raw[w].astype(float)
    assert Xtr_raw.loc[np.isposinf(x), mian].eq(0).all(), f"{w}: +inf bez zerowego mianownika"
    assert Xtr_raw.loc[x.isna(), mian].eq(0).all(), f"{w}: NaN bez zerowego mianownika"
print("OK: każde +inf i każde NaN w tych wskaźnikach odpowiada zerowemu mianownikowi.")

**Wniosek.** Wszystkie `+inf` i `NaN` w tych wskaźnikach odpowiadają zerowemu mianownikowi, np. `+inf` w `wsk_plynnosc_biez_1` oznacza brak zobowiązań krótkoterminowych (29% train). DR firm z `+inf` płynności (7,2%) jest wyższy niż firm z wartością skończoną (5,4%), dlatego wartości specjalnych nie traktujemy jak zwykłych braków ani jak najwyższej płynności:
* karta WoE: osobne przedziały `+INF` i `BRAK` z własnym WoE,
* black-box: `+inf` → górna granica cappingu + flaga `__inf`, `NaN` → mediana + flaga `__isna`.

### 6.3. Spójność księgowa i wartości niemożliwe

Sprawdzamy tożsamości bilansowe (dla aktywów ≠ 0) oraz ujemne wartości w pozycjach, które powinny być nieujemne. Odchylenie tożsamości > 1% oznacza rekord do weryfikacji.

In [ ]:
required_balance = ["Aktywa", "Aktywa_trwale", "Aktywa_obrotowe", "Kapital_wlasny", "Zobowiazania_rezerwy"]
assert set(required_balance) <= set(Xtr_raw.columns)
fin = Xtr_raw[required_balance].astype(float).replace([np.inf, -np.inf], np.nan)
valid_1 = fin[["Aktywa", "Aktywa_trwale", "Aktywa_obrotowe"]].notna().all(axis=1) & fin["Aktywa"].ne(0)
valid_2 = fin[["Aktywa", "Kapital_wlasny", "Zobowiazania_rezerwy"]].notna().all(axis=1) & fin["Aktywa"].ne(0)
gap1 = ((fin["Aktywa"] - fin["Aktywa_trwale"] - fin["Aktywa_obrotowe"]).abs() / fin["Aktywa"].abs()).where(valid_1)
gap2 = ((fin["Aktywa"] - fin["Kapital_wlasny"] - fin["Zobowiazania_rezerwy"]).abs() / fin["Aktywa"].abs()).where(valid_2)
bilanse = pd.DataFrame({
    "tożsamość": ["Aktywa = trwałe + obrotowe", "Aktywa = kapitał własny + zobowiązania i rezerwy"],
    "sprawdzonych": [int(valid_1.sum()), int(valid_2.sum())],
    "pominiętych (aktywa = 0)": [int((~valid_1).sum()), int((~valid_2).sum())],
    "odchylenie > 1%": [int((gap1 > 0.01).sum()), int((gap2 > 0.01).sum())],
    "DR przy odchyleniu > 1%": [ytr[gap1 > 0.01].mean(), ytr[gap2 > 0.01].mean()],
    "max odchylenie (%)": [100 * gap1.max(), 100 * gap2.max()],
}).set_index("tożsamość")
assert (bilanse["sprawdzonych"] + bilanse["pominiętych (aktywa = 0)"] == len(Xtr_raw)).all()
display(bilanse.round(3))

nonneg_candidates = [c for c in LEVEL_COLS if c.startswith(
    ("Aktywa", "Zapasy", "Srodki_", "Zobowiazania", "Kredyty", "Naleznosci", "Kapital_podstawowy", "przychody"))]
negative_counts = (Xtr[nonneg_candidates] < 0).sum().sort_values(ascending=False)
print("Ujemne wartości w pozycjach, które powinny być nieujemne (train):")
display(negative_counts[negative_counts > 0].to_frame("liczba ujemnych"))
anomalie = pd.DataFrame({
    "warunek": ["Aktywa = 0", "Aktywa < 0", "Kapitał własny < 0", "wiek firmy <= 0 dni",
                "jakakolwiek ujemna pozycja nieujemna"],
    "liczba": [int(fin["Aktywa"].eq(0).sum()), int(fin["Aktywa"].lt(0).sum()), int(fin["Kapital_wlasny"].lt(0).sum()),
               int((Xtr_raw["wsk_liczba_dni_istnienia"] <= 0).sum()), int((Xtr[nonneg_candidates] < 0).any(axis=1).sum())],
}).set_index("warunek")
anomalie["DR"] = [ytr[fin["Aktywa"].eq(0)].mean(), ytr[fin["Aktywa"].lt(0)].mean(), ytr[fin["Kapital_wlasny"].lt(0)].mean(),
                  ytr[Xtr_raw["wsk_liczba_dni_istnienia"] <= 0].mean(), ytr[(Xtr[nonneg_candidates] < 0).any(axis=1)].mean()]
display(anomalie.round(4))

**Reguły dla anomalii**

| Anomalia | Liczba (train) | Decyzja |
|---|---|---|
| Aktywa = 0 | 31 | zostają; wskaźniki z aktywami w mianowniku mają wartości specjalne obsłużone jak w 6.2 |
| Odchylenie tożsamości bilansowej > 1% | 48 / 42 | zostają bez korekty; DR tych grup (2,1% / 7,1%) nie odbiega wyraźnie od średniej |
| Ujemne kwoty w pozycjach nieujemnych | 15 | zostają; w black-boxie `signed_log1p` zachowuje znak, karta nie używa pozycji kwotowych |
| Aktywa < 0 | 0 (1 w pliku) | `log_aktywa` = NaN |

Obserwacji nie usuwamy, bo ta sama reguła musiałaby być stosowana przy ocenie nowych firm.

### 6.4. Ogony rozkładów i wartości odstające

Miara: odległość wartości minimalnej i maksymalnej od p01 / p99, wyrażona w szerokościach przedziału [p01, p99].

In [ ]:
spread = (dq["p99"] - dq["p01"]).replace(0, np.nan)
xtail = pd.DataFrame({"górny ogon": (dq["max"] - dq["p99"]) / spread,
                      "dolny ogon": (dq["p01"] - dq["min"]) / spread})
xtail["max miara"] = xtail.max(axis=1)
xtail = xtail.sort_values("max miara", ascending=False)
display(xtail.head(12).round(1))
print("Cechy z ogonem > 10 szerokości p01–p99:", int((xtail["max miara"] > 10).sum()),
      "| cechy o zerowej szerokości p01–p99:", int((dq["p99"] == dq["p01"]).sum()))
print("Polityka: black-box – capping do [p01, p99] z TRAIN; karta – binning kwantylowy (odporny na ogony).")

### 6.5. Zmienne kategoryczne

Liczba poziomów oraz liczba poziomów spełniających próg `MIN_CAT_SHARE` (5% train); rzadsze poziomy trafiają do `INNE`.

In [ ]:
cat_audit_cols = list(dict.fromkeys(CAT_RAW + CAT_COLS))
min_cat_n = int(np.ceil(MIN_CAT_SHARE * len(Xtr_feat)))
cat_report = pd.DataFrame({
    "liczba poziomów": Xtr_feat[cat_audit_cols].nunique(dropna=True),
    "braki": Xtr_feat[cat_audit_cols].isna().sum(),
    "udział najczęstszego (%)": [100 * Xtr_feat[c].value_counts(normalize=True).iloc[0] for c in cat_audit_cols],
    f"poziomy >= {min_cat_n} obs.": [(Xtr_feat[c].value_counts() >= min_cat_n).sum() for c in cat_audit_cols],
})
display(cat_report.round(1))
print(f"Spójność schematów: schemat bilansu = schemat RZiS w "
      f"{(Xtr_feat['schemat_wsk_bilans'] == Xtr_feat['schemat_wsk_rzis']).mean():.1%} rekordów.")

### 6.6. Kontrola `FinancialCleaner`

Sprawdzamy, czy `FinancialCleaner` usuwa te same cechy, które wskazały 6.1 i 2.3, oraz czy wynik w trybie black-box nie zawiera `NaN` ani `inf`.

In [ ]:
test_cleaner = FinancialCleaner(**CLEAN, impute=True, inf_policy="cap").fit(Xtr_feat[NUM_BLACK])
Xb = Xtr_feat[NUM_BLACK].astype(float)
manual_missing = Xb.isna().mean()
assert set(test_cleaner.dropped_["za_duzo_brakow"]) == set(manual_missing[manual_missing >= CLEAN["max_missing"]].index)
rest = [c for c in NUM_BLACK if c not in test_cleaner.dropped_["za_duzo_brakow"]]
manual_dom = Xb[rest].apply(lambda s: s.value_counts(normalize=True).iloc[0] if s.notna().any() else 1.0)
assert set(test_cleaner.dropped_["quasi_stale"]) == set(manual_dom[manual_dom >= CLEAN["quasi_const"]].index)
dup_pairs = pd.DataFrame({"duplikat": DUPLICATES, "odpowiada kolumnie": cechy_zduplikowane})
dup_in_scope = dup_pairs[dup_pairs["duplikat"].isin(NUM_BLACK) & dup_pairs["odpowiada kolumnie"].isin(NUM_BLACK)]
assert set(dup_in_scope["duplikat"]) <= set(test_cleaner.dropped_["duplikat"]), "Duplikat z pliku nie został usunięty."
Z_clean_test = test_cleaner.transform(Xtr_feat[NUM_BLACK])
assert Z_clean_test.index.equals(X_train.index) and Z_clean_test.columns.is_unique
assert np.isfinite(Z_clean_test.to_numpy(dtype=float)).all()
print("Usunięte za braki:", len(test_cleaner.dropped_["za_duzo_brakow"]))
print("Usunięte quasi-stałe:", len(test_cleaner.dropped_["quasi_stale"]))
print("Usunięte duplikaty kolumn (na train):", test_cleaner.dropped_["duplikat_kolumny"])
extra_dup = sorted(set(test_cleaner.dropped_["duplikat"]) - set(dup_in_scope["duplikat"]))
print("Duplikaty wykryte na train, a nieobecne w całym pliku:", extra_dup or "brak")
print("Flagi __isna:", test_cleaner.nan_ind_cols_, "| flagi __inf:", test_cleaner.inf_ind_cols_)
print("Kolumny po FinancialCleaner (z flagami):", Z_clean_test.shape[1])
print("OK: polityka czyszczenia zgodna z niezależnym audytem train; wynik bez NaN i inf.")

Na train usuwany jest dodatkowo `slog_Aktywa`, identyczny z `log_aktywa` – jedyna firma z ujemnymi aktywami (dla której te kolumny się różnią) nie trafiła do train.

## 7. EDA

Analiza obejmuje wyłącznie zbiór treningowy.

### 7.1. Zmienna celu

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.7))
ytr.value_counts().reindex([0, 1], fill_value=0).plot.bar(ax=ax[0], color=["#4daf4a", "#e41a1c"])
ax[0].set_xticklabels(["brak defaultu (0)", "default (1)"], rotation=0)
ax[0].set_title("Rozkład targetu – TRAIN"); ax[0].set_ylabel("Liczba obserwacji")
ax[1].bar(["DR train", "kotwica PD"], [ytr.mean(), TARGET_PD], color=["#377eb8", "#999999"])
ax[1].set_ylabel("Udział defaultu"); ax[1].set_ylim(0, max(ytr.mean(), TARGET_PD) * 1.35)
ax[1].set_title("DR próby a kotwica kalibracji")
savefig("eda_target")
plt.show()
print(f"TRAIN: {len(ytr)} obserwacji, {int(ytr.sum())} defaultów, DR = {ytr.mean():.2%}; "
      f"różnica do kotwicy {TARGET_PD:.0%}: {100*(ytr.mean() - TARGET_PD):+.2f} p.p.")

### 7.2. Zależności z targetem i rozkłady cech

Ranking według AUC z 5.3 (na górze cechy, których wyższa wartość oznacza wyższe ryzyko) oraz korelacja Spearmana z targetem. Wykresy pokazują rozkłady 8 cech o najwyższym AUC w klasach default / brak defaultu (przycięte do p01–p99 dla czytelności).

In [ ]:
from scipy import stats

ranking = uni.dropna(subset=["AUC"]).sort_values("AUC", ascending=False)
top = [c for c in ranking.index if c in Xtr_feat.columns][:8]
assert len(top) > 0, "Brak cech z obliczalnym jednowymiarowym AUC."

def safe_spearman(x, y):
    x = pd.to_numeric(x, errors="coerce").replace([np.inf, -np.inf], np.nan)
    m = x.notna() & y.notna()
    if m.sum() < 3 or x[m].nunique() < 2 or y[m].nunique() < 2:
        return np.nan
    return float(stats.spearmanr(x[m], y[m]).statistic)

sp = pd.Series({c: safe_spearman(Xtr_feat[c], ytr) for c in ranking.index[:30]}, name="Spearman_z_default")
top_assoc = ranking.loc[sp.index, ["AUC"]].join(sp)
display(top_assoc.head(20).round(4))

fig, axes = plt.subplots(2, 4, figsize=(17, 7))
for ax, c in zip(axes.ravel(), top):
    x = pd.to_numeric(Xtr_feat[c], errors="coerce").replace([np.inf, -np.inf], np.nan)
    lo, hi = x.quantile([0.01, 0.99])
    plot_df = pd.DataFrame({"x": x.clip(lo, hi),
                            "klasa": np.where(ytr.to_numpy() == 1, "default", "non-default")}, index=x.index).dropna()
    if not plot_df.empty:
        sns.histplot(data=plot_df, x="x", hue="klasa", stat="density", common_norm=False,
                     bins=30, element="step", ax=ax,
                     palette={"default": "#e41a1c", "non-default": "#4daf4a"})
    ax.set_title(f"{c} | AUC siła={ranking.loc[c, 'AUC']:.3f}", fontsize=9)
    ax.set_xlabel("")
for ax in axes.ravel()[len(top):]:
    ax.axis("off")
fig.suptitle("TRAIN: rozkłady cech według targetu (p01–p99 tylko do wizualizacji)")
savefig("eda_top_features")
plt.show()

### 7.3. Stopa defaultu w kategoriach

DR z 95% przedziałem Wilsona dla kategorii z co najmniej 30 obserwacjami.

In [ ]:
def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z**2 / n
    c = (p + z**2 / (2*n)) / d; h = z * np.sqrt(p*(1-p)/n + z**2/(4*n**2)) / d
    return c - h, c + h

def rate_table(col, min_n=30):
    g = Xtr_feat.assign(y=ytr).groupby(col)["y"].agg(["count", "sum"]).rename(columns={"count": "n", "sum": "defaulty"})
    g = g[g["n"] >= min_n].copy()
    g["DR"] = g["defaulty"] / g["n"]
    g["CI dolny"], g["CI gorny"] = zip(*[wilson(k, n_) for k, n_ in zip(g["defaulty"], g["n"])])
    return g.sort_values("n", ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
for ax, c in zip(axes, ["pkd_div", "forma_wl", "schemat_wsk_bilans"]):
    g = rate_table(c).head(12)
    ax.bar(g.index.astype(str), g["DR"], yerr=[g["DR"] - g["CI dolny"], g["CI gorny"] - g["DR"]], capsize=3, color="#6baed6")
    ax.axhline(ytr.mean(), color="k", ls="--", lw=1, label="DR train")
    ax.set_title(f"DR: {c} (n ≥ 30)"); ax.tick_params(axis="x", rotation=60); ax.legend()
savefig("eda_dr_kategorie")
plt.show()
display(rate_table("pkd_div").head(10).round(4))
display(rate_table("schemat_wsk_bilans").round(4))

### 7.4. Korelacje Spearmana

Korelacje między wskaźnikami z co najmniej 90% wartości skończonych; |ρ| > 0,90 oznacza silną redundancję. Selekcję cech karty z uwzględnieniem korelacji wykonuje `WoEEncoder` (rozdział 8).

In [ ]:
base_corr = Xtr_feat[RATIO_COLS].replace([np.inf, -np.inf], np.nan)
base_corr = base_corr.loc[:, (base_corr.notna().mean() > 0.90) & (base_corr.nunique() > 1)]
correlation = base_corr.corr(method="spearman")
top_corr = [c for c in ranking.index if c in correlation.columns][:40]
print("Liczba analizowanych wskaźników:", len(base_corr.columns))

if len(top_corr) >= 2:
    cc = correlation.loc[top_corr, top_corr]
    fig, ax = plt.subplots(figsize=(max(6, .24 * len(top_corr) + 2), max(5, .24 * len(top_corr) + 1)))
    sns.heatmap(cc, mask=np.triu(np.ones(cc.shape, dtype=bool), 1),
                cmap="RdBu_r", center=0, vmin=-1, vmax=1, ax=ax,
                cbar_kws={"shrink": .75})
    ax.tick_params(labelsize=6)
    ax.set_title("Spearman – najsilniejsze wskaźniki (TRAIN)")
    savefig("corr_top40")
    plt.show()
else:
    print("Za mało wskaźników do macierzy korelacji.")

pairs = correlation.where(np.triu(np.ones(correlation.shape, dtype=bool), 1)).stack()
strong_pairs = pairs[pairs.abs() > 0.90].sort_values(key=abs, ascending=False)
print(f"Pary |ρ| > 0.90: {len(strong_pairs)} / {len(pairs)} dostępnych par.")
display(strong_pairs.head(15).to_frame("rho"))

### 7.5. VIF wskaźników (diagnostycznie)

VIF wskaźników po cappingu i imputacji na train oraz po usunięciu par |r| > 0,95. VIF cech karty – w 8.4.

In [ ]:
X_vif = Xtr_feat[[c for c in NUM_INTERP if c in Xtr_feat]].astype(float).replace([np.inf, -np.inf], np.nan)
X_vif = X_vif.loc[:, (X_vif.isna().mean() < 0.10) & (X_vif.nunique() > 1)]
X_vif = X_vif.clip(X_vif.quantile(0.01), X_vif.quantile(0.99), axis=1).fillna(X_vif.median())
X_vif = X_vif.loc[:, X_vif.nunique() > 1]
c_abs = X_vif.corr().abs()
upper = c_abs.where(np.triu(np.ones(c_abs.shape, dtype=bool), k=1))
to_drop_vif = [col for col in upper.columns if (upper[col] > 0.95).any()]
X_vif = X_vif.drop(columns=to_drop_vif)
R = X_vif.corr().to_numpy(dtype=float)
rank, condition = int(np.linalg.matrix_rank(R)), float(np.linalg.cond(R))
if np.isfinite(condition) and condition < 1e10 and rank == len(R):
    vif_data = pd.DataFrame({"cecha": X_vif.columns, "VIF": np.diag(np.linalg.inv(R))}).sort_values("VIF", ascending=False)
    assert (vif_data["VIF"] >= 1 - 1e-6).all()
    print(f"VIF: {len(vif_data)} wskaźników po usunięciu {len(to_drop_vif)} kolumn z |r| > 0,95; "
          f"VIF > 5: {int((vif_data.VIF > 5).sum())}, VIF > 10: {int((vif_data.VIF > 10).sum())}")
    display(vif_data.head(12).round(2))
else:
    vif_data = pd.DataFrame({"cecha": X_vif.columns, "VIF": np.nan})
    print(f"VIF niestabilny (rząd = {rank}/{len(R)}, cond = {condition:.2e}) – wymagana dalsza redukcja cech.")

### 7.6. Wielkość firmy a ryzyko

DR w kwartylach aktywów; granice kwartyli wyznaczone na train (`AKT_EDGES`).

In [ ]:
asset = Xtr_raw["Aktywa"].astype(float).replace([np.inf, -np.inf], np.nan)
q_assets = asset.quantile([0.25, 0.50, 0.75]).to_numpy(dtype=float)
AKT_EDGES = np.r_[-np.inf, np.unique(q_assets), np.inf]
AKT_LABELS = (["Małe", "Średnie-niższe", "Średnie-wyższe", "Duże"] if len(AKT_EDGES) == 5
              else [f"Grupa {i+1}" for i in range(len(AKT_EDGES) - 1)])
grupy_aktywa = (pd.DataFrame({"grupa": pd.cut(asset, bins=AKT_EDGES, labels=AKT_LABELS, include_lowest=True), "y": ytr})
                .groupby("grupa", observed=True)["y"].agg(["count", "sum", "mean"])
                .rename(columns={"count": "n", "sum": "defaulty", "mean": "DR"}))
grupy_aktywa["CI"] = [f"[{a:.3f}; {b:.3f}]" for a, b in (wilson(k, n_) for k, n_ in zip(grupy_aktywa["defaulty"], grupy_aktywa["n"]))]
assert int(grupy_aktywa["n"].sum()) == int(asset.notna().sum())
display(grupy_aktywa.round(4))
fig, ax = plt.subplots(figsize=(8, 4))
grupy_aktywa["DR"].plot.bar(ax=ax, color="#fc8d62")
ax.axhline(ytr.mean(), color="black", ls="--", label=f"DR train {ytr.mean():.2%}")
ax.set(ylabel="DR", title="DR według kwartyli aktywów – TRAIN"); ax.legend(); plt.xticks(rotation=0)
savefig("eda_dr_aktywa")
plt.show()
print("Granice kwartyli aktywów (TRAIN):", np.round(AKT_EDGES, 2))

### 7.7. Podsumowanie EDA

In [ ]:
print(f"Train: n = {len(ytr)}, defaulty = {int(ytr.sum())}, DR = {ytr.mean():.2%}; "
      f"kotwica PD = {TARGET_PD:.0%} (różnica {100 * (ytr.mean() - TARGET_PD):+.2f} p.p.).")

top5 = (uni[~uni["usuwana (braki)"]].dropna(subset=["AUC"])
        .sort_values("siła AUC", ascending=False).head(5))
print("Top 5 cech według siły jednowymiarowego AUC (TRAIN, bez cech usuwanych z powodu braków):")
display(top5[["AUC", "siła AUC", "kierunek", "IV", "NaN (%)"]].round(4))

print(f"Redundancja: {len(strong_pairs)} par wskaźników o |ρ| > 0,90 "
      f"(w tym {int((strong_pairs.abs() > 0.9999).sum())} par praktycznie identycznych).")
if "vif_data" in globals() and vif_data["VIF"].notna().any():
    print(f"Diagnostyczny VIF > 10: {int((vif_data['VIF'] > 10).sum())} wskaźników.")

if len(grupy_aktywa) >= 2:
    print(f"DR wg aktywów: najmniejsze firmy {grupy_aktywa['DR'].iloc[0]:.2%} "
          f"vs największe {grupy_aktywa['DR'].iloc[-1]:.2%}.")

n_inf = int(np.isposinf(Xtr_raw["wsk_plynnosc_biez_1"]).sum())
print(f"Wartości specjalne: {n_inf} firm ({n_inf / len(Xtr_raw):.1%}) bez zobowiązań krótkoterminowych "
      f"(wsk_plynnosc_biez_1 = +inf).")

## 8. Przygotowanie cech

Obie ścieżki korzystają z `ScoringFeatureBuilder` dopasowanego na train; parametry pochodzą z konfiguracji (rozdział 0).

| | `mode="woe"` – model interpretowalny | `mode="blackbox"` |
|---|---|---|
| Kandydaci | wskaźniki `wsk_*` + `log_aktywa` + 4 kategorie | wskaźniki + `slog_*` pozycji kwotowych + 4 kategorie |
| Czyszczenie | usunięcie cech z nadmiarem braków, quasi-stałych i duplikatów | to samo |
| Wartości odstające | binning kwantylowy | capping do [p01, p99] z train |
| Braki i `±inf` | osobne przedziały `BRAK` / `+INF` / `-INF` | mediana / granica cappingu + flagi `__isna` / `__inf` |
| Transformacja | binning monotoniczny + WoE | brak; `scale=True` dla SVM / sieci |
| Kategorie | rzadkie → `INNE`, potem WoE | rzadkie → `INNE`, potem one-hot |
| Selekcja | IV ≥ próg, korelacja WoE < próg, limit liczby cech | brak |

In [ ]:
COMMON = dict(cat_cols=CAT_COLS, min_cat_share=MIN_CAT_SHARE, **CLEAN)
FB_PARAMS_WOE = dict(num_cols=NUM_INTERP, mode="woe", **COMMON,
                     **{k: BIN[k] for k in ("max_bins", "min_bin_frac", "smoothing",
                                            "iv_min", "corr_max", "max_features")})
FB_PARAMS_BB = dict(num_cols=NUM_BLACK, mode="blackbox", scale=False, **COMMON)

# dopasowanie na CAŁYM zbiorze treningowym (używane w rozdziałach 12–18); w CV fit powtarzany w każdym foldzie
Xg_train = X_git.iloc[idx_train]
fb_woe = ScoringFeatureBuilder(**FB_PARAMS_WOE).fit(Xg_train, y_all[idx_train])
fb_bb  = ScoringFeatureBuilder(**FB_PARAMS_BB).fit(Xg_train, y_all[idx_train])

print("Usunięte (za dużo braków):", fb_bb.cleaner_.dropped_["za_duzo_brakow"])
print("Usunięte (quasi-stałe):   ", fb_bb.cleaner_.dropped_["quasi_stale"][:8], "...")
print("Usunięte (duplikaty):     ", fb_bb.cleaner_.dropped_["duplikat"])
print(f"Cechy po transformacji: WoE = {len(fb_woe.feature_names_)}, black-box = {len(fb_bb.feature_names_)}")

iv = fb_woe.iv_table_
wybrane = iv[iv["status"] == "wybrana"]
# status odrzuconych zawiera powód, np. "korelacja z X (|r| = 0.85)" – grupujemy po rodzaju decyzji
print("Statusy cech:", iv["status"].str.split(" \\(").str[0].value_counts().to_dict())
print("Cechy wybrane z niemonotoniczną relacją WoE (powinno być 0):",
      int((wybrane["monotoniczna"] == False).sum()))

### 8.1. Tabela IV i selekcja cech karty

In [ ]:
assert len(fb_woe.feature_names_) <= BIN["max_features"]
wybrane = iv[iv["status"] == "wybrana"]
display(wybrane.round(4))
assert wybrane["monotoniczna"].fillna(True).astype(bool).all(), "Wybrana cecha liczbowa z niemonotonicznym WoE."
print("Najsilniejsze cechy odrzucone (z powodem):")
display(iv[iv["status"] != "wybrana"].head(12).round(4))
print(f"Cechy z IV > {IV_SUSPICIOUS} (do weryfikacji – rozdz. 5.4 i 9.3):",
      iv[iv["IV"] > IV_SUSPICIOUS].index.tolist())

### 8.2. Profile binów wybranych cech

Słupki – DR w przedziale, linia – WoE (WoE > 0 oznacza ryzyko niższe od średniego). Przedziały specjalne (`BRAK`, `+INF`) są oznaczone na szaro.

In [ ]:
sel = iv[iv["status"] == "wybrana"].index[:6].tolist()
fig, axes = plt.subplots(2, 3, figsize=(16, 7))
for ax, c in zip(axes.ravel(), sel):
    p = fb_woe.encoder_.profile(c)
    kol = ["#bdbdbd" if u.startswith(("wartość", "za mało")) else "#6baed6" for u in p["uwaga"]]
    ax.bar(range(len(p)), p["DR"], color=kol)
    ax.set_xticks(range(len(p))); ax.set_xticklabels(p["przedzial"], rotation=60, ha="right", fontsize=7)
    ax.axhline(y_all[idx_train].mean(), color="k", ls=":", lw=.8); ax.set_ylabel("default rate")
    ax2 = ax.twinx(); ax2.plot(range(len(p)), p["WoE"], "o-", color="#e6550d"); ax2.set_ylabel("WoE")
    ax.set_title(f"{c}  (IV={iv.loc[c, 'IV']:.2f})", fontsize=9)
for ax in axes.ravel()[len(sel):]:
    ax.axis("off")
plt.suptitle("Biny monotoniczne: default rate (słupki) i WoE (linia)")
savefig("woe_profiles")
plt.show()
display(fb_woe.encoder_.profile(sel[0]).round(4))

### 8.3. Kierunki zależności

Oczekiwany kierunek wpływu cechy na ryzyko jest określony z góry na podstawie znaczenia wskaźnika (Lab 3) i porównany z kierunkiem z binningu oraz znakiem korelacji Spearmana.

In [ ]:
sel = wybrane.index.tolist()      # wszystkie cechy karty (używane też w 9.4 i 10)

OCZEKIWANIA = [   # (fragment nazwy, oczekiwany kierunek ryzyka, uzasadnienie); sprawdzane w tej kolejności
    ("index_skr_zaw", "inc", "więcej wykreśleń/zawieszeń firm w branży -> gorsze otoczenie"),
    ("index_nowe", "?", "napływ nowych firm: konkurencja (↑) albo dobra koniunktura (↓)"),
    ("index_aktywne", "?", "zmiana liczby aktywnych firm: kierunek niejednoznaczny"),
    ("ebit", "dec", "wyższy wynik operacyjny / EBITDA (także względem zobowiązań i odsetek) -> niższe ryzyko"),
    ("zysk", "dec", "wyższy zysk (także względem zobowiązań i odsetek) -> niższe ryzyko"),
    ("rent", "dec", "wyższa rentowność -> niższe ryzyko"),
    ("ROE", "dec", "wyższa rentowność kapitału -> niższe ryzyko"),
    ("ROA", "dec", "wyższa rentowność aktywów -> niższe ryzyko"),
    ("ROS", "dec", "wyższa rentowność sprzedaży -> niższe ryzyko"),
    ("marza", "dec", "wyższa marża -> niższe ryzyko"),
    ("pokryc", "dec", "lepsze pokrycie zobowiązań / aktywów / odsetek -> niższe ryzyko"),
    ("zadluz", "inc", "wyższe zadłużenie -> wyższe ryzyko"),
    ("kap_obc", "inc", "większy udział kapitału obcego -> wyższe ryzyko"),
    ("kapitalu_obcego", "inc", "większy udział kapitału obcego -> wyższe ryzyko"),
    ("mnoznik_kap", "inc", "aktywa / kapitał własny – dźwignia"),
    ("dzwigni", "?", "efekt dźwigni: znak zależy od relacji ROA do kosztu długu"),
    ("zob", "inc", "wyższe zobowiązania względem aktywów / przychodów -> wyższe ryzyko"),
    ("koszty", "inc", "wyższe koszty względem przychodów -> niższa marża"),
    ("odsetk", "inc", "wyższe obciążenie odsetkami -> wyższe ryzyko"),
    ("kapitalu_obrotowego", "dec", "większy kapitał obrotowy netto -> bufor płynności"),
    ("plynnosc", "dec", "wyższa płynność -> niższe ryzyko"),
    ("kap_wlasn", "dec", "większy udział kapitału własnego -> niższe ryzyko"),
    ("kapital_do_aktywa", "dec", "większy udział kapitału własnego -> niższe ryzyko"),
    ("got", "dec", "więcej gotówki / przepływów -> niższe ryzyko"),
    ("dni_istnienia", "dec", "starsze firmy -> niższe ryzyko"),
    ("log_aktywa", "dec", "większe firmy -> niższe ryzyko"),
]

def oczekiwany(nazwa):
    for frag, kier, uzas in OCZEKIWANIA:
        if frag.lower() in nazwa.lower():
            return kier, uzas
    return "?", "brak jednoznacznej hipotezy – wymaga interpretacji"

# Komentarz do kierunku każdej cechy karty (opis zależności w danych, nie przyczynowości).
OCENY_EKONOMICZNE = {
    "wsk_index_skr_zaw": "Zgodny z oczekiwaniem: wyższy udział wykreśleń i zawieszeń w branży – wyższe ryzyko. "
                         "Okres pomiaru do potwierdzenia (5.4).",
    "wsk_index_nowe": "Brak hipotezy a priori; w danych wyższa wartość – wyższe ryzyko. Okres pomiaru do potwierdzenia (5.4).",
    "wsk_struktury_kapitalu": "Brak hipotezy a priori; w danych wyższa wartość – niższe ryzyko. Wzór do potwierdzenia w opisie zmiennych.",
    "wsk_ebit": "Zgodny z oczekiwaniem: wyższy wynik operacyjny – niższe ryzyko.",
    "wsk_efekt_dzwigni_fin_1": "Brak hipotezy a priori; w danych wyższa wartość – wyższe ryzyko. Wzór do potwierdzenia w opisie zmiennych.",
    "wsk_poziom_kapitalu_obrotowego_netto": "Zgodny z oczekiwaniem: wyższy kapitał obrotowy netto – niższe ryzyko.",
    "wsk_ebitda_3": "Zgodny z oczekiwaniem: wyższa EBITDA – niższe ryzyko.",
    "wsk_stopa_zysku_sprzedaz": "Zgodny z oczekiwaniem: wyższa marża zysku na sprzedaży – niższe ryzyko.",
}

KIER = {"inc": "↑ ryzyko", "dec": "↓ ryzyko", "?": "?", "n/d": "kategoria"}
rows = []
for c in sel:
    k_bin = iv.loc[c, "kierunek_ryzyka"]
    kategoria = c in CAT_COLS
    k_exp, uzas = ("n/d", "zmienna kategoryczna – WoE poziomów") if kategoria else oczekiwany(c)
    rho = np.nan if kategoria else safe_spearman(Xtr_feat[c], ytr)

    # (1) właściwy test: oczekiwanie a priori vs kierunek z binningu
    zg_apriori = "n/d" if kategoria else ("brak hipotezy" if k_exp == "?" else ("TAK" if k_exp == k_bin else "NIE"))
    # (2) kontrola spójności: znak Spearmana vs kierunek z binningu (oba z danych)
    if kategoria or pd.isna(rho) or k_bin not in ("inc", "dec"):
        zg_spearman = "n/d"
    elif abs(rho) < 1e-10:
        zg_spearman = "brak kierunku"
    else:
        zg_spearman = "TAK" if (rho > 0) == (k_bin == "inc") else "NIE"

    rows.append({"cecha": c, "IV": iv.loc[c, "IV"],
                 "oczekiwany (a priori)": KIER[k_exp], "z binningu": KIER.get(k_bin, k_bin),
                 "zgodność a priori": zg_apriori,
                 "Spearman z default": rho, "znak Spearmana zgodny": zg_spearman,
                 "uzasadnienie a priori": uzas,
                 "ocena ekonomiczna": OCENY_EKONOMICZNE.get(c, "Wymaga indywidualnej interpretacji ekonomicznej.")})

kierunki = pd.DataFrame(rows).set_index("cecha")
assert len(kierunki) == len(wybrane), "Tabela kierunków nie obejmuje wszystkich cech karty."
display(kierunki.round(4))
print("Niezgodne z oczekiwaniem a priori:", kierunki.index[kierunki["zgodność a priori"] == "NIE"].tolist() or "brak")
print("Bez hipotezy a priori (wymagają interpretacji):",
      kierunki.index[kierunki["zgodność a priori"] == "brak hipotezy"].tolist() or "brak")
print("Niespójny znak Spearmana:", kierunki.index[kierunki["znak Spearmana zgodny"] == "NIE"].tolist() or "brak")
print("Zgodność kierunku z oczekiwaniem nie oznacza zależności przyczynowej.")

**Wniosek.** Wszystkie 5 cech z określonym oczekiwanym kierunkiem ma kierunek zgodny z oczekiwaniem, a znak korelacji Spearmana jest zgodny z binningiem dla wszystkich 8 cech. Dla `wsk_index_nowe`, `wsk_struktury_kapitalu` i `wsk_efekt_dzwigni_fin_1` kierunku nie określono z góry – ich interpretację należy potwierdzić w opisie zmiennych. W regresji logistycznej na WoE oczekujemy ujemnych współczynników β.

### 8.4. Współliniowość cech karty

Korelacje między cechami WoE karty, VIF oraz liczba defaultów na parametr (EPV).

In [ ]:
Z_tr = fb_woe.transform(Xg_train)
Rw = Z_tr.corr()
fig, ax = plt.subplots(figsize=(7.5, 6))
sns.heatmap(Rw, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1, ax=ax, annot_kws={"size": 7})
ax.set_xticklabels([c.replace("woe__", "") for c in Rw.columns], rotation=60, ha="right", fontsize=7)
ax.set_yticklabels([c.replace("woe__", "") for c in Rw.index], fontsize=7)
ax.set_title("Korelacje Pearsona cech WoE karty (TRAIN)")
savefig("woe_corr")
plt.show()
vif_woe = pd.Series(np.diag(np.linalg.inv(Rw.to_numpy())), index=[c.replace("woe__", "") for c in Rw.columns],
                    name="VIF").sort_values(ascending=False)
display(vif_woe.to_frame().round(2))
print(f"Maks. |r| między cechami karty: {Rw.where(~np.eye(len(Rw), dtype=bool)).abs().max().max():.2f} "
      f"(próg {BIN['corr_max']}); maks. VIF: {vif_woe.max():.2f} (praktyczny próg 5).")
epv = int(y_all[idx_train].sum()) / (len(fb_woe.feature_names_) + 1)
print(f"Zdarzeń na parametr karty (EPV): {epv:.1f} = {int(y_all[idx_train].sum())} defaultów / "
      f"{len(fb_woe.feature_names_) + 1} parametrów.")

### 8.5. Transformacja zbiorów

Validation i test są przekształcane wyłącznie przez `transform()` transformerów dopasowanych na train. Sprawdzamy brak `NaN` / `inf`, zgodność kolumn oraz to, że wartości WoE pochodzą z train.

In [ ]:
X_sets = {}
for name, idx in [("train", idx_train), ("val", idx_val), ("test", idx_test)]:
    Xi = X_git.iloc[idx]
    X_sets[name] = {"woe": fb_woe.transform(Xi), "bb": fb_bb.transform(Xi)}
    for kind in ("woe", "bb"):
        Z = X_sets[name][kind]
        assert Z.index.equals(Xi.index) and np.isfinite(Z.to_numpy(dtype=float)).all(), f"{name}/{kind}: NaN lub inf"
    assert list(X_sets[name]["woe"].columns) == fb_woe.feature_names_
    assert list(X_sets[name]["bb"].columns) == fb_bb.feature_names_
for c in fb_woe.feature_names_:
    learned = set(np.round(X_sets["train"]["woe"][c], 10)) | {0.0}
    for name in ("val", "test"):
        assert set(np.round(X_sets[name]["woe"][c], 10)) <= learned, f"{c}: WoE spoza wartości z train ({name})"
print({k: (v["woe"].shape, v["bb"].shape) for k, v in X_sets.items()})
display(X_sets["train"]["woe"].describe().T.round(4))

## 9. Test poprawności pipeline'u (wymaganie 3.1)

W każdym foldzie cały pipeline (czyszczenie → binning/WoE → selekcja → model) jest dopasowywany od nowa na części treningowej foldu. Modele kontrolne: `LogisticRegression(C=0.1)` na cechach WoE i `HistGradientBoostingClassifier`. Nie są to modele finalne – służą do sprawdzenia przygotowania cech.

In [ ]:
def run_cv(params, model_factory, folds_glob, y, label, X=X_git):
    res = []
    test_rows = set(idx_test)
    for k, (tr, va) in enumerate(folds_glob, 1):
        assert test_rows.isdisjoint(tr) and test_rows.isdisjoint(va), "Wiersz testu w CV!"
        fb = ScoringFeatureBuilder(**params).fit(X.iloc[tr], y[tr])        # fit TYLKO na foldzie treningowym
        Z_tr, Z_va = fb.transform(X.iloc[tr]), fb.transform(X.iloc[va])
        m = model_factory().fit(Z_tr, y[tr])
        res.append(dict(model=label, fold=k, AUC_uczenie=roc_auc_score(y[tr], m.predict_proba(Z_tr)[:, 1]),
                        AUC=roc_auc_score(y[va], m.predict_proba(Z_va)[:, 1]), n_cech=len(fb.feature_names_),
                        cechy=tuple(fb.feature_names_)))
    return pd.DataFrame(res)

lr = lambda: LogisticRegression(C=0.1, max_iter=2000)
hgb = lambda: HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=150,
                                             min_samples_leaf=40, random_state=RNG_SEED)

def podsumuj(df_cv):
    g = df_cv.groupby("model")
    out = g.agg(AUC_srednia=("AUC", "mean"), AUC_std=("AUC", "std"), AUC_min=("AUC", "min"), AUC_max=("AUC", "max"),
                AUC_uczenie=("AUC_uczenie", "mean"), n_cech=("n_cech", "mean"))
    out["Gini_srednia"] = 2 * out["AUC_srednia"] - 1
    out["luka uczenie-walidacja"] = out["AUC_uczenie"] - out["AUC_srednia"]
    return out

### 9.1. Powtarzane CV na train

In [ ]:
cv_res = pd.concat([run_cv(FB_PARAMS_WOE, lr, cv_train_rep_glob, y_all, "karta: WoE + regresja logistyczna"),
                    run_cv(FB_PARAMS_BB, hgb, cv_train_rep_glob, y_all, "black-box: przygotowanie + HGB")])
cv_summary = podsumuj(cv_res)
print(f"Stratified {N_FOLDS}-fold CV × {N_REPEATS} powtórzenia na TRAIN:")
display(cv_summary.round(4))

**Wniosek.** AUC w CV: karta 0,7202 (Gini 0,4403), HGB 0,7103 (Gini 0,4206); różnica jest mniejsza niż odchylenie między foldami (0,03–0,05). HGB ma AUC na danych uczących 0,99, co wskazuje na przeuczenie – w rozdziale 14 stosujemy regularyzację i kontrolę przeuczenia. Luka uczenie–walidacja karty wynosi 0,08.

### 9.2. Test permutacji etykiet

Po losowym przetasowaniu `default` w train cały pipeline jest uczony od nowa. Jeśli transformacje i selekcja nie korzystają z danych walidacyjnych, AUC powinno wynosić około 0,5.

In [ ]:
N_PERM = 10
perm_rows = []
for r in range(N_PERM):
    y_perm = y_all.copy()
    y_perm[idx_train] = np.random.default_rng(RNG_SEED + r).permutation(y_all[idx_train])
    for params, factory, label in [(FB_PARAMS_WOE, lr, "karta"), (FB_PARAMS_BB, hgb, "black-box")]:
        perm_rows.append({"model": label, "permutacja": r,
                          "AUC": run_cv(params, factory, cv_train_glob, y_perm, label)["AUC"].mean()})
perm = pd.DataFrame(perm_rows)
perm_summary = perm.groupby("model")["AUC"].agg(["mean", "std", "min", "max"])
display(perm_summary.round(3))
for model, row in perm_summary.iterrows():
    z = (row["mean"] - 0.5) / (row["std"] / np.sqrt(N_PERM))
    print(f"{model}: średnie AUC po permutacji = {row['mean']:.3f} (test t względem 0,5: z = {z:+.2f})")
assert (perm_summary["mean"] < 0.56).all(), "AUC po permutacji wyraźnie > 0,5 – podejrzenie przecieku!"
print("OK: brak sygnału przecieku – po permutacji etykiet pipeline nie odtwarza informacji o targecie.")

**Wniosek.** Średnie AUC po permutacji: 0,515 (karta) i 0,518 (black-box), maksimum 0,57. Wyniki są bliskie 0,5 i znacznie niższe niż na prawdziwych etykietach (0,71–0,72), co nie wskazuje na przeciek danych. Dla black-boxa z = 1,93 jest blisko progu istotności 1,96.

### 9.3. Test wrażliwości na `wsk_index_*`

Na tych samych foldach porównujemy modele z `wsk_index_*` i bez nich; różnice liczone są fold do foldu.

In [ ]:
def bez_index(params):
    p = dict(params); p["num_cols"] = [c for c in params["num_cols"] if c not in SUSPICIOUS]
    return p

base_woe = FB_PARAMS_WOE if not DROP_SUSPICIOUS else dict(FB_PARAMS_WOE, num_cols=FB_PARAMS_WOE["num_cols"] + SUSPICIOUS)
base_bb = FB_PARAMS_BB if not DROP_SUSPICIOUS else dict(FB_PARAMS_BB, num_cols=FB_PARAMS_BB["num_cols"] + SUSPICIOUS)
wr = []
for label, params, factory in [("karta", base_woe, lr), ("black-box", base_bb, hgb)]:
    a = run_cv(params, factory, cv_train_rep_glob, y_all, label)["AUC"].to_numpy()
    b = run_cv(bez_index(params), factory, cv_train_rep_glob, y_all, label)["AUC"].to_numpy()
    d = a - b
    wr.append({"model": label, "AUC z wsk_index_*": a.mean(), "AUC bez wsk_index_*": b.mean(),
               "różnica AUC": d.mean(), "różnica Gini": 2 * d.mean(), "std różnicy (fold)": d.std(ddof=1),
               "foldy z poprawą": f"{int((d > 0).sum())}/{len(d)}"})
wrazliwosc = pd.DataFrame(wr).set_index("model")
display(wrazliwosc.round(4))

**Wniosek.** Bez `wsk_index_*` AUC karty spada o 0,06 (w 12 z 15 foldów), a black-boxa o 0,08 (w 14 z 15 foldów). Bez tych wskaźników Gini wynosi 0,32 (karta) i 0,27 (black-box), więc pozostałe cechy zachowują istotną moc predykcyjną. Wskaźniki pozostają w modelu (warunek z 5.4); dla modeli finalnych wariant bez nich jest raportowany w rozdziale 16.1.

### 9.4. Stabilność selekcji cech karty

Częstość wyboru cech w 15 foldach oraz średni indeks Jaccarda zbiorów wybranych cech – dla pojedynczych wskaźników i dla grup wskaźników (sektor, wynik i rentowność, zadłużenie i struktura kapitału, płynność, sprawność).

In [ ]:
sel_sets = [set(f) for f in cv_res.loc[cv_res["model"].str.startswith("karta"), "cechy"]]
n_f = len(sel_sets)
freq = pd.Series([c.replace("woe__", "") for s in sel_sets for c in s]).value_counts()
jac = np.mean([len(a & b) / len(a | b) for i, a in enumerate(sel_sets) for b in sel_sets[i + 1:]])
stab = freq.to_frame("liczba foldów").assign(**{"udział foldów": lambda d: d["liczba foldów"] / n_f,
                                                 "w finalnej karcie": lambda d: d.index.isin(sel)})
display(stab.head(20).round(2))
print(f"Cechy wybrane we wszystkich {n_f} foldach: {int((freq == n_f).sum())}; w >= 80% foldów: "
      f"{int((freq >= 0.8 * n_f).sum())}; w dowolnym foldzie: {len(freq)}.")
print(f"Średni indeks Jaccarda między foldami: {jac:.2f}")
print("Cechy finalnej karty wybierane w < 50% foldów:",
      stab.index[(stab["w finalnej karcie"]) & (stab["udział foldów"] < 0.5)].tolist() or "brak")

RODZINY = [   # (rodzina, fragmenty nazw) – kolejność ma znaczenie
    ("sektor", ("index_",)),
    ("wynik i rentowność", ("ebit", "zysk", "rent", "roe", "roa", "ros", "marza", "pokrycia_odsetek")),
    ("płynność i kapitał obrotowy", ("plynnosc", "kapitalu_obrotowego", "got", "kap_obrotowy")),
    ("zadłużenie i struktura kapitału", ("zadluz", "kap_obc", "kapitalu_obcego", "mnoznik", "kap_wlasn", "kapitalu_wlasnego",
                                         "struktur", "zob", "kapital_do_aktywa", "dzwigni", "fin_majatku", "pokrycie")),
    ("sprawność", ("rotacja", "obrot", "cykl", "sprzedaz")),
]
def rodzina(nazwa):
    n_ = nazwa.lower().replace("woe__", "")
    return next((r for r, frag in RODZINY if any(f in n_ for f in frag)), "inne")
fam_sets = [{rodzina(c) for c in s_} for s_ in sel_sets]
jac_fam = np.mean([len(a & b) / len(a | b) for i, a in enumerate(fam_sets) for b in fam_sets[i + 1:]])
fam_freq = pd.Series([r for s_ in fam_sets for r in s_]).value_counts().div(n_f).rename("udział foldów z rodziną")
display(fam_freq.to_frame().round(2))
print(f"Rodziny w finalnej karcie: {sorted({rodzina(c) for c in sel})}")
print(f"Średni indeks Jaccarda: pojedyncze wskaźniki {jac:.2f} vs rodziny wskaźników {jac_fam:.2f}")

**Wniosek.** Oba wskaźniki sektorowe są wybierane w każdym foldzie, a `wsk_ebitda_3`, `wsk_ebit`, `wsk_efekt_dzwigni_fin_1` i kapitał obrotowy netto w 73–87% foldów. Najrzadziej wybieraną cechą finalnej karty jest `wsk_stopa_zysku_sprzedaz` (33% foldów). Jaccard dla pojedynczych wskaźników wynosi 0,49, a dla grup 0,82 – między foldami zmieniają się głównie wskaźniki w obrębie tej samej grupy.

## 10. Zapis wyników

Zapisywane są: podziały i foldy, dopasowane transformery (`feature_builders.joblib`), tabela IV, profile binów, zbiory po transformacji oraz manifest z konfiguracją i sumą kontrolną danych. Do wczytania `feature_builders.joblib` poza tym notebookiem potrzebne są definicje klas z rozdziału 3.

In [ ]:
karta_final = wybrane.index.tolist()
for old in ART_DIR.glob("*.*"):          # usuwamy wyniki poprzednich uruchomień, by katalog odpowiadał temu przebiegowi
    old.unlink()

splits = {
    "seed": RNG_SEED, "split_mode": SPLIT_MODE, "group_split": GROUPS is not None,
    "index_train": X_train.index.tolist(), "index_val": X_valid.index.tolist(), "index_test": X_test.index.tolist(),
    "pos_train": idx_train.tolist(), "pos_val": idx_val.tolist(), "pos_test": idx_test.tolist(),
    "cv_train_local": [{"train": tr.tolist(), "val": va.tolist()} for tr, va in cv_train],
    "cv_train_global": [{"train": tr.tolist(), "val": va.tolist()} for tr, va in cv_train_glob],
    "cv_dev_global": [{"train": tr.tolist(), "val": va.tolist()} for tr, va in cv_dev_glob],
}
with open(ART_DIR / "splits.json", "w", encoding="utf-8") as f:
    json.dump(splits, f)
joblib.dump({"woe": fb_woe, "blackbox": fb_bb}, ART_DIR / "feature_builders.joblib")
iv.to_csv(ART_DIR / "iv_table.csv")
pd.concat({c: fb_woe.encoder_.profile(c) for c in karta_final}).to_csv(ART_DIR / "woe_bins_selected.csv")
dq.to_csv(ART_DIR / "dq_train.csv")
uni.to_csv(ART_DIR / "univariate_train.csv")
for name, idx in [("train", idx_train), ("val", idx_val), ("test", idx_test)]:
    for kind in ("woe", "bb"):
        X_sets[name][kind].assign(**{TARGET: y_all[idx]}).to_csv(ART_DIR / f"X_{name}_{kind}.csv")

with open(DATA_PATH, "rb") as f:
    data_md5 = hashlib.md5(f.read()).hexdigest()
manifest = {
    "data_file": DATA_PATH, "data_md5": data_md5, "n_obs": len(X_raw), "n_defaults": int(y_raw.sum()),
    "target_pd_anchor": TARGET_PD, "seed": RNG_SEED,
    "split": {"mode": SPLIT_MODE, "train": len(idx_train), "val": len(idx_val), "test": len(idx_test),
              "defaults": {"train": int(y_train.sum()), "val": int(y_valid.sum()), "test": int(y_test.sum())},
              "cv_folds": N_FOLDS, "cv_repeats_check": N_REPEATS},
    "config": {"CLEAN": CLEAN, "BIN": BIN, "MIN_CAT_SHARE": MIN_CAT_SHARE, "IV_SUSPICIOUS": IV_SUSPICIOUS},
    "decisions": {"drop_suspicious_wsk_index": DROP_SUSPICIOUS, "group_split": GROUPS is not None,
                  "woe_convention": "WoE = ln(%good/%bad) (Lab 1); expected beta < 0",
                  "special_values": "WoE: own bins BRAK/+INF/-INF; black-box: cap + __inf / median + __isna"},
    "features_woe": fb_woe.feature_names_, "n_features_blackbox": len(fb_bb.feature_names_),
    "dropped_blackbox": {k: v for k, v in fb_bb.cleaner_.dropped_.items() if k != "duplikat_kolumny"},
    "cv_check": cv_summary[["AUC_srednia", "AUC_std", "AUC_uczenie"]].round(4).to_dict("index"),
    "permutation_auc_mean": perm_summary["mean"].round(4).to_dict(),
    "sensitivity_wsk_index": wrazliwosc[["AUC z wsk_index_*", "AUC bez wsk_index_*", "różnica AUC"]].round(4).to_dict("index"),
    "selection_stability_jaccard": round(float(jac), 3),
    "versions": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
                 "scikit-learn": sklearn.__version__, "scipy": scipy.__version__},
}
with open(ART_DIR / "feature_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False, default=str)

# kontrola odczytu
chk = json.loads((ART_DIR / "splits.json").read_text(encoding="utf-8"))
assert chk["pos_test"] == idx_test.tolist() and len(chk["cv_train_global"]) == N_FOLDS
re_fb = joblib.load(ART_DIR / "feature_builders.joblib")["woe"]
pd.testing.assert_frame_equal(re_fb.transform(X_git.iloc[idx_val]), X_sets["val"]["woe"])
print("Zapisano:", sorted(p.name for p in ART_DIR.iterdir() if p.is_file()))
print("Wykresy:", sorted(p.name for p in FIG_DIR.iterdir()))

## 11. Podsumowanie części I

| Wymaganie | Realizacja | Rozdział |
|---|---|---|
| Podział stratyfikowany train / validation / test | 60/20/20, stratyfikacja po `default`; DR w zbiorach 5,8–6,0% | 4 |
| Stratified K-fold CV (min. 5 foldów) | 5 foldów na train (+ 5×3 do testów, + `cv_dev`) | 4.1, 9 |
| Podział grupowy przy powtórzeniach podmiotu | powtórzeń nie wykryto; kod obsługuje `StratifiedGroupKFold` | 2.2, 4, 5.2 |
| Dokumentacja schematu walidacji | tabela ról zbiorów i diagram | 4, 4.2 |
| Kontrola data leakage | asercje rozłączności i foldów, brak wspólnych obserwacji, audyt cech, test permutacji, test wrażliwości | 5, 9 |
| Test tylko do oceny końcowej | test nie jest używany w EDA, dopasowaniu ani CV | 4, 5.1, 9 |
| Data Quality | braki, `±inf`, tożsamości bilansowe, wartości niemożliwe, ogony, cechy quasi-stałe, duplikaty | 2, 6 |
| EDA | AUC, IV, rozkłady, DR w kategoriach i grupach wielkości, korelacje, VIF | 7 |
| Binning monotoniczny, WoE / encoding | binning z przedziałami specjalnymi, WoE wg Lab 1, kategorie → `INNE` → WoE / one-hot | 3, 8 |
| Capping, standaryzacja | capping p1–p99 z train (black-box), binning (karta), opcja `scale=True` | 3, 6.4, 8 |
| Transformacje tylko na train / foldach | wszystkie w `fit()`; potwierdza test permutacji | 3, 8.5, 9.2 |

**Decyzje**
1. WoE = ln(%dobrych / %złych) (Lab 1), oczekiwane β < 0.
2. `±inf` i `NaN` obsługiwane jako osobne przedziały WoE / flagi w black-boxie.
3. Karta: 8 cech (IV ≥ 0,02, |r| < 0,70, limit 8 cech – EPV ≈ 12), wszystkie z monotonicznym WoE.
4. `wsk_index_*` pozostają w modelu; wariant bez nich jest raportowany równolegle.

**Do potwierdzenia w opisie zmiennych:** okres pomiaru `wsk_index_*`, definicja defaultu i horyzont.

**Kolejne etapy (część II):** modele na cechach z rozdziału 8 (rozdz. 13–16), kalibracja na validation – porównanie metod i korekta poziomu do PD = 4% (rozdz. 17), progi i klasy ratingowe (rozdz. 18); ocena na teście w 17.4 i 18.

# Część II – modele, wyjaśnialność, kalibracja i ratingi (wymagania 3.3–3.6)

Część II korzysta bezpośrednio z obiektów części I: danych `X_git`, `y_all`, podziału `idx_train` / `idx_val` / `idx_test`, foldów `cv_train_glob`, parametrów `FB_PARAMS_WOE` / `FB_PARAMS_BB` oraz tabel z rozdziałów 7–8.

**Zasady**
1. Wybór cech, binning i hiperparametry – CV na train (te same 5 foldów co w części I); w każdym foldzie `ScoringFeatureBuilder` jest dopasowywany od nowa.
2. Kalibracja, korekta do PD = 4%, progi i granice klas ratingowych – validation.
3. Test – wyłącznie ocena końcowa (rozdziały 17.4 i 18).
4. WoE = ln(%dobrych / %złych), więc w karcie oczekujemy β < 0.

## 12. Konfiguracja części modelowej

In [ ]:
import warnings
from itertools import combinations

from sklearn.base import ClassifierMixin

import lightgbm as lgb
import shap
from lime.lime_tabular import LimeTabularExplainer
from scipy.optimize import brentq
from scipy.special import expit, logit
from scipy.stats import ks_2samp, loguniform, randint, spearmanr, uniform
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (average_precision_score, brier_score_loss, cohen_kappa_score, log_loss,
                             precision_recall_curve, roc_curve)
from sklearn.model_selection import ParameterSampler
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")

SEED = RNG_SEED
EPS = 1e-6
MODELS = ["Scorecard", "LightGBM"]

# dane surowe + cechy bezstanowe (X_git) dla train i validation; test wczytywany dopiero w 17.4
X_tr, y_tr = X_git.iloc[idx_train], y_all[idx_train]
X_va, y_va = X_git.iloc[idx_val], y_all[idx_val]

print(f"train: n = {len(y_tr)}, defaulty = {int(y_tr.sum())} | validation: n = {len(y_va)}, defaulty = {int(y_va.sum())}")
print(f"Foldy CV na train: {len(cv_train_glob)} | kandydaci karty: {len(fb_woe.feature_names_)} cech WoE | "
      f"black-box: {len(fb_bb.feature_names_)} kolumn")
print("lightgbm", lgb.__version__, "| shap", shap.__version__)

### 12.1. Funkcje pomocnicze

* `ScorecardLR` – regresja logistyczna na cechach WoE z selekcją: L1 usuwa cechy o zerowym lub niezgodnym znaku, następnie usuwana jest cecha o najwyższym VIF (do VIF ≤ 5), model końcowy L2; cecha ze znakiem niezgodnym po refit jest usuwana i procedura się powtarza.
* metryki: AUC, PR-AUC, KS, log-loss, Brier, calibration intercept (CITL) i slope, ECE,
* kalibratory: brak, intercept, Platt, izotoniczny, beta; przesunięcie logitu do zadanej średniej PD,
* ratingi: przypisanie klas, scalanie klas do monotonicznej obserwowanej częstości defaultów, tabela klas z przedziałem Wilsona (funkcja `wilson` z rozdziału 6),
* tabela decyzyjna dla progu odrzucenia.

In [ ]:
def clip_p(p):
    """Przycina prawdopodobieństwa do [EPS, 1 - EPS], żeby logit i log były skończone."""
    return np.clip(np.asarray(p, float), EPS, 1 - EPS)


def vif_series(X):
    """VIF każdej kolumny z odwrotności macierzy korelacji."""
    X = pd.DataFrame(X)
    if X.shape[1] == 1:
        return pd.Series([1.0], index=X.columns)
    R = np.nan_to_num(np.corrcoef(X.values, rowvar=False))
    np.fill_diagonal(R, 1.0)
    return pd.Series(np.diag(np.linalg.pinv(R)), index=X.columns)


class ScorecardLR(ClassifierMixin, BaseEstimator):
    """Regresja logistyczna na WoE z selekcją cech (L1 -> VIF -> limit cech -> L2 ze sprawdzeniem znaku).
    WoE = ln(%dobrych/%złych), więc oczekiwany znak współczynnika = -1."""

    def __init__(self, C_l1=0.1, C_final=1.0, vif_max=5.0, max_features=None, expected_sign=-1):
        self.C_l1 = C_l1
        self.C_final = C_final
        self.vif_max = vif_max
        self.max_features = max_features
        self.expected_sign = expected_sign

    def fit(self, X, y):
        X, y, sg = pd.DataFrame(X), np.asarray(y), self.expected_sign
        cols, final = list(X.columns), None
        for _ in range(300):
            # krok 1: L1 – zostają cechy z niezerowym współczynnikiem o oczekiwanym znaku
            l1 = LogisticRegression(penalty="l1", solver="liblinear", C=self.C_l1, max_iter=2000).fit(X[cols], y)
            c1 = pd.Series(l1.coef_[0], cols)
            keep = [c for c in cols if c1[c] * sg > 1e-8] or [c1.abs().idxmax()]
            if len(keep) < len(cols):
                cols = keep
                continue
            # krok 2: usuwamy cechę o najwyższym VIF
            v = vif_series(X[cols])
            if len(cols) > 1 and v.max() > self.vif_max:
                cols = [c for c in cols if c != v.idxmax()]
                continue
            # krok 3: limit liczby cech
            if self.max_features and len(cols) > self.max_features:
                cols = c1[cols].abs().sort_values(ascending=False).index[:self.max_features].tolist()
                continue
            # krok 4: model końcowy L2 i kontrola znaków
            l2 = LogisticRegression(C=self.C_final, max_iter=2000).fit(X[cols], y)
            c2 = pd.Series(l2.coef_[0], cols)
            bad = [c for c in cols if c2[c] * sg <= 1e-8]
            final = (l2, c2)
            if 0 < len(bad) < len(cols):
                cols = [c for c in cols if c not in bad]
                continue
            break
        l2, c2 = final
        self.features_ = list(c2.index)
        self.model_ = l2
        self.coef_ = c2.values
        self.intercept_ = float(l2.intercept_[0])
        self.classes_ = np.array([0, 1])
        return self

    def decision_function(self, X):
        return self.model_.decision_function(pd.DataFrame(X)[self.features_])

    def predict_proba(self, X):
        return self.model_.predict_proba(pd.DataFrame(X)[self.features_])


def ks_stat(y, p):
    """Statystyka KS: maks. różnica dystrybuant PD defaultów i niedefaultów."""
    y, p = np.asarray(y), np.asarray(p)
    return float(ks_2samp(p[y == 1], p[y == 0]).statistic)


def calib_intercept_slope(y, p, w=None):
    """Calibration-in-the-large (CITL: przesunięcie logitu przy nachyleniu 1) i calibration slope."""
    y = np.asarray(y, float)
    z = logit(clip_p(p))
    w = np.ones_like(y) if w is None else np.asarray(w, float)
    a = 0.0
    for _ in range(100):                                        # Newton-Raphson dla CITL
        q = expit(z + a)
        step = np.sum(w * (y - q)) / max(np.sum(w * q * (1 - q)), 1e-12)
        a += step
        if abs(step) < 1e-10:
            break
    lr_ = LogisticRegression(C=1e6, max_iter=1000).fit(z.reshape(-1, 1), y, sample_weight=w)
    return {"citl": float(a), "slope": float(lr_.coef_[0, 0])}


def ece_score(y, p, n_bins=8):
    """Expected Calibration Error w koszykach kwantylowych."""
    d = pd.DataFrame({"y": np.asarray(y), "p": np.asarray(p)})
    d["bin"] = pd.qcut(d["p"], n_bins, duplicates="drop")
    g = d.groupby("bin", observed=True).agg(n=("y", "size"), mean_pd=("p", "mean"), obs_rate=("y", "mean"))
    return float(np.sum(g["n"] * (g["mean_pd"] - g["obs_rate"]).abs()) / g["n"].sum())


def all_metrics(y, p, w=None):
    """Metryki rozróżniania i kalibracji."""
    y, p = np.asarray(y), clip_p(p)
    cs = calib_intercept_slope(y, p, w)
    return {"AUC": roc_auc_score(y, p), "PR_AUC": average_precision_score(y, p), "KS": ks_stat(y, p),
            "LogLoss": log_loss(y, p, sample_weight=w), "Brier": brier_score_loss(y, p, sample_weight=w),
            "CITL": cs["citl"], "Slope": cs["slope"], "ECE": ece_score(y, p) if w is None else np.nan,
            "mean_PD": float(np.average(p, weights=w)), "obs_rate": float(np.average(y, weights=w))}


class IdentityCalibrator:
    name = "brak"

    def fit(self, p, y):
        return self

    def predict(self, p):
        return clip_p(p)


class PlattCalibrator:
    name = "Platt"

    def fit(self, p, y):
        self.lr_ = LogisticRegression(C=1e6, max_iter=1000).fit(logit(clip_p(p)).reshape(-1, 1), np.asarray(y))
        return self

    def predict(self, p):
        return self.lr_.predict_proba(logit(clip_p(p)).reshape(-1, 1))[:, 1]


class IsotonicCalibrator:
    name = "Isotonic"

    def __init__(self, floor=1e-3):
        self.floor = floor

    def fit(self, p, y):
        self.iso_ = IsotonicRegression(y_min=self.floor, y_max=1 - self.floor, out_of_bounds="clip", increasing=True)
        self.iso_.fit(np.asarray(p, float), np.asarray(y, float))
        return self

    def predict(self, p):
        return clip_p(self.iso_.predict(np.asarray(p, float)))


def beta_features(p):
    """Cechy kalibracji beta: [ln p, -ln(1 - p)]."""
    p = clip_p(p)
    return np.c_[np.log(p), -np.log(1 - p)]


class BetaCalibrator:
    """Kalibracja beta: regresja logistyczna na [ln p, -ln(1 - p)] (3 parametry)."""
    name = "Beta"

    def fit(self, p, y):
        self.lr_ = LogisticRegression(C=1e6, max_iter=2000).fit(beta_features(p), np.asarray(y))
        return self

    def predict(self, p):
        return self.lr_.predict_proba(beta_features(p))[:, 1]


def shift_to_mean(p, target, w=None):
    """Przesunięcie δ w skali logitu, po którym średnia PD = target."""
    z = logit(clip_p(p))
    return float(brentq(lambda d: np.average(expit(z + d), weights=w) - target, -15, 15))


def apply_shift(p, delta):
    return expit(logit(clip_p(p)) + delta)


class InterceptCalibrator:
    """Kalibracja interceptu: średnia PD = częstość defaultów w zbiorze kalibracyjnym (ranking bez zmian)."""
    name = "Intercept"

    def fit(self, p, y):
        self.delta_ = shift_to_mean(p, target=float(np.mean(y)))
        return self

    def predict(self, p):
        return apply_shift(p, self.delta_)


RATING_LABELS = ["AAA", "AA", "A", "BBB", "BB", "B", "CCC"]


def assign_rating(pd_arr, bounds, labels):
    """Klasa ratingowa według rosnących progów PD."""
    idx = np.searchsorted(np.asarray(bounds), np.asarray(pd_arr), side="right")
    return np.asarray(labels, dtype=object)[idx]


def pick_labels(k):
    """k etykiet równomiernie rozłożonych na skali RATING_LABELS."""
    return [RATING_LABELS[i] for i in np.round(np.linspace(0, len(RATING_LABELS) - 1, k)).astype(int)]


def labels_from_master(pd_ref, bounds, master_bounds, labels=RATING_LABELS):
    """Klasa po scaleniu dostaje etykietę klasy skali nadrzędnej, w której leży jej średnia PD."""
    pd_ref = np.asarray(pd_ref)
    idx = np.searchsorted(bounds, pd_ref, side="right")
    edges = [0.0] + list(bounds) + [1.0]
    out = []
    for k in range(len(bounds) + 1):
        v = pd_ref[idx == k]
        rep = v.mean() if len(v) else np.sqrt(max(edges[k], 1e-4) * edges[k + 1])
        out.append(labels[int(np.searchsorted(master_bounds, rep, side="right"))])
    return out if len(set(out)) == len(out) else pick_labels(len(out))


def monotone_bounds(pd_arr, y, bounds, min_n=15):
    """Łączy sąsiednie klasy, dopóki obserwowana częstość defaultów nie rośnie ściśle z klasą
    lub klasa ma mniej niż min_n obserwacji."""
    bounds, pd_arr, y = list(bounds), np.asarray(pd_arr), np.asarray(y)
    while True:
        k = len(bounds) + 1
        idx = np.searchsorted(bounds, pd_arr, side="right")
        n = np.bincount(idx, minlength=k).astype(float)
        d = np.bincount(idx, weights=y, minlength=k)
        rate = np.where(n > 0, d / np.maximum(n, 1), 0.0)
        merged = False
        for i in range(k - 1):
            if n[i] < min_n or rate[i] >= rate[i + 1]:
                bounds.pop(i)
                merged = True
                break
        if not merged and k > 1 and n[-1] < min_n:
            bounds.pop()
            merged = True
        if not merged or len(bounds) == 0:
            return bounds


def rating_table(pd_arr, y, bounds, labels):
    """Liczebność, defaulty, średnia PD, obserwowana częstość defaultów i 95% przedział Wilsona w klasach."""
    cls = assign_rating(pd_arr, bounds, labels)
    d = pd.DataFrame({"klasa": pd.Categorical(cls, categories=labels, ordered=True),
                      "pd": np.asarray(pd_arr), "y": np.asarray(y)})
    g = d.groupby("klasa", observed=False).agg(n=("y", "size"), defaulty=("y", "sum"),
                                               srednia_PD=("pd", "mean"), obs_default_rate=("y", "mean"))
    g["udzial_%"] = 100 * g["n"] / g["n"].sum()
    ci = [wilson(k, n) for k, n in zip(g["defaulty"], g["n"])]
    g["CI95_lo"] = [a for a, _ in ci]
    g["CI95_hi"] = [b for _, b in ci]
    return g


def decision_table(y, p, threshold, margin, lgd):
    """Decyzja: odrzucenie, gdy PD >= próg. TP – odrzucony default, FP – odrzucony dobry (utracona marża),
    FN – przyjęty default (strata LGD), TN – przyjęty dobry. Zysk = margin·TN − lgd·FN."""
    y = np.asarray(y)
    rej = np.asarray(p) >= threshold
    tp, fp = int((rej & (y == 1)).sum()), int((rej & (y == 0)).sum())
    fn, tn = int((~rej & (y == 1)).sum()), int((~rej & (y == 0)).sum())
    return dict(prog=threshold, odrzucenia_proc=100 * rej.mean(), TP=tp, FP=fp, FN=fn, TN=tn,
                bad_rate_przyjetych=fn / max(fn + tn, 1), recall_defaultow=tp / max(tp + fn, 1),
                zysk_na_klienta=(margin * tn - lgd * fn) / len(y))


def prep_folds(params):
    """Dla każdego foldu cv_train_glob dopasowuje ScoringFeatureBuilder tylko na części treningowej foldu.
    Zwraca listę (X_tr_fold, y_tr_fold, X_va_fold, y_va_fold, tr, va)."""
    wynik = []
    for tr, va in cv_train_glob:
        fb = ScoringFeatureBuilder(**params).fit(X_git.iloc[tr], y_all[tr])
        wynik.append((fb.transform(X_git.iloc[tr]), y_all[tr], fb.transform(X_git.iloc[va]), y_all[va], tr, va))
    return wynik


def summarize_cv(rows):
    """Średnia, odchylenie standardowe, minimum i maksimum metryk z foldów."""
    t = pd.DataFrame(rows)
    return pd.DataFrame({"średnia": t.mean(), "odch.std": t.std(ddof=1), "min": t.min(), "max": t.max()}).T


def raw_name(f):
    """Nazwa cechy WoE ('woe__x') -> nazwa cechy wejściowej ('x')."""
    return f.replace("woe__", "", 1)


METRYKI = ["AUC", "PR_AUC", "KS", "LogLoss", "Brier", "CITL", "Slope", "mean_PD"]

## 13. Model interpretowalny – karta WoE + regresja logistyczna

Kandydatami są cechy WoE wybrane w rozdziale 8 (IV ≥ 0,02, |r| < 0,70, maks. 8 cech). `ScorecardLR` może dodatkowo usunąć cechy o niezgodnym znaku lub wysokim VIF.

Hiperparametry (siła L1 i L2) wybieramy w 5-fold CV na train, z `ScoringFeatureBuilder` dopasowywanym od nowa w każdym foldzie. Spośród konfiguracji z log-lossem w granicy 1 błędu standardowego od najlepszej wybieramy tę z najmniejszą liczbą cech (reguła 1-SE).

In [ ]:
folds_woe = prep_folds(FB_PARAMS_WOE)

grid = [{"C_l1": a, "C_final": b, "vif_max": 5.0} for a in [0.03, 0.1, 0.3, 1.0] for b in [0.1, 1.0]]
rows = []
for g in grid:
    ll, auc, nf = [], [], []
    for Xa, ya, Xb, yb, _, _ in folds_woe:
        m = ScorecardLR(**g).fit(Xa, ya)
        p = m.predict_proba(Xb)[:, 1]
        ll.append(log_loss(yb, p))
        auc.append(roc_auc_score(yb, p))
        nf.append(len(m.features_))
    rows.append({**g, "logloss": np.mean(ll), "logloss_se": np.std(ll, ddof=1) / np.sqrt(len(ll)),
                 "auc": np.mean(auc), "auc_std": np.std(auc, ddof=1), "n_cech": np.mean(nf)})

cv_grid = pd.DataFrame(rows).sort_values("logloss")
best = cv_grid.iloc[0]
w_1se = cv_grid[cv_grid["logloss"] <= best["logloss"] + best["logloss_se"]]
chosen = w_1se.sort_values(["n_cech", "logloss"]).iloc[0]
BEST_LR = {"C_l1": float(chosen["C_l1"]), "C_final": float(chosen["C_final"]), "vif_max": 5.0}

display(cv_grid)
print("Wybrana konfiguracja (reguła 1-SE):", BEST_LR)

In [ ]:
# model końcowy na całym train
pipe_lr = Pipeline([("prep", ScoringFeatureBuilder(**FB_PARAMS_WOE)), ("clf", ScorecardLR(**BEST_LR))])
pipe_lr.fit(X_tr, y_tr)
prep_lr = pipe_lr.named_steps["prep"]
clf_lr = pipe_lr.named_steps["clf"]

# predykcje out-of-fold na train
oof_lr = pd.Series(np.nan, index=idx_train)
cv_rows_lr, fold_models_lr = [], []
for Xa, ya, Xb, yb, tr, va in folds_woe:
    m = ScorecardLR(**BEST_LR).fit(Xa, ya)
    p = m.predict_proba(Xb)[:, 1]
    oof_lr.loc[va] = p
    fold_models_lr.append(m)
    cv_rows_lr.append(all_metrics(yb, p))

p_val_lr = pipe_lr.predict_proba(X_va)[:, 1]
p_tr_lr = pipe_lr.predict_proba(X_tr)[:, 1]

print(f"Cechy w modelu: {len(clf_lr.features_)} z {len(prep_lr.feature_names_)} kandydatów")
print("Karta – metryki w 5 foldach CV (train):")
display(summarize_cv(cv_rows_lr)[METRYKI])
tabela_lr = pd.DataFrame({"train (in-sample)": all_metrics(y_tr, p_tr_lr),
                          "OOF-CV (train)": all_metrics(y_tr, oof_lr.loc[idx_train].values),
                          "validation": all_metrics(y_va, p_val_lr)})
display(tabela_lr.T[METRYKI + ["obs_rate"]])

**Wniosek.** Reguła 1-SE wybrała `C_l1` = 0,1 i `C_final` = 0,1; `ScorecardLR` zachował wszystkie 8 cech z rozdziału 8. AUC w CV wynosi 0,716 ± 0,065, na validation 0,814 (validation ma 35 defaultów, więc AUC na nim ma duży błąd losowy). Luka AUC train – OOF wynosi 0,075. Średnie PD (5,9%) odpowiada częstości defaultów w próbie, a nie 4% – poziom jest korygowany w rozdziale 17.

### 13.1. Interpretacja globalna: współczynniki, ilorazy szans, ważność, VIF

Ważność cechy = |β| · odchylenie standardowe WoE na train. Iloraz szans exp(β) dotyczy wzrostu WoE o 1 (przejścia do przedziału o niższym ryzyku).

In [ ]:
feats_lr = clf_lr.features_
Wtr = prep_lr.transform(X_tr)[feats_lr]

coef_tab = pd.DataFrame({
    "β": clf_lr.coef_,
    "exp(β) na +1 WoE": np.exp(clf_lr.coef_),
    "std WoE": Wtr.std().values,
    "ważność |β|·std": np.abs(clf_lr.coef_) * Wtr.std().values,
    "VIF": vif_series(Wtr).values,
    "IV": [prep_lr.iv_table_.loc[raw_name(f), "IV"] for f in feats_lr],
    "kierunek ryzyka (binning)": [prep_lr.iv_table_.loc[raw_name(f), "kierunek_ryzyka"] for f in feats_lr],
}, index=feats_lr).sort_values("ważność |β|·std", ascending=False)
coef_tab["udział ważności (%)"] = 100 * coef_tab["ważność |β|·std"] / coef_tab["ważność |β|·std"].sum()

print(f"Wyraz wolny: {clf_lr.intercept_:.3f} | wszystkie β < 0: {bool((clf_lr.coef_ < 0).all())} | "
      f"maks. VIF: {coef_tab['VIF'].max():.2f}")
display(coef_tab)

fig, ax = plt.subplots(figsize=(8, 0.4 * len(coef_tab) + 1.5))
coef_tab["ważność |β|·std"][::-1].plot.barh(ax=ax, color="#3182bd")
ax.set_yticklabels([raw_name(f) for f in coef_tab.index[::-1]])
ax.set_title("Ważność cech karty (|β|·std WoE)")
savefig("lr_importance")
plt.show()

**Wniosek.** Wszystkie współczynniki są ujemne, zgodnie z konwencją WoE; maks. VIF = 2,04. Największy udział w ważności mają wskaźniki sektorowe `wsk_index_nowe` (26%) i `wsk_index_skr_zaw` (15%).

Kierunki cech modelu: oczekiwanie a priori i kierunek z binningu (tabela z rozdziału 8.3) oraz znak współczynnika.

In [ ]:
kier_lr = kierunki.reindex([raw_name(f) for f in feats_lr]).copy()
kier_lr.insert(0, "β", clf_lr.coef_)
kier_lr.insert(1, "β < 0", np.where(clf_lr.coef_ < 0, "TAK", "NIE"))
display(kier_lr[["β", "β < 0", "oczekiwany (a priori)", "z binningu", "zgodność a priori", "ocena ekonomiczna"]])
print("Cechy karty z rozdz. 8 usunięte przez ScorecardLR:",
      sorted(set(prep_lr.feature_names_) - set(feats_lr)) or "brak")

In [ ]:
# monotoniczność poza próbą: biny z train zastosowane do validation (Spearman: numer przedziału vs DR)
num_feats = [raw_name(f) for f in feats_lr if raw_name(f) in NUM_INTERP]
Zva = prep_lr.cleaner_.transform(X_va[prep_lr.num_cols])
rows = []
for f in num_feats:
    b = prep_lr.encoder_.bins_[f]
    v = Zva[f].values
    ok = np.isfinite(v)
    bi = np.searchsorted(b["edges"], v[ok], side="right")
    t = pd.DataFrame({"bin": bi, "y": y_va[ok]}).groupby("bin")["y"].mean()
    rho = spearmanr(t.index, t.values)[0] if len(t) > 2 else np.nan
    expected = 1 if b["direction"] == "inc" else -1
    rows.append({"cecha": f, "kierunek (train)": b["direction"], "przedziały na val": len(t),
                 "Spearman przedział vs DR (val)": rho,
                 "zgodny znak": None if np.isnan(rho) else bool(np.sign(rho) == expected)})
mono_val = pd.DataFrame(rows).set_index("cecha")
display(mono_val)
print(f"Zgodny kierunek na validation: {int(mono_val['zgodny znak'].sum())} z {mono_val['zgodny znak'].notna().sum()} cech "
      "(na train monotoniczność wynika z konstrukcji binningu).")

**Wniosek.** Kierunek z binningu jest zgodny z oczekiwaniem a priori dla 5 cech; dla `wsk_index_nowe`, `wsk_struktury_kapitalu` i `wsk_efekt_dzwigni_fin_1` nie ma hipotezy a priori (rozdział 8.3). Na validation kierunek zależności jest taki sam jak na train dla 8 z 8 cech.

### 13.2. Profile przedziałów i PDP/ICE

Wkład przedziału do logitu ryzyka = β · WoE(przedział); kolor czerwony – przedział zwiększa ryzyko względem WoE = 0.

In [ ]:
top_num = [f for f in coef_tab.index if raw_name(f) in NUM_INTERP][:6]
fig, axes = plt.subplots(2, 3, figsize=(16, 7))
for ax, f in zip(axes.ravel(), top_num):
    profil = prep_lr.encoder_.profile(raw_name(f))
    b = coef_tab.loc[f, "β"]
    wklady = b * profil["WoE"]
    ax.bar(range(len(profil)), wklady, color=["#e34a33" if v > 0 else "#31a354" for v in wklady])
    ax.set_xticks(range(len(profil)))
    ax.set_xticklabels(profil["przedzial"], rotation=60, ha="right", fontsize=7)
    ax.set_title(f"{raw_name(f)}  (β = {b:.2f})", fontsize=9)
    ax.set_ylabel("wkład do logitu")
for ax in axes.ravel()[len(top_num):]:
    ax.axis("off")
plt.suptitle("Karta: wkład przedziałów do logitu ryzyka (czerwony – wyższe ryzyko)")
savefig("lr_bin_contrib")
plt.show()

profil_top = prep_lr.encoder_.profile(raw_name(top_num[0]))
profil_top["wkład do logitu"] = coef_tab.loc[top_num[0], "β"] * profil_top["WoE"]
display(profil_top)

In [ ]:
def pdp_ice(pipe, X, feature, n_grid=15, n_ice=40, qlo=0.05, qhi=0.95, seed=SEED):
    """PDP (średnia PD) i ICE (PD pojedynczych obserwacji) dla siatki kwantyli p5–p95 cechy."""
    s = X[feature].replace([np.inf, -np.inf], np.nan).dropna()
    grid_v = np.unique(np.quantile(s, np.linspace(qlo, qhi, n_grid)))
    ice_rows = np.random.default_rng(seed).choice(len(X), min(n_ice, len(X)), replace=False)
    pdp, ice = [], np.zeros((len(ice_rows), len(grid_v)))
    for j, val in enumerate(grid_v):
        Xg = X.copy()
        Xg[feature] = val
        p = pipe.predict_proba(Xg)[:, 1]
        pdp.append(p.mean())
        ice[:, j] = p[ice_rows]
    return grid_v, np.array(pdp), ice


def plot_pdp(pipe, X, features, title, fname, ncols=3):
    """PDP (czerwona linia) i ICE (szare linie) dla listy cech."""
    nrows = int(np.ceil(len(features) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 3.6 * nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax, f in zip(axes, features):
        g, pdp, ice = pdp_ice(pipe, X, f)
        ax.plot(g, ice.T, color="grey", alpha=.15, lw=.8)
        ax.plot(g, pdp, color="#d7301f", lw=2.5)
        ax.set_title(f, fontsize=9)
        ax.set_ylabel("PD")
    for ax in axes[len(features):]:
        ax.axis("off")
    plt.suptitle(title)
    savefig(fname)
    plt.show()


pdp_feats = [raw_name(f) for f in top_num]
plot_pdp(pipe_lr, X_tr, pdp_feats, "Karta: PDP (czerwony) i ICE (szare), zakres p5–p95 (train)", "lr_pdp")

### 13.3. Interpretacja lokalna: dekompozycja logitu

logit(PD) = β₀ + Σ βⱼ·WoEⱼ. Wkład cechy względem przeciętnej firmy z train = βⱼ·(WoEⱼ(x) − średnie WoEⱼ); suma wkładów i logitu przeciętnej firmy daje logit predykcji (sprawdzane asercją).

In [ ]:
W_mean = Wtr.mean()
logit_avg = clf_lr.intercept_ + float((clf_lr.coef_ * W_mean.values).sum())


def lr_contrib(X_rows):
    """Wkłady cech karty do logitu względem przeciętnej firmy z train (wiersze = obserwacje)."""
    W = prep_lr.transform(X_rows)[feats_lr]
    return (W - W_mean) * clf_lr.coef_


C_va = lr_contrib(X_va)
assert np.allclose(expit(logit_avg + C_va.sum(axis=1).values), p_val_lr, atol=1e-8), "Dekompozycja niezgodna z predykcją."
print("OK: suma wkładów cech odtwarza logit predykcji karty na validation.")

ex = [int(np.argmax(p_val_lr)), int(np.argmin(np.abs(p_val_lr - np.median(p_val_lr)))), int(np.argmin(p_val_lr))]
dek = C_va.iloc[ex].T
dek.columns = [f"najwyższe PD (y={y_va[ex[0]]})", f"PD ≈ mediana (y={y_va[ex[1]]})", f"najniższe PD (y={y_va[ex[2]]})"]
dek.index = [raw_name(f) for f in dek.index]
dek.loc["logit przeciętnej firmy"] = logit_avg
dek.loc["logit predykcji"] = logit_avg + C_va.iloc[ex].sum(axis=1).values
dek.loc["PD"] = p_val_lr[ex]
display(dek)

### 13.4. Stabilność karty w CV

Częstość wyboru cech i zmienność współczynników w 5 foldach oraz indeks Jaccarda zbiorów cech między foldami.

In [ ]:
sel_freq = pd.Series({f: sum(f in m.features_ for m in fold_models_lr) for f in feats_lr})
coefs_cv = pd.DataFrame([pd.Series(m.coef_, m.features_) for m in fold_models_lr]).reindex(columns=feats_lr)
stab_lr = pd.DataFrame({"wybrana w foldach": sel_freq, "β średnie": coefs_cv.mean(), "β std": coefs_cv.std(),
                        "współczynnik zmienności β": coefs_cv.std() / coefs_cv.mean().abs()})
display(stab_lr.sort_values("wybrana w foldach", ascending=False))

sets_lr = [set(m.features_) for m in fold_models_lr]
jac_lr = np.mean([len(a & b) / len(a | b) for a, b in combinations(sets_lr, 2)])
print(f"Średni Jaccard zbiorów cech między foldami: {jac_lr:.2f} | cechy wybrane we wszystkich foldach: "
      f"{int((sel_freq == len(fold_models_lr)).sum())}/{len(feats_lr)} | wszystkie β w foldach < 0: "
      f"{bool((coefs_cv.fillna(-1) < 0).all().all())}")

**Wniosek.** Współczynniki mają ujemny znak we wszystkich foldach. Zbiory cech różnią się między foldami (Jaccard 0,43); w każdym z 5 foldów wybierane są tylko oba wskaźniki sektorowe, co jest zgodne z wynikiem z rozdziału 9.4.

## 14. Model black-box – LightGBM

* Wejście: `ScoringFeatureBuilder(mode="blackbox")` z rozdziału 8 (capping p1–p99, imputacja medianą, flagi `__isna` / `__inf`, one-hot).
* Bez oversamplingu i bez `scale_pos_weight` – prawdopodobieństwa nie są przesuwane przed kalibracją (rozdz. 17).
* Tuning: Random Search (40 konfiguracji) w tych samych 5 foldach co karta; kryterium – średni log-loss w CV.
* Kontrola przeuczenia: regularyzacja w przestrzeni poszukiwań (mała liczba liści, `min_child_samples`, L1/L2, subsampling), reguła 1-SE (najprostszy model w granicy 1 błędu standardowego), porównanie AUC train / CV / validation i krzywa log-lossu względem liczby drzew.

In [ ]:
N_ITER = 40
N_JOBS = 2
folds_bb = prep_folds(FB_PARAMS_BB)

space = {"n_estimators": randint(100, 501),
         "learning_rate": loguniform(0.01, 0.1),
         "num_leaves": randint(4, 25),
         "max_depth": [-1, 3, 4, 5],
         "min_child_samples": randint(20, 81),
         "subsample": uniform(0.6, 0.4),
         "colsample_bytree": uniform(0.4, 0.6),
         "reg_lambda": loguniform(0.1, 50),
         "reg_alpha": loguniform(1e-3, 5)}
cands = list(ParameterSampler(space, n_iter=N_ITER, random_state=SEED))


def mk_lgb(params):
    """LightGBM z podanymi hiperparametrami; pozostałe ustawienia stałe (powtarzalność)."""
    return lgb.LGBMClassifier(**params, subsample_freq=1, random_state=SEED, n_jobs=N_JOBS, verbose=-1)


rows = []
for k, c in enumerate(cands):
    ll, auc, auc_tr = [], [], []
    for Xa, ya, Xb, yb, _, _ in folds_bb:
        m = mk_lgb(c).fit(Xa, ya)
        p = m.predict_proba(Xb)[:, 1]
        ll.append(log_loss(yb, p))
        auc.append(roc_auc_score(yb, p))
        auc_tr.append(roc_auc_score(ya, m.predict_proba(Xa)[:, 1]))
    rows.append({"id": k, **c, "logloss": np.mean(ll), "logloss_se": np.std(ll, ddof=1) / np.sqrt(len(ll)),
                 "auc_cv": np.mean(auc), "auc_std": np.std(auc, ddof=1), "auc_train": np.mean(auc_tr)})

cv_bb = pd.DataFrame(rows).sort_values("logloss").reset_index(drop=True)
cv_bb["luka AUC train-CV"] = cv_bb["auc_train"] - cv_bb["auc_cv"]
cv_bb["złożoność"] = cv_bb["n_estimators"] * cv_bb["num_leaves"]
b0 = cv_bb.iloc[0]
pick = cv_bb[cv_bb["logloss"] <= b0["logloss"] + b0["logloss_se"]].sort_values("złożoność").iloc[0]
BEST_BB = {k: (int(pick[k]) if k in ("n_estimators", "num_leaves", "max_depth", "min_child_samples") else float(pick[k]))
           for k in space}

display(cv_bb[["id", "logloss", "logloss_se", "auc_cv", "auc_std", "auc_train", "luka AUC train-CV",
               "n_estimators", "num_leaves", "learning_rate", "min_child_samples"]].head(8))
print("Wybrane hiperparametry (reguła 1-SE):", BEST_BB)

In [ ]:
# log-loss na foldach walidacyjnych w funkcji liczby drzew (wybrane hiperparametry)
curves = np.array([mk_lgb(BEST_BB).fit(Xa, ya, eval_set=[(Xb, yb)]).evals_result_["valid_0"]["binary_logloss"]
                   for Xa, ya, Xb, yb, _, _ in folds_bb])
mean_c = curves.mean(0)

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(np.arange(1, len(mean_c) + 1), mean_c, label="log-loss (średnia z foldów walidacyjnych)")
ax.fill_between(np.arange(1, len(mean_c) + 1), curves.min(0), curves.max(0), alpha=.2)
ax.axvline(BEST_BB["n_estimators"], color="k", ls="--", label="wybrane n_estimators")
ax.set(xlabel="liczba drzew", ylabel="log-loss", title="LightGBM: log-loss w CV a liczba drzew")
ax.legend()
savefig("bb_overfit_curve")
plt.show()
print(f"Minimum średniego log-lossu przy {int(mean_c.argmin()) + 1} drzewach; wybrano {BEST_BB['n_estimators']}.")

In [ ]:
# model końcowy na całym train
pipe_bb = Pipeline([("prep", ScoringFeatureBuilder(**FB_PARAMS_BB)), ("clf", mk_lgb(BEST_BB))])
pipe_bb.fit(X_tr, y_tr)
prep_bb = pipe_bb.named_steps["prep"]
bb_model = pipe_bb.named_steps["clf"]

oof_bb = pd.Series(np.nan, index=idx_train)
cv_rows_bb = []
for Xa, ya, Xb, yb, tr, va in folds_bb:
    p = mk_lgb(BEST_BB).fit(Xa, ya).predict_proba(Xb)[:, 1]
    oof_bb.loc[va] = p
    cv_rows_bb.append(all_metrics(yb, p))

p_val_bb = pipe_bb.predict_proba(X_va)[:, 1]
p_tr_bb = pipe_bb.predict_proba(X_tr)[:, 1]

print("LightGBM – metryki w 5 foldach CV (train):")
display(summarize_cv(cv_rows_bb)[METRYKI])
tabela_bb = pd.DataFrame({"train (in-sample)": all_metrics(y_tr, p_tr_bb),
                          "OOF-CV (train)": all_metrics(y_tr, oof_bb.loc[idx_train].values),
                          "validation": all_metrics(y_va, p_val_bb)})
display(tabela_bb.T[METRYKI + ["obs_rate"]])
print(f"Kolumny wejściowe: {len(prep_bb.feature_names_)} | luka AUC train – OOF: "
      f"{tabela_bb.loc['AUC', 'train (in-sample)'] - tabela_bb.loc['AUC', 'OOF-CV (train)']:.3f}")

### 14.1. Wyjaśnialność globalna: SHAP

TreeSHAP na validation; wartości w skali log-odds. Wykresy: beeswarm, ważność (średnie |SHAP|) i zależności dla trzech najważniejszych cech.

In [ ]:
Xbb_tr = prep_bb.transform(X_tr)
Xbb_va = prep_bb.transform(X_va)
bb_cols = list(Xbb_tr.columns)
explainer = shap.TreeExplainer(bb_model)


def shap_pos(expl, X):
    """Wartości SHAP dla klasy 'default' jako macierz (obserwacje × cechy) i wartość bazowa
    (różne wersje shap zwracają różne kształty)."""
    sv = expl.shap_values(X)
    if isinstance(sv, list):
        sv = sv[-1]
    sv = np.asarray(sv)
    if sv.ndim == 3:
        sv = sv[:, :, -1]
    return sv, float(np.atleast_1d(expl.expected_value)[-1])


sv_va, ev_va = shap_pos(explainer, Xbb_va)
imp_shap = pd.Series(np.abs(sv_va).mean(0), index=bb_cols).sort_values(ascending=False)
display(imp_shap.head(15).to_frame("średnie |SHAP| (log-odds)"))

shap.summary_plot(sv_va, Xbb_va, max_display=15, show=False)
savefig("shap_beeswarm")
plt.show()
shap.summary_plot(sv_va, Xbb_va, plot_type="bar", max_display=15, show=False)
savefig("shap_bar")
plt.show()
for f in imp_shap.index[:3]:
    shap.dependence_plot(f, sv_va, Xbb_va, show=False)
    savefig(f"shap_dep_{f}")
    plt.show()

### 14.2. PDP/ICE black-boxa i porównanie z kartą

PDP/ICE dla tych samych cech co w 13.2 oraz porównanie rankingów ważności: karta (|β|·std WoE) vs LightGBM (średnie |SHAP|).

In [ ]:
common_pdp = [f for f in pdp_feats if f in NUM_BLACK]
plot_pdp(pipe_bb, X_tr, common_pdp, "LightGBM: PDP (czerwony) i ICE (szare), te same cechy co w karcie", "bb_pdp")

In [ ]:
lr_imp = coef_tab["ważność |β|·std"].copy()
lr_imp.index = [raw_name(f) for f in lr_imp.index]
common = [f for f in lr_imp.index if f in imp_shap.index]

cmp_imp = pd.DataFrame({"ważność karta": lr_imp.loc[common], "ważność SHAP": imp_shap.loc[common]})
cmp_imp["ranga karta"] = cmp_imp["ważność karta"].rank(ascending=False)
cmp_imp["ranga SHAP (wśród tych cech)"] = cmp_imp["ważność SHAP"].rank(ascending=False)
cmp_imp["ranga SHAP (wszystkie kolumny)"] = [int(imp_shap.index.get_loc(f)) + 1 for f in cmp_imp.index]
display(cmp_imp.sort_values("ranga karta"))

rho_imp = spearmanr(cmp_imp["ranga karta"], cmp_imp["ranga SHAP (wśród tych cech)"])[0]
top_k = 5
jacc_imp = len(set(lr_imp.head(top_k).index) & set(imp_shap.head(top_k).index)) / \
    len(set(lr_imp.head(top_k).index) | set(imp_shap.head(top_k).index))
print(f"Spearman rang (cechy karty, n = {len(common)}): {rho_imp:.2f} | Jaccard top-{top_k} karta vs SHAP: {jacc_imp:.2f}")
print("Top-5 SHAP: ", imp_shap.head(5).index.tolist())
print("Top-5 karta:", lr_imp.head(5).index.tolist())

### 14.3. Interpretacja lokalna: wybrane obserwacje (validation)

Cztery obserwacje: (1) default o najwyższym PD LightGBM, (2) niedefault o najniższym PD, (3) obserwacja o PD najbliższym 85. percentylowi, (4) default o najniższym PD. Dla każdej: wkłady karty, SHAP i LIME (5 cech o największym |wkładzie|).

In [ ]:
q85 = np.quantile(p_val_bb, 0.85)
poz_def, poz_dob = np.where(y_va == 1)[0], np.where(y_va == 0)[0]
cases = {"1. wykryty default": int(poz_def[np.argmax(p_val_bb[y_va == 1])]),
         "2. niedefault o najniższym PD": int(poz_dob[np.argmin(p_val_bb[y_va == 0])]),
         "3. PD ≈ 85. percentyl": int(np.argmin(np.abs(p_val_bb - q85))),
         "4. niewykryty default": int(poz_def[np.argmin(p_val_bb[y_va == 1])])}


def predict_fn(a):
    """Predykcja LightGBM dla tablicy numpy (wymagana przez LIME)."""
    return bb_model.predict_proba(pd.DataFrame(a, columns=bb_cols))


bin_idx = [i for i, c in enumerate(bb_cols) if set(np.unique(Xbb_tr[c])) <= {0, 1}]   # kolumny 0/1 -> kategoryczne w LIME


def make_lime(seed):
    return LimeTabularExplainer(Xbb_tr.values, feature_names=bb_cols, class_names=["niedefault", "default"],
                                mode="classification", categorical_features=bin_idx,
                                discretize_continuous=True, random_state=seed)


def lime_score(e):
    """Lokalne R² modelu zastępczego LIME."""
    s = e.score
    return float(s[1]) if isinstance(s, dict) else float(s)


def lime_local_pred(e):
    """Predykcja lokalnego modelu LIME dla objaśnianej obserwacji."""
    lp = e.local_pred[1] if isinstance(e.local_pred, dict) else e.local_pred
    return float(np.atleast_1d(lp)[0])


def lime_explain(i, seed=0, k=10, n=5000):
    """LIME dla obserwacji validation o pozycji i (k cech, n próbek)."""
    return make_lime(seed).explain_instance(Xbb_va.iloc[i].values, predict_fn, num_features=k, num_samples=n)


def top_cech(seria, k):
    """Tekst z k cechami o największym |wkładzie|."""
    s = seria.reindex(seria.abs().sort_values(ascending=False).index).head(k)
    return ", ".join(f"{n} ({v:+.2f})" for n, v in s.items())


def case_table(name, i, k=5):
    c = lr_contrib(X_va.iloc[[i]]).iloc[0]
    c.index = [raw_name(f) for f in c.index]
    e = lime_explain(i, seed=SEED)
    lw = pd.Series({bb_cols[j]: w for j, w in e.local_exp[1]})
    s = pd.Series(sv_va[i], index=bb_cols)
    print(f"=== {name} | y = {y_va[i]} | PD karta = {p_val_lr[i]:.3f}, PD LightGBM = {p_val_bb[i]:.3f} | "
          f"LIME R² = {lime_score(e):.2f}")
    print("  karta (wkład do logitu):", top_cech(c, k))
    print("  SHAP (log-odds):        ", top_cech(s, k))
    print("  LIME (waga lokalna):    ", top_cech(lw, k))
    print()


for name, i in cases.items():
    case_table(name, i)

In [ ]:
for name, i in cases.items():
    shap.plots.waterfall(shap.Explanation(values=sv_va[i], base_values=ev_va, data=Xbb_va.iloc[i].values,
                                          feature_names=bb_cols), max_display=10, show=False)
    plt.title(name)
    savefig(f"shap_waterfall_{name[0]}")
    plt.show()

### 14.4. Analiza what-if

Dla obserwacji 1 (wykryty default) trzy wskaźniki o największym dodatnim SHAP ustawiamy na medianę z train – pojedynczo i łącznie – i porównujemy PD obu modeli. Zmiany dotyczą danych wejściowych, więc przechodzą przez cały pipeline (binning/WoE, capping, imputacja).

In [ ]:
i0 = cases["1. wykryty default"]
shap_sorted = pd.Series(sv_va[i0], index=bb_cols).sort_values(ascending=False)
raw_feats = [f for f in shap_sorted.index if f in NUM_INTERP][:3]

med = X_tr[raw_feats].replace([np.inf, -np.inf], np.nan).median()
base = X_va.iloc[[i0]]
scen = {"obserwacja (bez zmian)": {}, **{f"{f} -> mediana": {f: med[f]} for f in raw_feats},
        "wszystkie 3 -> mediana": {f: med[f] for f in raw_feats}}

wi = []
for name, ch in scen.items():
    r = base.copy()
    for f, v in ch.items():
        r[f] = v
    wi.append({"scenariusz": name, "PD karta": pipe_lr.predict_proba(r)[0, 1],
               "PD LightGBM": pipe_bb.predict_proba(r)[0, 1]})
display(pd.DataFrame(wi).set_index("scenariusz"))
display(pd.DataFrame({"wartość obserwacji": base[raw_feats].iloc[0], "mediana train": med,
                      "w karcie": [("woe__" + f) in feats_lr for f in raw_feats]}))

## 15. Wierność i stabilność wyjaśnień

### 15.1. Wierność (fidelity)
* **LIME:** lokalne R² modelu zastępczego oraz |PD lokalnego modelu − PD LightGBM| dla 30 losowych obserwacji validation.
* **SHAP:** addytywność – wartość bazowa + Σ SHAP = wynik LightGBM w skali log-odds.
* **Karta:** wkłady są dokładną dekompozycją modelu (asercja w 13.3).

In [ ]:
sample_i = np.random.default_rng(SEED).choice(len(X_va), 30, replace=False)
fid = []
for i in sample_i:
    e = lime_explain(int(i), seed=SEED, k=10)
    lp, mp = lime_local_pred(e), float(np.asarray(e.predict_proba)[1])
    fid.append({"i": int(i), "R2 lokalne": lime_score(e), "PD model": mp, "PD LIME": lp, "|ΔPD|": abs(lp - mp)})
fid = pd.DataFrame(fid)
display(fid.drop(columns="i").describe().loc[["mean", "50%", "min", "max"]])
print(f"LIME: mediana lokalnego R² = {fid['R2 lokalne'].median():.2f}; udział obserwacji z R² < 0,5: "
      f"{(fid['R2 lokalne'] < 0.5).mean():.0%}; mediana |ΔPD| = {fid['|ΔPD|'].median():.3f}")

raw_val = bb_model.predict(Xbb_va, raw_score=True)
add_err = np.abs(ev_va + sv_va.sum(1) - raw_val).max()
print(f"SHAP: maks. błąd addytywności = {add_err:.2e} (log-odds)")

fig, ax = plt.subplots(figsize=(6, 3.4))
fid["R2 lokalne"].plot.hist(bins=12, ax=ax, color="#6baed6")
ax.set(title="LIME: lokalne R² (30 obserwacji validation)", xlabel="R²")
savefig("lime_fidelity")
plt.show()

### 15.2. Stabilność
Trzy źródła niestabilności:
1. **metoda** – LIME dla tej samej obserwacji i tego samego modelu z 10 ziarnami (Spearman wag, Jaccard top-5, zgodność znaku top-3),
2. **próbka objaśniana** – bootstrap obserwacji validation dla globalnego rankingu SHAP,
3. **model** – LightGBM uczony od nowa w 5 foldach train (z przygotowaniem cech w foldzie), rankingi SHAP na tym samym zbiorze validation; ten sam test dla ważności karty.

In [ ]:
def lime_w(i, seed):
    """Wektor wag LIME (długość = liczba kolumn); cechy spoza wyjaśnienia mają wagę 0."""
    w = np.zeros(len(bb_cols))
    for j, v in lime_explain(i, seed=seed, k=10).local_exp[1]:
        w[j] = v
    return w


def topk(w, k=5):
    return set(np.argsort(-np.abs(w))[:k])


def pair_stats(W, k=5):
    """Średni Spearman wag i średni Jaccard top-k po wszystkich parach uruchomień."""
    used = np.where((W != 0).any(0))[0]
    sp, jc = [], []
    for a, b in combinations(range(len(W)), 2):
        sp.append(spearmanr(W[a, used], W[b, used])[0])
        jc.append(len(topk(W[a], k) & topk(W[b], k)) / len(topk(W[a], k) | topk(W[b], k)))
    return np.nanmean(sp), np.mean(jc)


rows = []
for name, i in cases.items():
    W = np.array([lime_w(i, s) for s in range(10)])
    sp, jc = pair_stats(W)
    top3 = np.argsort(-np.abs(W.mean(0)))[:3]
    same_sign = np.mean([np.mean(np.sign(W[:, j]) == np.sign(np.median(W[:, j]))) for j in top3])
    rows.append({"przypadek": name, "Spearman wag": sp, "Jaccard top-5": jc, "zgodność znaku top-3": same_sign})
lime_stab = pd.DataFrame(rows).set_index("przypadek")
display(lime_stab)

In [ ]:
K = 10


def pairwise_rank_stats(M, k=10):
    """Średni Spearman i Jaccard top-k po parach kolumn M (kolumna = ranking ważności; brak cechy = 0)."""
    sps, jcs = [], []
    for a, b in combinations(M.columns, 2):
        rho = spearmanr(M[a], M[b])[0]
        if np.isfinite(rho):
            sps.append(rho)
        ta, tb = set(M[a].nlargest(k).index), set(M[b].nlargest(k).index)
        jcs.append(len(ta & tb) / len(ta | tb))
    return float(np.nanmean(sps)), float(np.nanmean(jcs))


# (2) bootstrap obserwacji validation – ten sam model
B = 200
rng_stab = np.random.default_rng(SEED)
full_imp = pd.Series(np.abs(sv_va).mean(0), index=bb_cols)
ref_top = set(full_imp.nlargest(K).index)
boot_sp, boot_jc = [], []
boot_freq = pd.Series(0.0, index=bb_cols)
for _ in range(B):
    imp = pd.Series(np.abs(sv_va[rng_stab.integers(0, len(sv_va), len(sv_va))]).mean(0), index=bb_cols)
    boot_sp.append(spearmanr(full_imp.values, imp.values)[0])
    tb = set(imp.nlargest(K).index)
    boot_jc.append(len(ref_top & tb) / len(ref_top | tb))
    boot_freq.loc[list(tb)] += 1 / B

# (3) LightGBM uczony od nowa w foldach – SHAP na tym samym zbiorze validation
refit_imp = []
for k, (tr, va) in enumerate(cv_train_glob, start=1):
    prep_k = ScoringFeatureBuilder(**FB_PARAMS_BB).fit(X_git.iloc[tr], y_all[tr])
    model_k = mk_lgb(BEST_BB).fit(prep_k.transform(X_git.iloc[tr]), y_all[tr])
    X_ref = prep_k.transform(X_va)
    sv_k, _ = shap_pos(shap.TreeExplainer(model_k), X_ref)
    refit_imp.append(pd.Series(np.abs(sv_k).mean(0), index=X_ref.columns, name=f"fold_{k}"))
refit_matrix = pd.concat(refit_imp, axis=1).fillna(0.0)
refit_sp, refit_jc = pairwise_rank_stats(refit_matrix, k=K)
fold_freq = pd.Series(0.0, index=refit_matrix.index)
for col in refit_matrix.columns:
    fold_freq.loc[list(refit_matrix[col].nlargest(K).index)] += 1 / refit_matrix.shape[1]

# karta uczona w foldach – ważność |β|·std WoE
lr_matrix = pd.concat([pd.Series(np.abs(m.coef_) * folds_woe[k][0][m.features_].std().values,
                                 index=[raw_name(f) for f in m.features_])
                       for k, m in enumerate(fold_models_lr)], axis=1).fillna(0.0)
lr_sp, lr_jc = pairwise_rank_stats(lr_matrix, k=K)

stab_tab = pd.DataFrame([
    {"eksperyment": "SHAP – bootstrap validation, ten sam model", "Spearman": np.mean(boot_sp), "Jaccard top-10": np.mean(boot_jc)},
    {"eksperyment": "SHAP – LightGBM uczony od nowa w 5 foldach", "Spearman": refit_sp, "Jaccard top-10": refit_jc},
    {"eksperyment": "Karta – model uczony od nowa w 5 foldach", "Spearman": lr_sp, "Jaccard top-10": lr_jc},
]).set_index("eksperyment")
display(stab_tab)
print(f"LIME – 10 ziaren (średnia z 4 obserwacji): Spearman wag = {lime_stab['Spearman wag'].mean():.2f}, "
      f"Jaccard top-5 = {lime_stab['Jaccard top-5'].mean():.2f}")
print(f"Bootstrap SHAP: 90% przedział Spearmana [{np.quantile(boot_sp, .05):.3f}; {np.quantile(boot_sp, .95):.3f}], "
      f"Jaccard top-10 [{np.quantile(boot_jc, .05):.2f}; {np.quantile(boot_jc, .95):.2f}]")
print("Cechy najczęściej w top-10 SHAP (foldy):")
display(fold_freq.sort_values(ascending=False).head(12).to_frame("udział foldów w top-10"))

### 15.3. Korelacje między najważniejszymi cechami SHAP

Silnie skorelowane cechy dzielą między siebie wpływ w SHAP i LIME, więc ranking pojedynczej cechy należy czytać razem z jej odpowiednikami. SHAP i LIME opisują zachowanie modelu, a nie zależności przyczynowe.

In [ ]:
top10 = imp_shap.head(10).index
Cv = Xbb_tr[top10].corr(method="spearman").abs().values.copy()
np.fill_diagonal(Cv, 0)
display(pd.DataFrame({"maks. |ρ| z inną cechą top-10": Cv.max(1),
                      "najbardziej skorelowana": [top10[j] for j in Cv.argmax(1)]}, index=top10))

## 16. Porównanie modeli (validation) i zamrożenie

Różnica AUC na validation z 95% przedziałem bootstrap (1000 losowań). Po porównaniu modele są zapisywane i nie są dalej zmieniane.

In [ ]:
def format_cv(wiersze, metryka):
    v = [r[metryka] for r in wiersze]
    return f"{np.mean(v):.3f} ± {np.std(v, ddof=1):.3f}"


cmp = pd.DataFrame({"Scorecard": all_metrics(y_va, p_val_lr), "LightGBM": all_metrics(y_va, p_val_bb)}).T
cmp["liczba cech"] = [len(feats_lr), len(prep_bb.feature_names_)]
cmp["CV AUC"] = [format_cv(cv_rows_lr, "AUC"), format_cv(cv_rows_bb, "AUC")]
cmp["CV LogLoss"] = [format_cv(cv_rows_lr, "LogLoss"), format_cv(cv_rows_bb, "LogLoss")]
display(cmp[METRYKI + ["liczba cech", "CV AUC", "CV LogLoss"]])

rbs = np.random.default_rng(SEED)
diffs = []
for _ in range(1000):
    r = rbs.integers(0, len(y_va), len(y_va))
    if y_va[r].min() == y_va[r].max():
        continue
    diffs.append(roc_auc_score(y_va[r], p_val_bb[r]) - roc_auc_score(y_va[r], p_val_lr[r]))
lo_d, hi_d = np.percentile(diffs, [2.5, 97.5])
print(f"ΔAUC (LightGBM − karta) na validation: {np.mean(diffs):+.3f}, 95% przedział bootstrap [{lo_d:+.3f}; {hi_d:+.3f}]"
      f" – przedział {'zawiera' if lo_d <= 0 <= hi_d else 'nie zawiera'} 0.")

FROZEN = {"BEST_LR": BEST_LR, "BEST_BB": BEST_BB, "features_scorecard": feats_lr,
          "FB_PARAMS_WOE": {k: v for k, v in FB_PARAMS_WOE.items() if k not in ("num_cols", "cat_cols")},
          "FB_PARAMS_BB": {k: v for k, v in FB_PARAMS_BB.items() if k not in ("num_cols", "cat_cols")}}
joblib.dump({"pipe_lr": pipe_lr, "pipe_bb": pipe_bb}, ART_DIR / "models_frozen.joblib")
with open(ART_DIR / "models_frozen_params.json", "w", encoding="utf-8") as f:
    json.dump(FROZEN, f, indent=2, default=float, ensure_ascii=False)
print("Zapisano: models_frozen.joblib, models_frozen_params.json")

### 16.1. Wariant bez `wsk_index_*`

Kontynuacja testu z rozdziału 9.3 dla modeli finalnych (te same hiperparametry, te same foldy): AUC i log-loss w CV z wskaźnikami sektorowymi i bez nich.

In [ ]:
susp_woe = ["woe__" + c for c in SUSPICIOUS]
sens = []
for label, drop in [("wszystkie cechy", False), ("bez wsk_index_*", True)]:
    r_lr, r_bb = [], []
    for (Xa, ya, Xb, yb, _, _), (Xa2, _, Xb2, _, _, _) in zip(folds_woe, folds_bb):
        keep = [c for c in Xa.columns if not (drop and c in susp_woe)]
        p = ScorecardLR(**BEST_LR).fit(Xa[keep], ya).predict_proba(Xb[keep])[:, 1]
        r_lr.append((roc_auc_score(yb, p), log_loss(yb, p)))
        keep2 = [c for c in Xa2.columns if not (drop and any(c.startswith(s) for s in SUSPICIOUS))]
        p2 = mk_lgb(BEST_BB).fit(Xa2[keep2], ya).predict_proba(Xb2[keep2])[:, 1]
        r_bb.append((roc_auc_score(yb, p2), log_loss(yb, p2)))
    for model, r in [("Scorecard", r_lr), ("LightGBM", r_bb)]:
        r = np.array(r)
        sens.append({"model": model, "wariant": label, "AUC CV": r[:, 0].mean(), "AUC std": r[:, 0].std(ddof=1),
                     "LogLoss CV": r[:, 1].mean()})
sens = pd.DataFrame(sens).set_index(["model", "wariant"])
display(sens)
for m in MODELS:
    d = sens.loc[(m, "wszystkie cechy"), "AUC CV"] - sens.loc[(m, "bez wsk_index_*"), "AUC CV"]
    print(f"{m}: spadek AUC po usunięciu wsk_index_* = {d:.3f}")

## 17. Kalibracja PD

Modele są uczone na train; kalibratory i korekta do 4% są dopasowywane na validation, a oceniane na teście (17.4). Validation zawiera 35 defaultów, dlatego metody kalibracji porównujemy walidacją krzyżową wewnątrz validation (20 × Stratified 5-fold: kalibrator uczony na 4/5, oceniany na 1/5).

Miary: Brier, log-loss, CITL (calibration-in-the-large: > 0 – model zaniża PD, < 0 – zawyża), slope (< 1 – predykcje zbyt rozproszone, > 1 – zbyt skupione), ECE.

### 17.1. Diagnostyka przed kalibracją (validation)

In [ ]:
PIPES = {"Scorecard": pipe_lr, "LightGBM": pipe_bb}
P_VAL = {"Scorecard": p_val_lr, "LightGBM": p_val_bb}


def plot_reliability(curves, title, fname, n_bins=8):
    """Krzywa reliability (koszyki kwantylowe) i histogram PD."""
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1.2, 1]})
    for lab, (y, p) in curves.items():
        obs, pred = calibration_curve(y, p, n_bins=n_bins, strategy="quantile")
        a1.plot(pred, obs, "o-", label=lab)
        a2.hist(p, bins=30, alpha=.5, label=lab)
    a1.plot([0, 1], [0, 1], "k--", lw=1, label="idealna kalibracja")
    a1.set(xlabel="średnie PD w koszyku", ylabel="obserwowana częstość defaultów", title="Krzywa reliability")
    a1.legend(fontsize=8)
    a2.set(xlabel="PD", ylabel="liczba obserwacji", title="Rozkład PD")
    a2.legend(fontsize=8)
    fig.suptitle(title)
    savefig(fname)
    plt.show()


pre = pd.DataFrame({m: all_metrics(y_va, P_VAL[m]) for m in MODELS}).T
display(pre[["AUC", "Brier", "LogLoss", "CITL", "Slope", "ECE", "mean_PD", "obs_rate"]])
plot_reliability({m: (y_va, P_VAL[m]) for m in MODELS},
                 f"Przed kalibracją – validation (8 koszyków, {int(y_va.sum())} defaultów)", "reliability_precal")

### 17.2. Porównanie metod kalibracji: intercept, Platt, izotoniczna, beta

Kryterium wyboru: średni log-loss z CV wewnątrz validation. Metody o log-lossie w granicy 0,5% od najlepszego traktujemy jako równoważne i wybieramy najprostszą (kolejność: brak, intercept, Platt, beta, izotoniczna).

In [ ]:
CALS = {"brak": IdentityCalibrator, "Intercept": InterceptCalibrator, "Platt": PlattCalibrator,
        "Isotonic": IsotonicCalibrator, "Beta": BetaCalibrator}


def cv_in_val(p, y, cal_cls, n_rep=20, n_splits=5):
    """CV kalibratora wewnątrz validation; zwraca średnie i odchylenia metryk oraz predykcje OOF z 1. powtórzenia."""
    wyniki, pierwsze_oof = [], None
    for rep in range(n_rep):
        oof = np.zeros(len(y))
        for a, b in StratifiedKFold(n_splits, shuffle=True, random_state=rep).split(p, y):
            oof[b] = cal_cls().fit(p[a], y[a]).predict(p[b])
        wyniki.append(all_metrics(y, oof))
        if rep == 0:
            pierwsze_oof = oof
    t = pd.DataFrame(wyniki)
    return t.mean(), t.std(), pierwsze_oof


cal_rows, oof_first = [], {}
for m in MODELS:
    for name, cls in CALS.items():
        mu, sd, oof = cv_in_val(P_VAL[m], y_va, cls)
        oof_first[(m, name)] = oof
        cal_rows.append({"model": m, "metoda": name, "Brier": mu["Brier"], "LogLoss": mu["LogLoss"],
                         "LogLoss std": sd["LogLoss"], "CITL": mu["CITL"], "Slope": mu["Slope"],
                         "ECE": mu["ECE"], "AUC": mu["AUC"]})
cal_cmp = pd.DataFrame(cal_rows).set_index(["model", "metoda"])
ref_brak = cal_cmp.xs("brak", level="metoda").reindex(cal_cmp.index.get_level_values("model"))
cal_cmp["ΔLogLoss vs brak"] = cal_cmp["LogLoss"].values - ref_brak["LogLoss"].values
cal_cmp["ΔBrier vs brak"] = cal_cmp["Brier"].values - ref_brak["Brier"].values
display(cal_cmp)

CAL_ORDER = ["brak", "Intercept", "Platt", "Beta", "Isotonic"]
REL_TOL = 0.005
BEST_CAL = {}
for m in MODELS:
    t = cal_cmp.loc[m]
    eligible = t.index[t["LogLoss"] <= t["LogLoss"].min() * (1 + REL_TOL)]
    BEST_CAL[m] = next(c for c in CAL_ORDER if c in eligible)
print("Wybrane metody kalibracji:", BEST_CAL)
for m in MODELS:
    print(f"{m}: log-loss Platt {cal_cmp.loc[(m, 'Platt'), 'LogLoss']:.4f} vs izotoniczna "
          f"{cal_cmp.loc[(m, 'Isotonic'), 'LogLoss']:.4f}; slope izotonicznej {cal_cmp.loc[(m, 'Isotonic'), 'Slope']:.2f}")

In [ ]:
for m in MODELS:
    plot_reliability({"przed kalibracją": (y_va, P_VAL[m]),
                      f"po kalibracji ({BEST_CAL[m]}, OOF w validation)": (y_va, oof_first[(m, BEST_CAL[m])])},
                     f"{m}: przed i po kalibracji (validation)", f"reliability_postcal_{m}")

### 17.3. Korekta do średniego PD = 4%

Kalibracja z 17.2 dopasowuje PD do częstości defaultów w próbie (5,83% w validation). Średnie PD = 4% jest założeniem projektu o portfelu docelowym: po kalibracji przesuwamy logit o stałą δ tak, by średnie PD na validation wynosiło 4%. Przesunięcie jest monotoniczne, więc nie zmienia rankingu (AUC bez zmian); zmienia poziom PD, więc względem częstości w próbie PD będą zaniżone.

In [ ]:
cal_final, p_val_cal, delta, p_val_4 = {}, {}, {}, {}
for m in MODELS:
    cal_final[m] = CALS[BEST_CAL[m]]().fit(P_VAL[m], y_va)
    p_val_cal[m] = cal_final[m].predict(P_VAL[m])
    delta[m] = shift_to_mean(p_val_cal[m], TARGET_PD)
    p_val_4[m] = apply_shift(p_val_cal[m], delta[m])

ref = pd.DataFrame({"DR train": y_tr.mean(), "DR validation": y_va.mean(), "docelowe PD": TARGET_PD,
                    "śr. PD przed kalibracją": [P_VAL[m].mean() for m in MODELS],
                    "śr. PD po kalibracji": [p_val_cal[m].mean() for m in MODELS],
                    "δ (logit)": [delta[m] for m in MODELS],
                    "exp(δ) – mnożnik szans": [np.exp(delta[m]) for m in MODELS],
                    "śr. PD po korekcie do 4%": [p_val_4[m].mean() for m in MODELS]}, index=MODELS)
display(ref.T)
for m in MODELS:
    assert abs(p_val_4[m].mean() - TARGET_PD) < 1e-6
    assert abs(roc_auc_score(y_va, p_val_4[m]) - roc_auc_score(y_va, p_val_cal[m])) < 1e-12
print("OK: średnie PD po korekcie = 4%, AUC bez zmian.")

### 17.4. Ocena na zbiorze testowym

Test jest używany pierwszy raz. Trzy etapy PD: surowe, po kalibracji, po korekcie do 4%. Dodatkowo metryki kalibracji po przeważeniu testu do częstości defaultów 4% (wagi: defaulty 0,04/π, niedefaulty 0,96/(1 − π)); przeważenie zakłada, że rozkłady cech w klasach są takie same jak w próbie.

In [ ]:
X_te, y_te = X_git.iloc[idx_test], y_all[idx_test]
p_te_raw, p_te_cal, p_te_4 = {}, {}, {}
for m in MODELS:
    p_te_raw[m] = PIPES[m].predict_proba(X_te)[:, 1]
    p_te_cal[m] = cal_final[m].predict(p_te_raw[m])
    p_te_4[m] = apply_shift(p_te_cal[m], delta[m])
print(f"TEST: n = {len(y_te)}, defaulty = {int(y_te.sum())} ({y_te.mean():.2%})")


def stages(m):
    return [("surowe", p_te_raw[m]), ("po kalibracji", p_te_cal[m]), ("po korekcie do 4%", p_te_4[m])]


test_tab = pd.DataFrame([{"model": m, "etap": s, **all_metrics(y_te, p)} for m in MODELS for s, p in stages(m)]
                        ).set_index(["model", "etap"])
display(test_tab[["AUC", "PR_AUC", "KS", "LogLoss", "Brier", "CITL", "Slope", "ECE", "mean_PD", "obs_rate"]])

w_te = np.where(y_te == 1, TARGET_PD / y_te.mean(), (1 - TARGET_PD) / (1 - y_te.mean()))
wtab = pd.DataFrame([{"model": m, "etap": s, **{k: v for k, v in all_metrics(y_te, p, w_te).items()
                                                   if k in ("Brier", "LogLoss", "CITL", "Slope", "mean_PD", "obs_rate")}}
                     for m in MODELS for s, p in stages(m)[1:]]).set_index(["model", "etap"])
print("Test przeważony do częstości defaultów 4%:")
display(wtab)

for m in MODELS:
    plot_reliability({s: (y_te, p) for s, p in stages(m)}, f"{m}: reliability na teście", f"reliability_test_{m}")

dist = pd.DataFrame({f"{m} | {s}": pd.Series(p).describe(percentiles=[.1, .5, .9, .99])
                     for m in MODELS for s, p in stages(m)}).T[["mean", "min", "10%", "50%", "90%", "99%", "max"]]
display(dist)

## 18. Progi decyzyjne, ratingi i analiza kontrfaktyczna

### 18.1. Funkcja kosztu i próg decyzyjny

Założenia (parametry `MARGIN` i `LGD`): przyjęty dobry klient daje marżę MARGIN, przyjęty default – stratę LGD (jako ułamek ekspozycji). Klient jest odrzucany, gdy PD ≥ próg. Zysk = MARGIN·TN − LGD·FN. Teoretyczny próg dla skalibrowanego PD: MARGIN / (MARGIN + LGD).

Próg wybieramy na validation, maksymalizując zysk na PD po kalibracji (skala częstości w próbie); odpowiednik w skali PD 4% otrzymujemy przesunięciem δ.

In [ ]:
MARGIN = 0.06
LGD = 0.45
t_theory = MARGIN / (MARGIN + LGD)

t_star, krzywe_zysku = {}, {}
for m in MODELS:
    grid_t = np.unique(np.quantile(p_val_cal[m], np.linspace(0.5, 0.995, 150)))
    cu = pd.DataFrame([decision_table(y_va, p_val_cal[m], t, MARGIN, LGD) for t in grid_t])
    krzywe_zysku[m] = cu
    t_star[m] = float(cu.loc[cu["zysk_na_klienta"].idxmax(), "prog"])

thr_tab = pd.DataFrame({
    "próg* (skala próby)": t_star,
    "próg* (skala PD 4%)": {m: float(apply_shift(t_star[m], delta[m])) for m in MODELS},
    "% odrzuceń (val)": {m: 100 * (p_val_cal[m] >= t_star[m]).mean() for m in MODELS},
    "zysk/klienta przy progu* (val)": {m: krzywe_zysku[m]["zysk_na_klienta"].max() for m in MODELS},
    "zysk/klienta – przyjęcie wszystkich (val)": MARGIN * (1 - y_va.mean()) - LGD * y_va.mean(),
    "próg teoretyczny (skala próby)": t_theory,
})
display(thr_tab)

fig, ax = plt.subplots(1, 3, figsize=(17, 4.2))
for m in MODELS:
    ax[0].plot(krzywe_zysku[m]["prog"], krzywe_zysku[m]["zysk_na_klienta"], label=m)
    ax[0].axvline(t_star[m], ls=":", color="grey")
    fpr, tpr, _ = roc_curve(y_va, p_val_cal[m])
    ax[1].plot(fpr, tpr, label=f"{m} (AUC = {roc_auc_score(y_va, p_val_cal[m]):.3f})")
    pr, rc, _ = precision_recall_curve(y_va, p_val_cal[m])
    ax[2].plot(rc, pr, label=m)
ax[0].set(title="Zysk na klienta a próg (validation)", xlabel="próg PD (skala próby)")
ax[1].plot([0, 1], [0, 1], "k--")
ax[1].set(title="ROC (validation)", xlabel="FPR", ylabel="TPR")
ax[2].axhline(y_va.mean(), color="k", ls="--")
ax[2].set(title="Precision-Recall (validation)", xlabel="recall", ylabel="precision")
for a in ax:
    a.legend()
savefig("thresholds")
plt.show()

Tabele decyzyjne na teście: próg* z validation oraz progi odpowiadające odsetkom odrzuceń 5–30% (wyznaczone na validation).

In [ ]:
rows = []
for m in MODELS:
    rows.append({"model": m, "wariant": "próg* (val)", **decision_table(y_te, p_te_cal[m], t_star[m], MARGIN, LGD)})
    for r in [5, 10, 15, 20, 30]:
        prog_r = float(np.quantile(p_val_cal[m], 1 - r / 100))
        rows.append({"model": m, "wariant": f"odrzuć ~{r}% (próg z val)",
                     **decision_table(y_te, p_te_cal[m], prog_r, MARGIN, LGD)})
    rows.append({"model": m, "wariant": "przyjmij wszystkich", **decision_table(y_te, p_te_cal[m], 1.01, MARGIN, LGD)})
dec_tab = pd.DataFrame(rows).set_index(["model", "wariant"])
display(dec_tab)

rej = {m: p_te_cal[m] >= t_star[m] for m in MODELS}
print(f"Zgodność decyzji obu modeli (test, progi*): {np.mean(rej['Scorecard'] == rej['LightGBM']):.1%} | "
      f"kappa Cohena: {cohen_kappa_score(rej['Scorecard'], rej['LightGBM']):.2f}")
display(pd.crosstab(pd.Series(rej["Scorecard"], name="karta odrzuca"), pd.Series(rej["LightGBM"], name="LightGBM odrzuca")))

### 18.2. Mapowanie PD → klasy ratingowe

Skala nadrzędna na PD po korekcie do 4%: progi 0,5%, 1%, 2%, 4%, 8%, 15% → AAA, AA, A, BBB, BB, B, CCC. Na validation sąsiednie klasy są łączone, jeśli obserwowana częstość defaultów nie rośnie z klasą albo klasa ma mniej niż 15 obserwacji. Klasa po scaleniu dostaje etykietę klasy skali nadrzędnej, w której leży jej średnie PD – etykiety są porównywalne między modelami. Monotoniczność sprawdzamy na validation i na teście.

In [ ]:
BOUNDS0 = [0.005, 0.01, 0.02, 0.04, 0.08, 0.15]
rating_def = {}
for m in MODELS:
    b = monotone_bounds(p_val_4[m], y_va, BOUNDS0, min_n=15)
    rating_def[m] = (b, labels_from_master(p_val_4[m], b, BOUNDS0))
    print(f"{m}: granice PD (skala 4%) = {[round(x, 4) for x in b]} -> klasy {rating_def[m][1]}")


def monotonicznosc(t):
    """(czy średnie PD rośnie z klasą, czy obserwowana częstość rośnie z klasą) – klasy niepuste."""
    t = t[t["n"] > 0]
    return bool(np.all(np.diff(t["srednia_PD"].values) > 0)), bool(np.all(np.diff(t["obs_default_rate"].values) > 0))


for m in MODELS:
    granice, etykiety = rating_def[m]
    tv = rating_table(p_val_4[m], y_va, granice, etykiety)
    tt = rating_table(p_te_4[m], y_te, granice, etykiety)
    mv, mt = monotonicznosc(tv), monotonicznosc(tt)
    print(f"\n=== {m} – VALIDATION: średnie PD rośnie: {mv[0]}, obserwowana częstość rośnie: {mv[1]}")
    display(tv)
    print(f"=== {m} – TEST: średnie PD rośnie: {mt[0]}, obserwowana częstość rośnie: {mt[1]}")
    display(tt)

    fig, ax = plt.subplots(figsize=(7.5, 3.8))
    x = np.arange(len(tt))
    ax.bar(x - .2, tt["srednia_PD"], .4, label="średnie PD (skala 4%)")
    ax.bar(x + .2, tt["obs_default_rate"], .4, label="obserwowana częstość (test)",
           yerr=[tt["obs_default_rate"] - tt["CI95_lo"], tt["CI95_hi"] - tt["obs_default_rate"]], capsize=3)
    ax.set_xticks(x)
    ax.set_xticklabels(tt.index)
    ax.set_title(f"{m}: PD a obserwowana częstość defaultów w klasach (test)")
    ax.legend()
    savefig(f"ratings_{m}")
    plt.show()

In [ ]:
cl = {m: assign_rating(p_te_4[m], *rating_def[m]) for m in MODELS}
cat = lambda a: pd.Categorical(a, categories=RATING_LABELS, ordered=True)
display(pd.crosstab(cat(cl["Scorecard"]), cat(cl["LightGBM"]), rownames=["karta"], colnames=["LightGBM"], dropna=False))
pos = {lab: i for i, lab in enumerate(RATING_LABELS)}
dd = np.abs(np.array([pos[a] for a in cl["Scorecard"]]) - np.array([pos[b] for b in cl["LightGBM"]]))
print(f"Ta sama klasa: {np.mean(dd == 0):.1%} | różnica ≤ 1 stopień skali: {np.mean(dd <= 1):.1%} "
      f"(liczba klas: karta {len(rating_def['Scorecard'][1])}, LightGBM {len(rating_def['LightGBM'][1])})")

### 18.3. Audyt podgrup (test)

Grupy wielkości firmy według kwartyli aktywów z train (`AKT_EDGES`, rozdział 7.6) oraz forma własności pogrupowana regułą z części I (poziomy < 5% train → `INNE`). AUC raportujemy, gdy grupa ma co najmniej 5 defaultów i 5 niedefaultów; przy tak małych liczebnościach różnice między grupami są orientacyjne.

In [ ]:
g_akt = pd.cut(X_te["Aktywa"].astype(float), bins=AKT_EDGES, labels=AKT_LABELS, include_lowest=True)
g_forma = fb_woe.grouper_.transform(X_te[CAT_COLS])["forma_wl"]
aud = []
for gname, g in {"wielkość (Aktywa)": g_akt, "forma własności": g_forma}.items():
    for lvl in g.dropna().unique():
        mask = (g == lvl).to_numpy()
        n, d = int(mask.sum()), int(y_te[mask].sum())
        for m in MODELS:
            aud.append({"grupa": gname, "poziom": str(lvl), "model": m, "n": n, "defaulty": d,
                        "AUC": roc_auc_score(y_te[mask], p_te_4[m][mask]) if d >= 5 and n - d >= 5 else np.nan,
                        "Brier (po kalibracji)": np.mean((p_te_cal[m][mask] - y_te[mask]) ** 2),
                        "śr. PD po kalibracji": p_te_cal[m][mask].mean(), "obs. DR": y_te[mask].mean(),
                        "śr. PD (skala 4%)": p_te_4[m][mask].mean()})
aud = pd.DataFrame(aud).set_index(["grupa", "poziom", "model"]).sort_index()
display(aud)

### 18.4. Analiza kontrfaktyczna

Dla jednej firmy z validation (niedefault, klasa BB w karcie) szukamy najmniejszej zmiany wskaźników, która obniża PD (skala 4%) poniżej granicy klasy lepszej o jeden stopień.

Ograniczenia:
* zmieniane są tylko wskaźniki karty (bez `wsk_index_*` i wielkości firmy), w kierunku obniżającym ryzyko, w zakresie p5–p95 rozkładu train; 1 lub 2 wskaźniki naraz,
* koszt zmiany = przesunięcie w punktach percentylowych rozkładu train; maks. 50 punktów na wskaźnik,
* wskaźniki skorelowane ze zmienianym (|ρ Spearmana| ≥ 0,7) przesuwają się razem według regresji liniowej z train (obcięcie do p1–p99),
* wynik sprawdzamy również w LightGBM.

In [ ]:
bounds_lr, labels_lr = rating_def["Scorecard"]
assert len(labels_lr) >= 2, "Za mało klas ratingowych po scaleniu."
ci = labels_lr.index("BB") if "BB" in labels_lr and labels_lr.index("BB") >= 1 else \
    min(len(labels_lr) - 2, max(1, len(labels_lr) // 2 + 1))

cls_val = assign_rating(p_val_4["Scorecard"], bounds_lr, labels_lr)
cand = np.where((cls_val == labels_lr[ci]) & (y_va == 0))[0]
if len(cand) == 0:
    cand = np.where(cls_val == labels_lr[ci])[0]
i_cf = int(cand[np.argmin(np.abs(p_val_4["Scorecard"][cand] - np.median(p_val_4["Scorecard"][cand])))])
target_pd = bounds_lr[ci - 1] * 0.98                 # tuż poniżej granicy lepszej klasy


def pd4(m, rows_):
    """PD po kalibracji i korekcie do 4% dla wierszy danych wejściowych."""
    return apply_shift(cal_final[m].predict(PIPES[m].predict_proba(rows_)[:, 1]), delta[m])


base = X_va.iloc[[i_cf]]
print(f"Obserwacja (validation, pozycja {i_cf}): y = {y_va[i_cf]} | klasa karty {labels_lr[ci]} -> cel {labels_lr[ci - 1]} | "
      f"PD karta = {pd4('Scorecard', base)[0]:.4f}, LightGBM = {pd4('LightGBM', base)[0]:.4f} | cel: PD < {target_pd:.4f}")

In [ ]:
iv_t = prep_lr.iv_table_
LEVERS = [raw_name(c) for c in feats_lr
          if raw_name(c) in RATIO_COLS and raw_name(c) not in SUSPICIOUS and np.isfinite(base[raw_name(c)].iloc[0])]
direction = {f: -1 if iv_t.loc[f, "kierunek_ryzyka"] == "inc" else 1 for f in LEVERS}   # kierunek obniżający ryzyko

srt = {f: np.sort(X_tr[f].replace([np.inf, -np.inf], np.nan).dropna().values) for f in RATIO_COLS}
pct = lambda f, v: 100 * np.searchsorted(srt[f], v) / len(srt[f])
q_of = lambda f, q: float(np.quantile(srt[f], q))
clipped = lambda f: np.clip(X_tr[f].replace([np.inf, -np.inf], np.nan), q_of(f, .01), q_of(f, .99))
corr_sp = X_tr[RATIO_COLS].replace([np.inf, -np.inf], np.nan).corr(method="spearman")

dep = {}
for f in LEVERS:
    xf, lista = clipped(f), []
    for j in RATIO_COLS:
        if j != f and abs(corr_sp.loc[f, j]) >= 0.7 and np.isfinite(base[j].iloc[0]):
            xj = clipped(j)
            ok = xf.notna() & xj.notna()
            lista.append((j, float(np.cov(xf[ok], xj[ok])[0, 1] / np.var(xf[ok], ddof=1))))
    dep[f] = lista
print("Wskaźniki zmieniane:", LEVERS)
print("Liczba wskaźników skorelowanych (przesuwanych razem):", {f: len(dep[f]) for f in LEVERS})


def build_rows(changes):
    """Kopie obserwacji ze zmienionymi wskaźnikami; skorelowane wskaźniki przesuwane według regresji z train."""
    n = len(changes[0][1])
    rows_ = base.loc[base.index.repeat(n)].reset_index(drop=True)
    zmieniane = [f for f, _ in changes]
    upd = {}
    for f, vals in changes:
        v0 = float(base[f].iloc[0])
        rows_[f] = vals
        for j, b in dep[f]:
            if j not in zmieniane:
                upd[j] = upd.get(j, 0) + b * (vals - v0)
    for j, d in upd.items():
        xj = float(base[j].iloc[0])
        rows_[j] = np.clip(xj + d, min(q_of(j, .01), xj), max(q_of(j, .99), xj))
    return rows_


MAX_MOVE = 50.0


def grid_for(f, n):
    """n wartości od bieżącej do p95 (lub p5) w kierunku obniżającym ryzyko; None, gdy brak miejsca na zmianę."""
    v0 = float(base[f].iloc[0])
    e = q_of(f, .95) if direction[f] > 0 else q_of(f, .05)
    return None if direction[f] * (e - v0) <= 0 else np.linspace(v0, e, n + 1)[1:]


def cost(f, vals):
    return np.abs(pct(f, vals) - pct(f, float(base[f].iloc[0])))


res = []
for f in LEVERS:
    g = grid_for(f, 25)
    if g is None:
        continue
    rws = build_rows([(f, g)])
    pl, pb, c = pd4("Scorecard", rws), pd4("LightGBM", rws), cost(f, g)
    res += [{"wskaźniki": (f,), "nowe": (float(g[k]),), "koszt": float(c[k]), "PD karta": float(pl[k]),
             "PD LightGBM": float(pb[k])} for k in range(len(g))]
for f, h in combinations(LEVERS, 2):
    gf, gh = grid_for(f, 10), grid_for(h, 10)
    if gf is None or gh is None:
        continue
    A, Bv = (a.ravel() for a in np.meshgrid(gf, gh))
    rws = build_rows([(f, A), (h, Bv)])
    pl, pb, c = pd4("Scorecard", rws), pd4("LightGBM", rws), cost(f, A) + cost(h, Bv)
    res += [{"wskaźniki": (f, h), "nowe": (float(A[k]), float(Bv[k])), "koszt": float(c[k]), "PD karta": float(pl[k]),
             "PD LightGBM": float(pb[k])} for k in range(len(A))]
res = pd.DataFrame(res)
res["ruch ≤ limit"] = [all(cost(f, np.array([v]))[0] <= MAX_MOVE for f, v in zip(d, nv))
                       for d, nv in zip(res["wskaźniki"], res["nowe"])]
feas = res[(res["PD karta"] <= target_pd) & res["ruch ≤ limit"]]
print(f"Sprawdzone kombinacje: {len(res)}; spełniające cel w karcie i limit zmiany: {len(feas)}")

In [ ]:
def show_cf(row, title):
    """Zmiany wskaźników (przed/po, percentyle), koszt oraz PD i klasa w obu modelach."""
    t = pd.DataFrame({"wskaźnik": row["wskaźniki"],
                      "przed": [float(base[f].iloc[0]) for f in row["wskaźniki"]], "po": row["nowe"],
                      "percentyl przed": [pct(f, float(base[f].iloc[0])) for f in row["wskaźniki"]],
                      "percentyl po": [pct(f, v) for f, v in zip(row["wskaźniki"], row["nowe"])]})
    print(f"--- {title}: koszt = {row['koszt']:.1f} pkt percentylowych")
    display(t)
    print(f"    karta:    PD {pd4('Scorecard', base)[0]:.4f} -> {row['PD karta']:.4f} | klasa {labels_lr[ci]} -> "
          f"{assign_rating([row['PD karta']], bounds_lr, labels_lr)[0]}")
    bnd_b, lab_b = rating_def["LightGBM"]
    pb0 = pd4("LightGBM", base)[0]
    print(f"    LightGBM: PD {pb0:.4f} -> {row['PD LightGBM']:.4f} | klasa {assign_rating([pb0], bnd_b, lab_b)[0]} -> "
          f"{assign_rating([row['PD LightGBM']], bnd_b, lab_b)[0]}")


cf_rows = {}
for kind in [1, 2]:
    s = feas[feas["wskaźniki"].map(len) == kind]
    if len(s):
        cf_rows[kind] = s.sort_values("koszt").iloc[0]
        show_cf(cf_rows[kind], f"Najtańsza zmiana – {kind} wskaźnik" + ("" if kind == 1 else "i"))
if not cf_rows:
    print("Brak rozwiązania w przyjętych ograniczeniach (MAX_MOVE, zakres p5–p95, maks. 2 wskaźniki).")

In [ ]:
# porównanie: czynniki zwiększające ryzyko według karty, SHAP i LIME a wskaźniki zmieniane w analizie kontrfaktycznej
c_lr = lr_contrib(X_va.iloc[[i_cf]]).iloc[0]
c_lr.index = [raw_name(f) for f in c_lr.index]
s_sh = pd.Series(sv_va[i_cf], index=bb_cols)
s_li = pd.Series({bb_cols[j]: w for j, w in lime_explain(i_cf, seed=SEED).local_exp[1]})
up = lambda s, k=5: s[s > 0].sort_values(ascending=False).head(k).index.tolist()

cmp_cf = {"karta (wkład > 0)": up(c_lr), "SHAP (> 0)": up(s_sh), "LIME (> 0)": up(s_li)}
for n_, v in cmp_cf.items():
    print(f"{n_:20s}: {v}")
if cf_rows:
    cfset = {f for r in cf_rows.values() for f in r["wskaźniki"]}
    print(f"{'wskaźniki zmieniane':20s}: {sorted(cfset)}")
    for n_, v in cmp_cf.items():
        print(f"  wspólne z {n_}: {sorted(cfset & set(v)) or 'brak'}")

## 19. Podsumowanie i zapis wyników

In [ ]:
summary = test_tab.xs("po korekcie do 4%", level="etap")[["AUC", "PR_AUC", "KS", "LogLoss", "Brier", "CITL", "Slope", "mean_PD"]].copy()
summary["liczba cech"] = [len(feats_lr), len(prep_bb.feature_names_)]
summary["kalibracja"] = [BEST_CAL[m] for m in summary.index]
summary["klasy ratingowe"] = [len(rating_def[m][1]) for m in summary.index]
summary["zysk/klienta (test, próg*)"] = [dec_tab.loc[(m, "próg* (val)"), "zysk_na_klienta"] for m in summary.index]
print("Wyniki na teście (PD po korekcie do 4%):")
display(summary)


def pred_cols(idx, y, praw, pcal, p4):
    dane = {"idx": idx, "y": y}
    for m in MODELS:
        dane[f"pd_raw_{m}"], dane[f"pd_cal_{m}"], dane[f"pd_4pct_{m}"] = praw[m], pcal[m], p4[m]
        dane[f"rating_{m}"] = assign_rating(p4[m], *rating_def[m])
    return pd.DataFrame(dane)


pred_cols(idx_val, y_va, P_VAL, p_val_cal, p_val_4).to_csv(ART_DIR / "predictions_val.csv", index=False)
pred_cols(idx_test, y_te, p_te_raw, p_te_cal, p_te_4).to_csv(ART_DIR / "predictions_test.csv", index=False)
test_tab.to_csv(ART_DIR / "metrics_test.csv")
cal_cmp.to_csv(ART_DIR / "calibration_cv_comparison.csv")
dec_tab.to_csv(ART_DIR / "decision_tables_test.csv")
aud.to_csv(ART_DIR / "subgroup_audit_test.csv")
joblib.dump({"calibrators": cal_final, "delta": delta, "best_cal": BEST_CAL, "rating_def": rating_def, "t_star": t_star},
            ART_DIR / "calibration_ratings.joblib")
with open(ART_DIR / "calibration_decisions.json", "w", encoding="utf-8") as f:
    json.dump({"TARGET_PD": TARGET_PD, "delta": delta, "best_cal": BEST_CAL, "t_star": t_star, "MARGIN": MARGIN,
               "LGD": LGD, "rating_bounds": {m: rating_def[m][0] for m in MODELS},
               "rating_labels": {m: rating_def[m][1] for m in MODELS}}, f, indent=2, default=float, ensure_ascii=False)

from importlib.metadata import version, PackageNotFoundError
req = []
for nazwa in ["numpy", "pandas", "scikit-learn", "scipy", "matplotlib", "seaborn", "joblib", "lightgbm", "shap", "lime"]:
    try:
        req.append(f"{nazwa}=={version(nazwa)}")
    except PackageNotFoundError:
        req.append(nazwa)
Path("requirements.txt").write_text("\n".join(req) + "\n", encoding="utf-8")
print("requirements.txt:", req)
print("Zapisano:", sorted(p.name for p in ART_DIR.iterdir() if p.is_file()))

| Wymaganie | Realizacja | Rozdział |
|---|---|---|
| 3.3 Model interpretowalny | karta WoE + regresja logistyczna (β < 0), selekcja L1/VIF, tuning w CV z regułą 1-SE; AUC, PR-AUC, KS, log-loss, Brier; współczynniki, ilorazy szans, profile przedziałów, PDP/ICE, dekompozycja logitu, stabilność w foldach | 13 |
| 3.4 Model black-box | LightGBM, Random Search w 5-fold CV na train, reguła 1-SE, krzywa log-lossu, porównanie AUC train / CV / validation | 14 |
| Wyjaśnialność globalna i lokalna | SHAP (beeswarm, ważność, zależności, waterfall), LIME, PDP/ICE, what-if, porównanie rankingów karta vs SHAP | 13, 14 |
| Wierność i stabilność wyjaśnień | lokalne R² LIME, addytywność SHAP; LIME z 10 ziarnami, bootstrap SHAP, SHAP i karta uczone w foldach | 15 |
| 3.5 Kalibracja | diagnostyka przed kalibracją, porównanie intercept / Platt / izotoniczna / beta (CV w validation), korekta do średniego PD = 4%, ocena na teście (także przeważonym do 4%) | 17 |
| 3.6 Progi i ratingi | funkcja kosztu, próg z validation, tabele decyzyjne, klasy ratingowe z monotoniczną częstością defaultów, porównanie decyzji modeli, audyt podgrup, analiza kontrfaktyczna | 18 |

Zbiór testowy jest używany wyłącznie w 17.4 i 18 (ocena). Wszystkie progi, kalibratory i granice klas są wyznaczone na validation.